### Multi-Echo Pilot: Statistical Analysis & Visualization

This notebook contains the complete analysis pipeline for the multiecho pilot study.

**Structure:**
1. **Setup** — Imports, configuration, shared utility functions
2. **Demographics** — Subject filtering and sample characterization
3. **Shared Dependencies** — Smoothness data loading
4. **Spatial Smoothness** — Pre/post smoothness analysis (Fig. 3)
5. **Whole-Brain TSNR** — Temporal SNR with smoothness covariate (Fig. 4)
6. **ROI TSNR** — ROI-based TSNR analysis (Fig. 5)
7. **ROI Activation (EMMs)** — Beta, Z-stat, and Weighted Beta analyses for VS, FFA, Motor (Fig. 6)
8. **Motor vs Cerebellum Comparison** — Dedicated ROI comparison analysis (beta, z-stat, weighted beta)
9. **PPI Connectivity** — PPI-based analyses (Fig. 7)
10. **Design Matrices for FSL randomise** — Group-level design matrix generation (Fig. 8)
11. **TEDANA Denoising Efficacy** — Delta R² analyses (per-acquisition)
12. **TEDANA Whole-Brain Exploratory** — dlPFC and Left IPL single-ROI explorations
13. **TEDANA Pairwise Comparisons** — Pairwise acquisition contrasts
14. **TEDANA Whole-Brain Clusters** — Cluster-level pairwise analyses
15. **SP Subject Analysis** — Special participant analyses (Fig. 9)

# Setup: Imports, Configuration & Shared Utility Functions

All imports, path configuration, ROI definitions, and reusable analysis/plotting functions.
This cell must be run first — all subsequent cells depend on it.

In [ ]:
%matplotlib inline

# =============================================================================
# IMPORTS
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import os
import re
import glob
import warnings
from pathlib import Path
from scipy import stats

from pymer4.models import Lmer

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, Formula
import rpy2.robjects.numpy2ri as numpy2ri
from rpy2.robjects.packages import importr

# Activate R conversions
numpy2ri.activate()
pandas2ri.activate()

# Import R packages
lme4 = importr('lme4')
lmerTest = importr('lmerTest')
emmeans = importr('emmeans')
r_base = importr('base')
r_stats = importr('stats')

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================

# Paths — adjust project_root if needed
project_root = Path(os.path.expanduser("~/Documents/GitHub/multiecho-pilot"))
extractions_dir = project_root / 'derivatives' / 'extractions'
smoothness_path = project_root / 'code' / 'smoothness-all.csv'
output_dir = project_root / 'derivatives' / 'plots'
output_dir.mkdir(parents=True, exist_ok=True)

# Standard acquisition parameters
ACQ_PARAMS = ["mb1me1", "mb3me1", "mb6me1", "mb1me4", "mb3me4", "mb6me4"]

# Standard analysis parameters
TYPE_VALUE = "act"
DENOISE_VALUE = "base"

# ROI definitions
ROI_CONFIGS = {
    "3roi": {
        "rois": ["VSconstrained", "rFFA", "bilateralMotor"],
        "labels": {
            "VSconstrained": "Ventral Striatum",
            "rFFA": "Right FFA",
            "bilateralMotor": "Motor Cortex"
        },
        "ylabels": {
            "VSconstrained": "Reward>Punishment",
            "rFFA": "Face>Non-Face",
            "bilateralMotor": "Button Press>Non-Press"
        }
    },
    "motor_cerebellum": {
        "rois": ["bilateralMotor", "bilateralCerebellum"],
        "labels": {
            "bilateralMotor": "Motor Cortex",
            "bilateralCerebellum": "Cerebellum"
        },
        "ylabels": {
            "bilateralMotor": "Button Press>Non-Press",
            "bilateralCerebellum": "Button Press>Non-Press"
        }
    }
}

# Plot styling
ME_COLORS = {'me1': 'royalblue', 'me4': 'darkorange'}

# Factor maps for converting R factor integers back to strings
MB_MAP = {1: 'mb1', 2: 'mb3', 3: 'mb6'}
ME_MAP = {1: 'me1', 2: 'me4'}
HC_MAP = {1: '20', 2: '64'}

# Standard effect names and df for the 3-factor model (HC * MB * ME + smoothness)
STANDARD_EFFECT_NAMES = {
    'headcoil': 'Head Coil', 'mb': 'Multiband', 'me': 'Multi-echo',
    'smoothness': 'Smoothness',
    'headcoil:mb': 'Head Coil × Multiband', 'headcoil:me': 'Head Coil × Multi-echo',
    'mb:me': 'Multiband × Multi-echo', 'headcoil:mb:me': 'Head Coil × Multiband × Multi-echo'
}
STANDARD_DF_DICT = {
    'Head Coil': 1, 'Multiband': 2, 'Multi-echo': 1, 'Smoothness': 1,
    'Head Coil × Multiband': 2, 'Head Coil × Multi-echo': 1,
    'Multiband × Multi-echo': 2, 'Head Coil × Multiband × Multi-echo': 2
}

# Extended effect names/df for 4-factor model (adds ROI)
ROI_COMPARISON_EFFECT_NAMES = {
    **STANDARD_EFFECT_NAMES,
    'roi': 'ROI',
    'headcoil:roi': 'Head Coil × ROI', 'mb:roi': 'Multiband × ROI',
    'me:roi': 'Multi-echo × ROI',
    'headcoil:mb:roi': 'Head Coil × Multiband × ROI',
    'headcoil:me:roi': 'Head Coil × Multi-echo × ROI',
    'mb:me:roi': 'Multiband × Multi-echo × ROI',
    'headcoil:mb:me:roi': 'Head Coil × Multiband × Multi-echo × ROI'
}
ROI_COMPARISON_DF_DICT = {
    **STANDARD_DF_DICT,
    'ROI': 1,
    'Head Coil × ROI': 1, 'Multiband × ROI': 2, 'Multi-echo × ROI': 1,
    'Head Coil × Multiband × ROI': 2, 'Head Coil × Multi-echo × ROI': 1,
    'Multiband × Multi-echo × ROI': 2, 'Head Coil × Multiband × Multi-echo × ROI': 2
}

# =============================================================================
# SHARED UTILITY FUNCTIONS
# =============================================================================

def extract_roi_data(base_dir, type_value, img_value, mask_value, denoise_value,
                     valid_subjects, headcoil_mapping, acq_params=None):
    """
    Extract ROI data from .txt files for any image type (beta, zstat, varcope, etc.).

    Parameters
    ----------
    base_dir : str or Path
        Directory containing extraction .txt files.
    type_value : str
        Analysis type (e.g., 'act', 'ppi_seed-VS_thr5').
    img_value : str
        Image type -- used for filename filtering AND as the value column name.
    mask_value : str
        ROI mask name.
    denoise_value : str
        Denoising label.
    valid_subjects : list
        Subject IDs to include.
    headcoil_mapping : dict
        subject_id -> headcoil string mapping.
    acq_params : list, optional
        Acquisition labels to include. Defaults to ACQ_PARAMS.

    Returns
    -------
    pd.DataFrame or None
    """
    if acq_params is None:
        acq_params = ACQ_PARAMS

    pattern = re.compile(
        r"ts_sub-(\d+)_acq_([^_]+)_type-((?:act|ppi_seed-VS_thr5))_img-([^_]+)_mask-([^_]+)_denoise_([^\.]+)\.txt"
    )
    data_records = []
    file_paths = glob.glob(os.path.join(str(base_dir), "*.txt"))

    for file_path in file_paths:
        filename = os.path.basename(file_path)
        match = pattern.match(filename)
        if not match:
            continue

        sub_id, acq, file_type, img, mask, denoise = match.groups()

        if (sub_id not in valid_subjects or
            file_type != type_value or
            img != img_value or
            mask != mask_value or
            denoise != denoise_value or
            acq not in acq_params):
            continue

        try:
            with open(file_path, 'r') as f:
                value = float(f.read().strip())

            mb_match = re.search(r'(mb\d)', acq)
            me_match = re.search(r'(me\d)', acq)

            data_records.append({
                'subject': sub_id,
                'headcoil': headcoil_mapping.get(sub_id, None),
                'mb': mb_match.group(1) if mb_match else None,
                'me': me_match.group(1) if me_match else None,
                'acq_combined': acq,
                img_value: value
            })
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    if not data_records:
        return None

    df = pd.DataFrame(data_records)
    df['headcoil'] = pd.Categorical(df['headcoil'], categories=['20', '64'])
    df['mb'] = pd.Categorical(df['mb'], categories=['mb1', 'mb3', 'mb6'])
    df['me'] = pd.Categorical(df['me'], categories=['me1', 'me4'])
    df['subject'] = df['subject'].astype(str)
    df = df.dropna()
    return df


def load_smoothness_data(csv_path, valid_subjects, headcoil_mapping):
    """Load and process smoothness data with shift correction."""
    try:
        data = pd.read_csv(csv_path)
        data = data.rename(columns={data.columns[0]: 'path', 'Unnamed: 3': 'smoothness'})

        # Apply shift correction (smoothness value is offset by one row from its filepath)
        data['file_path'] = data['path'].shift(1)
        data = data[data['smoothness'].notnull() & data['file_path'].notnull()]

        data['subject'] = data['file_path'].str.extract(r'sub-(\d+)')
        data['acq'] = data['file_path'].str.extract(r'acq-(mb\dme\d)')
        data['mb'] = data['acq'].str[:3]
        data['me'] = data['acq'].str[3:]

        data = data[data['subject'].isin(valid_subjects)]
        data['headcoil'] = data['subject'].map(headcoil_mapping)
        data['acq_combined'] = data['mb'].astype(str) + data['me'].astype(str)
        data = data[['subject', 'headcoil', 'mb', 'me', 'acq_combined', 'smoothness']]

        data['headcoil'] = pd.Categorical(data['headcoil'], categories=['20', '64'])
        data['mb'] = pd.Categorical(data['mb'], categories=['mb1', 'mb3', 'mb6'])
        data['me'] = pd.Categorical(data['me'], categories=['me1', 'me4'])
        data = data.dropna()
        return data

    except Exception as e:
        print(f"Error processing smoothness data: {str(e)}")
        return None


def identify_complete_subjects(data_df, value_col, expected_acq=None):
    """
    Identify subjects with complete data across all expected acquisitions.

    Parameters
    ----------
    data_df : pd.DataFrame
    value_col : str
        Column to check for completeness.
    expected_acq : list, optional
        Acquisition labels to require. Defaults to ACQ_PARAMS.
    """
    if expected_acq is None:
        expected_acq = ACQ_PARAMS

    pivot = data_df.pivot_table(
        values=value_col, index='subject',
        columns='acq_combined', aggfunc='first'
    ).reindex(columns=expected_acq)
    return pivot.dropna().index.tolist()


def build_apa_table(model, anova_results, effect_names, df_dict):
    """
    Build an APA-style ANOVA table from pymer4 model results.

    Parameters
    ----------
    model : pymer4 Lmer model (fitted)
    anova_results : pd.DataFrame from model.anova()
    effect_names : dict mapping raw effect names to display names.
    df_dict : dict mapping display names to numerator df values.

    Returns
    -------
    pd.DataFrame
    """
    apa_data = []
    for effect in anova_results.index:
        if effect in ['(Intercept)', 'Residuals']:
            continue
        effect_name = effect_names.get(effect, effect)
        apa_data.append({
            'Effect': effect_name,
            'Sum Sq': anova_results.loc[effect, 'SS'] if 'SS' in anova_results.columns else np.nan,
            'Mean Sq': anova_results.loc[effect, 'MS'] if 'MS' in anova_results.columns else np.nan,
            'Num df': df_dict.get(effect_name, np.nan),
            'Den df': anova_results.loc[effect, 'DenomDF'] if 'DenomDF' in anova_results.columns else np.nan,
            'F': anova_results.loc[effect, 'F-stat'] if 'F-stat' in anova_results.columns else np.nan,
            'p': anova_results.loc[effect, 'P-val'] if 'P-val' in anova_results.columns else np.nan,
            'Partial eta_sq': np.nan
        })

    # Compute partial eta-squared
    try:
        residual_var = model.ranef_var.loc[model.ranef_var['Name'] == '', 'Var'].iloc[0]
    except (KeyError, IndexError):
        residual_var = model.ranef_var.iloc[1]['Var']

    for i, row in enumerate(apa_data):
        ss_effect = row['Sum Sq']
        denom_df = row['Den df']
        if pd.notna(ss_effect) and pd.notna(residual_var) and pd.notna(denom_df):
            apa_data[i]['Partial eta_sq'] = ss_effect / (ss_effect + (residual_var * denom_df))

    apa_table = pd.DataFrame(apa_data)
    apa_table['Sum Sq'] = apa_table['Sum Sq'].round(2)
    apa_table['Mean Sq'] = apa_table['Mean Sq'].round(2)
    apa_table['Num df'] = apa_table['Num df'].astype('Int64').fillna(pd.NA)
    apa_table['Den df'] = apa_table['Den df'].round(2)
    apa_table['F'] = apa_table['F'].round(2)
    apa_table['p'] = apa_table['p'].apply(
        lambda x: '< .001' if pd.notna(x) and x < 0.001 else f'{x:.3f}' if pd.notna(x) else 'N/A'
    )
    apa_table['Partial eta_sq'] = apa_table['Partial eta_sq'].round(3)
    return apa_table


def run_lme_analysis(merged_data, img_value, roi_name):
    """
    Run LME analysis: img_value ~ headcoil * mb * me + smoothness + (1 | subject)

    Returns (apa_table, model_data).
    """
    model_data = merged_data.copy()
    model = Lmer(f'{img_value} ~ headcoil * mb * me + smoothness + (1 | subject)',
                 data=model_data)
    model.fit()
    anova_results = model.anova()
    apa_table = build_apa_table(model, anova_results, STANDARD_EFFECT_NAMES, STANDARD_DF_DICT)
    return apa_table, model_data


def compute_emms(data, img_value, roi_name):
    """
    Compute estimated marginal means via R's emmeans package.

    Parameters
    ----------
    data : pd.DataFrame
        Merged data with subject, headcoil, mb, me, smoothness, and the DV column.
    img_value : str
        Name of the dependent variable column (e.g., 'beta', 'zstat').
    roi_name : str
        ROI identifier (used for R variable naming).

    Returns
    -------
    pd.DataFrame with columns: MB, ME, HC, emmean, SE, df, lower.CL, upper.CL
    """
    # Use a generic DV name in R to avoid issues with special characters
    dv_name = 'DV'

    data_r = data.rename(columns={
        'subject': 'Subj', 'headcoil': 'HC', 'mb': 'MB', 'me': 'ME',
        img_value: dv_name
    })
    data_r = data_r.dropna(subset=[dv_name, 'MB', 'ME', 'HC', 'Subj', 'smoothness'])
    for col in ['Subj', 'MB', 'ME', 'HC']:
        data_r[col] = data_r[col].astype(str)

    r_df_temp = pandas2ri.py2rpy(data_r)
    r_df_name = f"temp_emm_{roi_name}"
    ro.globalenv[r_df_name] = r_df_temp

    ro.r('levels_MB <- c("mb1", "mb3", "mb6"); levels_ME <- c("me1", "me4"); levels_HC <- c("20", "64")')
    r_df = ro.r(f'transform({r_df_name}, MB=factor(MB, levels=levels_MB), ME=factor(ME, levels=levels_ME), HC=factor(HC, levels=levels_HC))')
    del ro.globalenv[r_df_name]

    formula = Formula(f'{dv_name} ~ HC * MB * ME + smoothness + (1 | Subj)')
    model = lme4.lmer(formula, data=r_df)
    emm = emmeans.emmeans(model, 'MB', by=ro.StrVector(['HC', 'ME']))
    emm_df = pandas2ri.rpy2py(ro.r('as.data.frame')(emm))

    # Convert R factor integers back to strings
    if emm_df['MB'].dtype in ['int64', 'int32', 'float64']:
        emm_df['MB'] = emm_df['MB'].map(MB_MAP)
    if emm_df['ME'].dtype in ['int64', 'int32', 'float64']:
        emm_df['ME'] = emm_df['ME'].map(ME_MAP)
    if emm_df['HC'].dtype in ['int64', 'int32', 'float64']:
        emm_df['HC'] = emm_df['HC'].map(HC_MAP)

    return emm_df


def plot_emm_panel(ax, emm_df, headcoil, n_subjects, me_colors=None):
    """
    Plot EMM data for a single headcoil on a single axis.

    Parameters
    ----------
    ax : matplotlib Axes
    emm_df : pd.DataFrame with MB, ME, HC, emmean, SE columns
    headcoil : str ('20' or '64')
    n_subjects : int
    me_colors : dict, optional
    """
    if me_colors is None:
        me_colors = ME_COLORS

    coil_data = emm_df[emm_df['HC'] == headcoil]
    if coil_data.empty:
        ax.set_title(f"{headcoil}-Channel (n=0)", fontsize=38)
        return

    x_positions = {'mb1': 0, 'mb3': 1, 'mb6': 2}
    jitter = 0.05
    me_jitter = {'me1': -jitter, 'me4': jitter}

    for me in ['me1', 'me4']:
        me_data = coil_data[coil_data['ME'] == me]
        me_data = me_data.sort_values('MB', key=lambda x: x.str.extract(r'mb(\d+)')[0].astype(int))
        x_vals = [x_positions[mb] + me_jitter[me] for mb in me_data['MB']]

        ax.plot(x_vals, me_data['emmean'], marker='o', color=me_colors[me],
                label=me, linewidth=5, markersize=15)
        ax.errorbar(x_vals, me_data['emmean'], yerr=me_data['SE'],
                    fmt='none', color=me_colors[me], capsize=8, capthick=4, elinewidth=3)

    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['mb1', 'mb3', 'mb6'])
    ax.set_title(f"{headcoil}-Channel (n={n_subjects})", fontsize=38)
    ax.tick_params(axis='both', which='major', labelsize=29)


def compute_emm_ylimits(emm_results_dict):
    """Compute shared y-limits across a dict of {roi: emm_df}."""
    all_y, all_e = [], []
    for emm_df in emm_results_dict.values():
        all_y.extend(emm_df['emmean'].values)
        all_e.extend(emm_df['SE'].values)
    if all_y:
        y_max = max(v + e for v, e in zip(all_y, all_e))
        y_min = min(v - e for v, e in zip(all_y, all_e))
        margin = (y_max - y_min) * 0.1
        return (y_min - margin, y_max + margin)
    return (-1, 1)


def generate_emm_grid_plot(rois, roi_labels, roi_ylabels, roi_results, emm_results,
                           img_value, plot_filename, fig_ylabel=None, legend_loc='lower right'):
    """
    Generate the standard N_ROI x 2-headcoil EMM grid plot.

    Parameters
    ----------
    rois : list of str
    roi_labels : dict
    roi_ylabels : dict
    roi_results : dict with keys per ROI containing 'n_20ch', 'n_64ch'
    emm_results : dict {roi: emm_df}
    img_value : str ('beta' or 'zstat')
    plot_filename : str or Path
    fig_ylabel : str, optional
    legend_loc : str
    """
    plt.rcParams.update({'font.size': 38})
    n_rois = len(rois)
    fig, axes = plt.subplots(n_rois, 2, figsize=(16, 8 * n_rois))

    # Handle single-ROI case (axes won't be 2D)
    if n_rois == 1:
        axes = axes.reshape(1, -1)

    y_limits = compute_emm_ylimits(emm_results)

    for roi_idx, roi in enumerate(rois):
        if roi not in emm_results:
            continue

        emm_df = emm_results[roi]
        roi_data = roi_results[roi]

        plot_emm_panel(axes[roi_idx, 0], emm_df, '20', roi_data['n_20ch'])
        axes[roi_idx, 0].set_ylim(y_limits)

        plot_emm_panel(axes[roi_idx, 1], emm_df, '64', roi_data['n_64ch'])
        axes[roi_idx, 1].set_ylim(y_limits)

        axes[roi_idx, 0].set_ylabel(roi_ylabels.get(roi, ''), fontsize=29)

    axes[0, 1].legend(title='Multi-echo', fontsize=29, title_fontsize=29, loc=legend_loc)

    if fig_ylabel is None:
        label_map = {'beta': 'Betas', 'zstat': 'ZStats'}
        fig_ylabel = f'Task-Relevant Activation (Estimated Marginal Means, {label_map.get(img_value, img_value)})'
    fig.text(0.001, 0.5, fig_ylabel, va='center', rotation='vertical', fontsize=38)
    fig.supxlabel('Multiband Factor', fontsize=38, y=0.04)

    plt.tight_layout()
    fig.subplots_adjust(top=0.88, left=0.12, hspace=0.35)

    for roi_idx, roi in enumerate(rois):
        if roi in emm_results:
            bbox = axes[roi_idx, 0].get_position()
            fig.text(0.5, bbox.y1 + 0.03, roi_labels[roi],
                     ha='center', fontsize=42, fontweight='bold', transform=fig.transFigure)

    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"Plot saved to '{plot_filename}'")
    plt.show()

    return fig


def load_and_merge_roi_data(rois, img_value, smoothness_data, complete_smoothness,
                             valid_subjects, headcoil_mapping):
    """
    Load ROI data, merge with smoothness, and identify complete subjects for each ROI.

    Returns roi_results dict.
    """
    roi_results = {}

    for roi in rois:
        print(f"\nProcessing {roi}...")
        roi_data = extract_roi_data(extractions_dir, TYPE_VALUE, img_value,
                                     roi, DENOISE_VALUE, valid_subjects, headcoil_mapping)
        if roi_data is None:
            print(f"  Warning: No data found for {roi}")
            continue

        complete_roi = identify_complete_subjects(roi_data, img_value)
        print(f"  Complete {roi} subjects: {len(complete_roi)}")

        common_complete = sorted(list(set(complete_roi) & set(complete_smoothness)))
        print(f"  Common complete subjects: {len(common_complete)}")

        if roi == "bilateralMotor" and len(common_complete) < len(complete_smoothness):
            missing = sorted(list(set(complete_smoothness) - set(complete_roi)))
            print(f"  Subjects missing from Motor Cortex: {missing}")

        if not common_complete:
            print(f"  No common complete subjects for {roi}")
            continue

        roi_filtered = roi_data[roi_data['subject'].isin(common_complete)].copy()
        smoothness_filtered = smoothness_data[smoothness_data['subject'].isin(common_complete)].copy()

        merged = pd.merge(
            roi_filtered[['subject', 'headcoil', 'mb', 'me', img_value]],
            smoothness_filtered[['subject', 'mb', 'me', 'smoothness']],
            on=['subject', 'mb', 'me'], how='inner'
        ).dropna()

        roi_results[roi] = {
            'data': merged,
            'n_subjects': merged['subject'].nunique(),
            'n_20ch': len(merged[merged['headcoil'] == '20']['subject'].unique()),
            'n_64ch': len(merged[merged['headcoil'] == '64']['subject'].unique())
        }

    return roi_results


def run_full_roi_analysis(img_value, roi_config_key, valid_subjects, headcoil_mapping,
                          smoothness_data, complete_smoothness, legend_loc='lower right'):
    """
    Run the complete ROI analysis pipeline for a given image type.

    Parameters
    ----------
    img_value : str ('beta' or 'zstat')
    roi_config_key : str (key into ROI_CONFIGS)
    valid_subjects, headcoil_mapping : from demographics
    smoothness_data, complete_smoothness : from smoothness loading
    legend_loc : str

    Returns
    -------
    roi_results, emm_results
    """
    config = ROI_CONFIGS[roi_config_key]
    rois = config['rois']
    roi_labels = config['labels']
    roi_ylabels = config['ylabels']

    label_map = {'beta': 'Beta', 'zstat': 'Z-Stat'}
    label = label_map.get(img_value, img_value)

    print("=" * 80)
    print(f"MULTI-ROI {label.upper()} ANALYSIS PIPELINE WITH SMOOTHNESS COVARIATE AND EMMs")
    print("=" * 80)
    print(f"Processing {img_value} values for ROIs: {', '.join(rois)}")

    # Part 1: Load data
    print("\n" + "=" * 80)
    print("PART 1: LOADING DATA AND IDENTIFYING COMPLETE SUBJECTS")
    print("=" * 80)

    roi_results = load_and_merge_roi_data(
        rois, img_value, smoothness_data, complete_smoothness,
        valid_subjects, headcoil_mapping
    )

    # Part 2: LME analyses
    print("\n" + "=" * 80)
    print("PART 2: LINEAR MIXED EFFECTS ANALYSES")
    print("=" * 80)

    emm_results = {}
    for roi in rois:
        if roi not in roi_results:
            continue

        print(f"\n=== {roi_labels[roi]} ({roi}) ===")
        rd = roi_results[roi]
        print(f"N = {rd['n_subjects']} subjects ({rd['n_20ch']} 20-ch, {rd['n_64ch']} 64-ch)")

        apa_table, _ = run_lme_analysis(rd['data'], img_value, roi)
        print(f"\nAPA-Style ANOVA Table:")
        print(f"Model: {img_value} ~ headcoil * mb * me + smoothness + (1 | subject)")
        print(f"Data: {roi_labels[roi]} ROI\n")
        print(apa_table.to_string(index=False))

        anova_file = output_dir / f'{roi}_{img_value}_lme_anova_with_smoothness.csv'
        apa_table.to_csv(anova_file, index=False)
        print(f"\nSaved to '{anova_file.name}'")

        emm_df = compute_emms(rd['data'], img_value, roi)
        emm_results[roi] = emm_df

    # Part 3: Visualization
    print("\n" + "=" * 80)
    print("PART 3: GENERATING COMBINED EMM PLOTS")
    print("=" * 80)

    plot_file = output_dir / f'multi_roi_{img_value}_emm_plot.png'
    generate_emm_grid_plot(rois, roi_labels, roi_ylabels, roi_results, emm_results,
                           img_value, plot_file, legend_loc=legend_loc)

    # Summary
    print("\n" + "=" * 80)
    print("ANALYSIS COMPLETE - SUMMARY")
    print("=" * 80)
    print(f"ROIs analyzed: {', '.join(rois)}")
    for roi in rois:
        if roi in roi_results:
            r = roi_results[roi]
            print(f"  {roi_labels[roi]}: {r['n_subjects']} total ({r['n_20ch']} 20-ch, {r['n_64ch']} 64-ch)")
    print("=" * 80)

    return roi_results, emm_results


print("Setup complete. All imports, configuration, and utility functions loaded.")


# Subject Demographics & Data Quality Analysis

Load demographics, apply quality filters, remove motion outliers, report final sample.

In [ ]:
# =============================================================================
# SUBJECT DEMOGRAPHICS & DATA QUALITY FILTERING
# =============================================================================
# Produces: main_df_final, sp_df
# Dependencies: project_root, output_dir

import seaborn as sns

demographics_path = project_root / 'code' / 'multiecho-pilot_Demographics.xlsx'
outlier_path = project_root / 'derivatives' / 'Task-sharedreward_Level-Acq_Outlier-info_mriqc-0.16.1.tsv'

# --- 1. Data Loading and Separation ---
df = pd.read_excel(demographics_path)
sp_df = df[df['spSubs'] == 1].copy()
main_df = df[df['spSubs'] == 0].copy()

# --- 2. Sequential Filtering ---
# Filter 1: SixRuns
main_df_filtered = main_df[main_df['SixRuns'] != 0].copy()
subjects_removed_sixruns = len(main_df) - len(main_df_filtered)

# Filter 2: Motion Outliers
outlier_df = pd.read_csv(outlier_path, sep='\t')
outlier_runs = outlier_df[outlier_df['outlier_acq_Custom1'] == True]

if len(outlier_runs) > 0:
    outlier_subjects_raw = outlier_runs['Sub'].unique()
    outlier_subjects_clean = [s.replace('sub-', '') for s in outlier_subjects_raw if s.startswith('sub-')]
    main_df_filtered['Subject_str'] = main_df_filtered['Subject'].astype(str)
    main_df_final = main_df_filtered[~main_df_filtered['Subject_str'].isin(outlier_subjects_clean)].copy()
    main_df_final = main_df_final.drop(columns=['Subject_str'])
    subjects_removed_outliers = len(main_df_filtered) - len(main_df_final)
else:
    main_df_final = main_df_filtered.copy()
    subjects_removed_outliers = 0

# --- 3. Nature-Quality Visualization ---
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial']})
fig, ax = plt.subplots(figsize=(14, 10))

# Custom Professional Palette
# Headcoil 64: Deep Blue, Headcoil 20: Muted Rose/Burgundy
colors = {64: '#2E86AB', 20: '#A23B72'}
hc_labels = {64: '64-channel', 20: '20-channel'}

# Plot Distributions using Seaborn for professional smoothing
for hc in [20, 64]:
    data = main_df_final[main_df_final['Headcoil'] == hc]['Age']
    n = len(data)
    
    # Histogram Step (filled)
    sns.histplot(data, bins=10, kde=True, ax=ax, color=colors[hc], 
                 label=f'{hc_labels[hc]} (n={n})', 
                 element="step", fill=True, alpha=0.3, 
                 line_kws={'linewidth': 5}, 
                 edgecolor=colors[hc])
    
    # Rug plot (Individual data points)
    sns.rugplot(data, ax=ax, color=colors[hc], height=0.05, linewidth=2, alpha=0.8)

# Calculate Stats for the legend/annotation
stats_text = []
for hc in [20, 64]:
    data = main_df_final[main_df_final['Headcoil'] == hc]['Age']
    stats_text.append(f"{hc_labels[hc]}: M={data.mean():.1f}, SD={data.std():.1f}")

# Formatting Axes
ax.set_xlabel('Age (Years)', fontsize=34, fontweight='bold', labelpad=15)
ax.set_ylabel('Frequency / Density', fontsize=34, fontweight='bold', labelpad=15)
ax.set_title('Participant Age Distribution by Head Coil', fontsize=38, fontweight='bold', pad=25)

# Clean up Spines
sns.despine(trim=True) # Nature style: offset axes
ax.tick_params(axis='both', which='major', labelsize=28)

# Legend with specific Nature formatting
leg = ax.legend(fontsize=24, loc='upper right', frameon=False)
for line in leg.get_lines():
    line.set_linewidth(10)

# Add a subtle annotation box for the mean/SD stats
ax.text(0.05, 0.95, "\n".join(stats_text), transform=ax.transAxes, 
        fontsize=22, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.5, edgecolor='none'))

plt.tight_layout()

# Save output
save_path = output_dir / 'age_distribution_nature_style.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"\nFinal Population: N={len(main_df_final)}")
print(f"Plot saved to: {save_path}")

# Load Shared Data Dependencies

Derive subject lists from demographics and load smoothness data.
These are used by all subsequent ROI analyses.

In [ ]:
# Derive valid_subjects and headcoil_mapping from demographics
valid_subjects = main_df_final['Subject'].astype(str).tolist()
headcoil_mapping = dict(zip(main_df_final['Subject'].astype(str),
                           main_df_final['Headcoil'].astype(int).astype(str)))

print(f"Valid subjects: {len(valid_subjects)}")

# Load smoothness data (shared across all ROI analyses)
smoothness_data = load_smoothness_data(smoothness_path, valid_subjects, headcoil_mapping)
if smoothness_data is None:
    raise Exception("Failed to load smoothness data")

complete_smoothness = identify_complete_subjects(smoothness_data, 'smoothness')
print(f"Complete smoothness subjects: {len(complete_smoothness)}")


# Spatial Smoothness Analysis (Fig. 3)
LME analysis of pre- and post-smoothing spatial smoothness across acquisition parameters. 2×2 bar plot output.

In [ ]:
# =============================================================================
# SPATIAL SMOOTHNESS ANALYSIS — PRE AND POST SMOOTHING (Fig. 3)
# =============================================================================
# Dependencies: project_root, output_dir, valid_subjects, headcoil_mapping,
#               load_smoothness_data, identify_complete_subjects (from Setup)
#
# NOTE: This analysis uses sum-to-zero headcoil encoding (20ch=-0.5, 64ch=0.5)
# and a different partial eta-squared formula than the activation analyses.
# These are intentional modeling choices for the smoothness DV.
#
# This version uses jittered points + cell mean markers + SE error bars
# instead of bar charts.

print("=" * 80)
print("SMOOTHNESS ANALYSIS PIPELINE - PRE AND POST SMOOTHING")
print("=" * 80)

# --- Paths ---
csv_path_pre = project_root / 'code' / 'smoothness-all-zero.csv'
csv_path_post = project_root / 'code' / 'smoothness-all.csv'

# --- Output files ---
complete_subjects_file_pre = output_dir / 'complete_subjects_smoothness_pre_with_headcoil.csv'
complete_subjects_file_post = output_dir / 'complete_subjects_smoothness_post_with_headcoil.csv'
anova_table_file_pre = output_dir / 'smoothness_pre_lme_anova_complete_subjects.csv'
anova_table_file_post = output_dir / 'smoothness_post_lme_anova_complete_subjects.csv'
combined_plot_file = output_dir / 'smoothness_combined_emm_style_plot.png'

# Nature-quality color scheme
ME_COLORS_NATURE = {'me1': '#4A90A4', 'me4': '#E07B39'}

# =============================================================================
# LOCAL FUNCTIONS (unique to this analysis)
# =============================================================================

def run_smoothness_lme(data_complete, label):
    """
    Run LME for smoothness with sum-to-zero headcoil coding.
    Uses a different eta-squared formula than the activation analyses.
    """
    print(f"\nRunning LME analysis for {label} data")
    print(f"Observations: {len(data_complete)}, Subjects: {data_complete['subject'].nunique()}")

    model_data = data_complete.copy()
    model_data['headcoil'] = model_data['headcoil'].cat.codes - 0.5  # 20=-0.5, 64=0.5

    model = Lmer('smoothness ~ headcoil * mb * me + (1 | subject)', data=model_data)
    model.fit()
    anova_table = model.anova()

    effect_map = {
        'headcoil': 'Head Coil', 'mb': 'Multiband', 'me': 'Multi-echo',
        'headcoil:mb': 'Head Coil x Multiband', 'headcoil:me': 'Head Coil x Multi-echo',
        'mb:me': 'Multiband x Multi-echo', 'headcoil:mb:me': 'Head Coil x Multiband x Multi-echo'
    }
    df_dict = {
        'Head Coil': 1, 'Multiband': 2, 'Multi-echo': 1,
        'Head Coil x Multiband': 2, 'Head Coil x Multi-echo': 1,
        'Multiband x Multi-echo': 2, 'Head Coil x Multiband x Multi-echo': 2
    }

    apa_data = []
    for effect in anova_table.index:
        if effect in ['(Intercept)', 'Residuals']:
            continue
        effect_name = effect_map.get(effect, effect)
        apa_data.append({
            'Effect': effect_name,
            'Sum Sq': anova_table.loc[effect, 'SS'] if 'SS' in anova_table.columns else np.nan,
            'Mean Sq': anova_table.loc[effect, 'MS'] if 'MS' in anova_table.columns else np.nan,
            'Num df': df_dict.get(effect_name, np.nan),
            'Den df': anova_table.loc[effect, 'DenomDF'] if 'DenomDF' in anova_table.columns else np.nan,
            'F': anova_table.loc[effect, 'F-stat'] if 'F-stat' in anova_table.columns else np.nan,
            'p': anova_table.loc[effect, 'P-val'] if 'P-val' in anova_table.columns else np.nan,
            'Partial eta_sq': np.nan
        })

    # Partial eta-squared (smoothness formula: uses total residual SS)
    try:
        residual_var = model.ranef_var.loc[model.ranef_var['Name'] == '', 'Var'].iloc[0]
    except (KeyError, IndexError):
        residual_var = model.ranef_var.iloc[1]['Var']

    n_obs = len(model_data)
    n_fixed = sum(df_dict.values())
    n_subj = model_data['subject'].nunique()
    ss_residual = residual_var * (n_obs - n_fixed - n_subj)

    for i, row in enumerate(apa_data):
        ss_effect = row['Sum Sq']
        if pd.notna(ss_effect):
            apa_data[i]['Partial eta_sq'] = ss_effect / (ss_effect + ss_residual)

    apa_table = pd.DataFrame(apa_data)
    apa_table['Sum Sq'] = apa_table['Sum Sq'].round(3)
    apa_table['Mean Sq'] = apa_table['Mean Sq'].round(3)
    apa_table['Num df'] = apa_table['Num df'].astype('Int64')
    apa_table['Den df'] = apa_table['Den df'].round(2)
    apa_table['F'] = apa_table['F'].round(2)
    apa_table['Partial eta_sq'] = apa_table['Partial eta_sq'].round(3)
    apa_table['p'] = apa_table['p'].apply(lambda p: f"{p:.4f}" if pd.notna(p) else '')

    return apa_table


def build_complete_subjects_table(data, complete_list, headcoil_mapping):
    """Build a complete-subjects pivot table with headcoil info for CSV output."""
    expected_cols = ['mb1me1', 'mb3me1', 'mb6me1', 'mb1me4', 'mb3me4', 'mb6me4']
    data_c = data[data['subject'].isin(complete_list)].copy()
    data_c['mb_me'] = data_c['mb'].astype(str) + data_c['me'].astype(str)

    pivot = data_c.pivot_table(values='smoothness', index='subject',
                                columns='mb_me', aggfunc='mean'
                                ).reindex(columns=expected_cols).round(3)

    hc_info = data_c[['subject', 'headcoil']].drop_duplicates().set_index('subject')
    pivot = pivot.join(hc_info)
    pivot = pivot[['headcoil'] + expected_cols]
    return pivot.sort_index()


def process_data_for_plotting(data, value_column='smoothness'):
    """Calculate mean and standard error by multiband, multi-echo, and headcoil."""
    if data is None or data.empty:
        return pd.DataFrame()
    data = data.dropna(subset=['mb', 'me', 'headcoil', value_column])
    if data.empty:
        return pd.DataFrame()

    agg_data = data.groupby(['mb', 'me', 'headcoil'], observed=True).agg({
        value_column: 'mean', 'subject': 'nunique'
    }).reset_index()

    std_error = data.groupby(['mb', 'me', 'headcoil'], observed=True)[value_column].apply(
        lambda x: x.std() / np.sqrt(len(x))
    ).reset_index()
    std_error.columns = ['mb', 'me', 'headcoil', 'se']

    result = pd.merge(agg_data, std_error, on=['mb', 'me', 'headcoil'])
    result.columns = ['mb', 'me', 'headcoil', value_column, 'n_subjects', 'se']
    return result


def process_smoothness_dataset(csv_path, label, complete_subjects_file, anova_file):
    """Load, filter, save table, run LME for one smoothness dataset."""
    print(f"\n{'=' * 80}")
    print(f"{label.upper()} DATA ANALYSIS")
    print(f"{'=' * 80}")

    # Reuse shared loader (same CSV format for pre and post)
    data = load_smoothness_data(csv_path, valid_subjects, headcoil_mapping)
    if data is None:
        raise Exception(f"Failed to load {label} data from {csv_path}")

    # Identify complete subjects (shared function returns a list)
    complete_list = identify_complete_subjects(data, 'smoothness')

    # Build and save complete-subjects table
    table = build_complete_subjects_table(data, complete_list, headcoil_mapping)
    hc_counts = table['headcoil'].value_counts().reindex(['20', '64'], fill_value=0)

    print(f"\n{label} complete subjects: {len(table)}")
    print(f"  - 20-channel: {hc_counts['20']}")
    print(f"  - 64-channel: {hc_counts['64']}")

    table.to_csv(complete_subjects_file)
    print(f"Saved to '{complete_subjects_file.name}'")

    # Filter to complete subjects
    data_complete = data[data['subject'].isin(complete_list)].copy()
    
    # Ensure categorical columns
    data_complete['mb'] = pd.Categorical(data_complete['mb'], categories=['mb1', 'mb3', 'mb6'], ordered=True)
    data_complete['me'] = pd.Categorical(data_complete['me'], categories=['me1', 'me4'], ordered=True)
    data_complete['headcoil'] = pd.Categorical(data_complete['headcoil'], categories=['20', '64'], ordered=True)

    # Run LME
    apa_table = run_smoothness_lme(data_complete, label)
    print(f"\nAPA-Style ANOVA Table for {label} Data:")
    print(apa_table.to_string(index=False))
    apa_table.to_csv(anova_file, index=False)

    return data_complete, table, hc_counts


def plot_smoothness_emm_style(ax, raw_data, summary_data, n_subjects_per_coil, headcoil, 
                               y_limits, me_colors, show_legend=False):
    """
    Plot smoothness with jittered individual points + cell mean markers + SE error bars.
    Matches Nature-quality EMM-style from other figures.
    """
    coil_raw = raw_data[raw_data['headcoil'] == headcoil] if raw_data is not None else None
    coil_summary = summary_data[summary_data['headcoil'] == headcoil] if summary_data is not None else None
    n_subjects = n_subjects_per_coil.get(headcoil, 0)
    
    if coil_summary is None or coil_summary.empty:
        ax.set_title(f"{headcoil}-ch (n=0)", fontsize=12)
        ax.set_ylim(y_limits)
        return
    
    x_positions = {'mb1': 0, 'mb3': 1, 'mb6': 2}
    jitter_mean = 0.08
    jitter_raw = 0.12
    me_offsets = {'me1': -jitter_mean, 'me4': jitter_mean}
    
    # --- Individual data points (behind means) ---
    #if coil_raw is not None and len(coil_raw) > 0:
    #    np.random.seed(42)
    #    for me in ['me1', 'me4']:
    #        me_raw = coil_raw[coil_raw['me'] == me]
    #        for mb in ['mb1', 'mb3', 'mb6']:
    #            mb_me_data = me_raw[me_raw['mb'] == mb]['smoothness'].dropna()
    #            if len(mb_me_data) > 0:
    #                x_base = x_positions[mb] + me_offsets[me]
    #                jitter_vals = np.random.uniform(-jitter_raw, jitter_raw, size=len(mb_me_data))
    #                ax.scatter(
    #                    np.full(len(mb_me_data), x_base) + jitter_vals,
    #                    mb_me_data.values,
    #                    color=me_colors[me], s=20, alpha=0.25,
    #                    edgecolors='none', zorder=1
    #                )
    
    # --- Mean lines connecting MB levels ---
    for me in ['me1', 'me4']:
        me_data = coil_summary[coil_summary['me'] == me].copy()
        if me_data.empty:
            continue
        me_data = me_data.sort_values('mb', key=lambda x: x.astype(str).str.extract(r'mb(\d+)')[0].astype(int))
        x_vals = [x_positions[mb] + me_offsets[me] for mb in me_data['mb']]
        
        ax.plot(x_vals, me_data['smoothness'], 
                color=me_colors[me], linewidth=2, linestyle='-', zorder=3,
                label=me if show_legend else None)
        
        ax.errorbar(x_vals, me_data['smoothness'], yerr=me_data['se'],
                    fmt='o', color=me_colors[me], markersize=7,
                    capsize=3, capthick=1.5, elinewidth=1.5,
                    markeredgecolor='white', markeredgewidth=0.8, zorder=4)
    
    # --- Axis formatting ---
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['1', '3', '6'], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.set_title(f"{headcoil}-ch (n={n_subjects})", fontsize=12, fontweight='normal')
    ax.set_ylim(y_limits)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_linewidth(1)
    ax.grid(False)
    
    if show_legend:
        ax.legend(title='ME', fontsize=9, title_fontsize=9, loc='upper right',
                  framealpha=0.9, edgecolor='none')


# =============================================================================
# PART 1 & 2: PRE AND POST SMOOTHING DATA
# =============================================================================

data_complete_pre, table_pre, hc_pre = process_smoothness_dataset(
    csv_path_pre, "Pre-smoothing", complete_subjects_file_pre, anova_table_file_pre
)
data_complete_post, table_post, hc_post = process_smoothness_dataset(
    csv_path_post, "Post-smoothing", complete_subjects_file_post, anova_table_file_post
)

# =============================================================================
# PART 3: COMBINED VISUALIZATION
# =============================================================================

print(f"\n{'=' * 80}")
print("PART 3: GENERATING COMBINED EMM-STYLE PLOTS")
print(f"{'=' * 80}")

smoothness_processed_pre = process_data_for_plotting(data_complete_pre)
smoothness_processed_post = process_data_for_plotting(data_complete_post)

n_per_coil_pre = data_complete_pre.groupby('headcoil', observed=True)['subject'].nunique().to_dict()
n_per_coil_post = data_complete_post.groupby('headcoil', observed=True)['subject'].nunique().to_dict()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

y_limits = (3.5, 5.5)
panel_labels = ['a', 'b', 'c', 'd']

# Add panel labels BEFORE tight_layout
for idx, ax in enumerate(axes.flat):
    ax.text(-0.12, 1.08, panel_labels[idx], transform=ax.transAxes,
            fontsize=16, fontweight='bold', va='bottom', ha='left')

# Top row: Pre-smoothing (20-ch, 64-ch)
plot_smoothness_emm_style(axes[0, 0], data_complete_pre, smoothness_processed_pre, 
                           n_per_coil_pre, '20', y_limits, ME_COLORS_NATURE, show_legend=False)
plot_smoothness_emm_style(axes[0, 1], data_complete_pre, smoothness_processed_pre, 
                           n_per_coil_pre, '64', y_limits, ME_COLORS_NATURE, show_legend=True)

# Bottom row: Post-smoothing (20-ch, 64-ch)
plot_smoothness_emm_style(axes[1, 0], data_complete_post, smoothness_processed_post, 
                           n_per_coil_post, '20', y_limits, ME_COLORS_NATURE, show_legend=False)
plot_smoothness_emm_style(axes[1, 1], data_complete_post, smoothness_processed_post, 
                           n_per_coil_post, '64', y_limits, ME_COLORS_NATURE, show_legend=False)

# Y-axis labels with row identifiers
axes[0, 0].set_ylabel('Pre-smoothing\nSmoothness (mm)', fontsize=13)
axes[1, 0].set_ylabel('Post-smoothing\nSmoothness (mm)', fontsize=13)

plt.tight_layout()
fig.subplots_adjust(top=0.94, hspace=0.35, left=0.14)

fig.text(0.55, 0.02, 'Multiband Factor', ha='center', fontsize=13)

plt.savefig(combined_plot_file, dpi=300, bbox_inches='tight')
print(f"Combined plot saved to '{combined_plot_file}'")
plt.show()

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{'=' * 80}")
print("ANALYSIS COMPLETE - SUMMARY")
print(f"{'=' * 80}")
print(f"\nPre-smoothing: {len(table_pre)} complete subjects "
      f"({hc_pre.get('20', 0)} 20-ch, {hc_pre.get('64', 0)} 64-ch)")
print(f"Post-smoothing: {len(table_post)} complete subjects "
      f"({hc_post.get('20', 0)} 20-ch, {hc_post.get('64', 0)} 64-ch)")
print(f"\nFiles generated in {output_dir}:")
for f in [complete_subjects_file_pre, complete_subjects_file_post,
          anova_table_file_pre, anova_table_file_post, combined_plot_file]:
    print(f"  - {f.name}")
print("=" * 80)

# Whole-Brain TSNR Analysis (Fig. 4)
LME analysis of median TSNR with smoothness covariate. Bar plot by MB factor, headcoil, and ME condition.

In [ ]:
# =============================================================================
# WHOLE-BRAIN TSNR ANALYSIS WITH SMOOTHNESS COVARIATE (Fig. 4)
# =============================================================================
# Nature-quality 1×2 grid: 20-ch + 64-ch panels.
# EMM-style plots with individual subject points + model-based EMM means/SEs.
#
# Dependencies: project_root, output_dir, valid_subjects, headcoil_mapping,
#               smoothness_data, complete_smoothness (from Shared Deps cell),
#               identify_complete_subjects, compute_emms, MB_MAP, ME_MAP, HC_MAP (from Setup)

print("=" * 80)
print("WHOLE-BRAIN TSNR ANALYSIS PIPELINE WITH SMOOTHNESS COVARIATE")
print("=" * 80)

# --- Config ---
IMG_VALUE_TSNR = "tsnrMedian"

# Nature-quality color scheme
ME_COLORS_NATURE = {'me1': '#4A90A4', 'me4': '#E07B39'}

# --- Paths & output files ---
tsnr_path = project_root / 'code' / 'outputs' / 'tables' / 'combined_tsnr_coil_output.csv'
complete_subjects_file = output_dir / 'complete_subjects_tsnr_common_with_headcoil.csv'
anova_table_file = output_dir / 'tsnr_lme_anova_with_smoothness.csv'
tsnr_plot_file = output_dir / 'wholebrain_tsnr_emm_plot_publication.png'

# =============================================================================
# LOCAL FUNCTIONS
# =============================================================================

def load_and_process_tsnr_data(csv_path, valid_subjects, headcoil_mapping):
    """Load whole-brain TSNR data from combined_tsnr_coil_output.csv."""
    try:
        data = pd.read_csv(csv_path)
        data['Subject'] = data['Subject'].astype(str)
        data['AcquisitionType'] = data['AcquisitionType'].astype(str)
        data = data[data['Subject'].isin(valid_subjects)]

        data['mb'] = data['AcquisitionType'].str.extract(r'(mb\d)')
        data['me'] = data['AcquisitionType'].str.extract(r'(me\d)')
        data['headcoil'] = data['Subject'].map(headcoil_mapping).str.replace('.0', '', regex=False)
        data = data.rename(columns={'Subject': 'subject'})
        data['acq_combined'] = data['mb'].astype(str) + data['me'].astype(str)

        data = data[['subject', 'headcoil', 'mb', 'me', 'acq_combined', 'tsnrMean', 'tsnrMedian']]
        data['headcoil'] = pd.Categorical(data['headcoil'], categories=['20', '64'])
        data['mb'] = pd.Categorical(data['mb'], categories=['mb1', 'mb3', 'mb6'])
        data['me'] = pd.Categorical(data['me'], categories=['me1', 'me4'])
        data = data.dropna()
        return data

    except Exception as e:
        print(f"Error processing TSNR data: {str(e)}")
        return None


def plot_wholebrain_tsnr_emm_panel(ax, emm_df, raw_data, headcoil, n_subjects, me_colors=None):
    """
    Plot whole-brain TSNR EMM data for a single headcoil with individual data points overlay.
    Matches ROI activation figure style.
    """
    if me_colors is None:
        me_colors = ME_COLORS_NATURE
    
    coil_emm = emm_df[emm_df['HC'] == headcoil]
    coil_raw = raw_data[raw_data['headcoil'] == headcoil] if raw_data is not None else None
    
    if coil_emm.empty:
        ax.set_title(f"{headcoil}-ch (n=0)", fontsize=12)
        return
    
    x_positions = {'mb1': 0, 'mb3': 1, 'mb6': 2}
    jitter_emm = 0.08
    jitter_raw = 0.12
    me_offsets = {'me1': -jitter_emm, 'me4': jitter_emm}
    
    # --- Individual data points (behind EMMs) ---
    #if coil_raw is not None and len(coil_raw) > 0:
    #    np.random.seed(42)
    #    for me in ['me1', 'me4']:
    #        me_raw = coil_raw[coil_raw['me'] == me]
    #        for mb in ['mb1', 'mb3', 'mb6']:
    #            mb_me_data = me_raw[me_raw['mb'] == mb][IMG_VALUE_TSNR].dropna()
    #            if len(mb_me_data) > 0:
    #                x_base = x_positions[mb] + me_offsets[me]
    #                jitter_vals = np.random.uniform(-jitter_raw, jitter_raw, size=len(mb_me_data))
    #                ax.scatter(
    #                    np.full(len(mb_me_data), x_base) + jitter_vals,
    #                    mb_me_data.values,
    #                    color=me_colors[me], s=20, alpha=0.25,
    #                    edgecolors='none', zorder=1
    #                )
    
    # --- EMM lines and error bars ---
    for me in ['me1', 'me4']:
        me_data = coil_emm[coil_emm['ME'] == me].copy()
        if me_data.empty:
            continue
        me_data = me_data.sort_values('MB', key=lambda x: x.str.extract(r'mb(\d+)')[0].astype(int))
        x_vals = [x_positions[mb] + me_offsets[me] for mb in me_data['MB']]
        
        ax.plot(x_vals, me_data['emmean'], 
                color=me_colors[me], linewidth=2, linestyle='-', zorder=3)
        
        ax.errorbar(x_vals, me_data['emmean'], yerr=me_data['SE'],
                    fmt='o', color=me_colors[me], markersize=7,
                    capsize=3, capthick=1.5, elinewidth=1.5,
                    markeredgecolor='white', markeredgewidth=0.8, zorder=4)
    
    # --- Axis formatting ---
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['1', '3', '6'], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.set_title(f"{headcoil}-ch (n={n_subjects})", fontsize=12, fontweight='normal')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_linewidth(1)
    ax.grid(False)


def compute_wholebrain_ylimits(emm_df, raw_data):
    """Compute y-limits across EMMs and raw data."""
    all_y, all_e = [], []
    
    all_y.extend(emm_df['emmean'].values)
    all_e.extend(emm_df['SE'].values)
    
    if IMG_VALUE_TSNR in raw_data.columns:
        vals = raw_data[IMG_VALUE_TSNR].dropna().values
        all_y.extend(vals)
        all_e.extend([0] * len(vals))
    
    if all_y:
        y_max = max(v + e for v, e in zip(all_y, all_e))
        y_min = min(v - e for v, e in zip(all_y, all_e))
        margin = (y_max - y_min) * 0.08
        return (y_min - margin, y_max + margin)
    return (0, 100)


def generate_wholebrain_tsnr_emm_plot(merged_data, emm_df, n_20ch, n_64ch, 
                                       plot_filename, fig_ylabel='TSNR (EMM)'):
    """
    Generate Nature-quality 1×2 grid: 20-ch + 64-ch panels.
    """
    fig, axes = plt.subplots(1, 2, figsize=(10, 4),
                              gridspec_kw={'wspace': 0.25})
    
    # Compute y-limits
    y_limits = compute_wholebrain_ylimits(emm_df, merged_data)
    
    # Panel labels
    panel_labels = 'ab'
    
    # --- Panel a: 20-channel ---
    axes[0].text(-0.15, 1.08, panel_labels[0], transform=axes[0].transAxes,
                 fontsize=16, fontweight='bold', va='bottom', ha='left')
    plot_wholebrain_tsnr_emm_panel(axes[0], emm_df, merged_data, '20', n_20ch)
    axes[0].set_ylim(y_limits)
    axes[0].set_ylabel('Whole-Brain TSNR', fontsize=13)
    axes[0].set_xlabel('Multiband Factor', fontsize=13)
    
    # --- Panel b: 64-channel ---
    axes[1].text(-0.15, 1.08, panel_labels[1], transform=axes[1].transAxes,
                 fontsize=16, fontweight='bold', va='bottom', ha='left')
    plot_wholebrain_tsnr_emm_panel(axes[1], emm_df, merged_data, '64', n_64ch)
    axes[1].set_ylim(y_limits)
    axes[1].set_xlabel('Multiband Factor', fontsize=13)
    
    # Legend in right panel
    handles = [
        plt.Line2D([0], [0], color=ME_COLORS_NATURE['me1'], marker='o', markersize=6, 
                   linewidth=2, markeredgecolor='white', markeredgewidth=0.5, label='ME=1'),
        plt.Line2D([0], [0], color=ME_COLORS_NATURE['me4'], marker='o', markersize=6, 
                   linewidth=2, markeredgecolor='white', markeredgewidth=0.5, label='ME=4')
    ]
    axes[1].legend(handles=handles, fontsize=10, loc='lower right', frameon=False)
    
    plt.tight_layout()
    fig.subplots_adjust(top=0.88)
    
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  Whole-brain TSNR EMM plot saved: {plot_filename}")
    plt.show()


# Effect names and df dict for APA table
effect_names = {
    'headcoil_encoded': 'Head Coil', 'mb': 'Multiband', 'me': 'Multi-echo',
    'smoothness': 'Smoothness',
    'headcoil_encoded:mb': 'Head Coil x Multiband', 'headcoil_encoded:me': 'Head Coil x Multi-echo',
    'mb:me': 'Multiband x Multi-echo', 'headcoil_encoded:mb:me': 'Head Coil x Multiband x Multi-echo'
}
df_dict = {
    'Head Coil': 1, 'Multiband': 2, 'Multi-echo': 1, 'Smoothness': 1,
    'Head Coil x Multiband': 2, 'Head Coil x Multi-echo': 1,
    'Multiband x Multi-echo': 2, 'Head Coil x Multiband x Multi-echo': 2
}

# =============================================================================
# PART 1: LOAD DATA AND IDENTIFY COMMON COMPLETE SUBJECTS
# =============================================================================

print("\n" + "=" * 80)
print("PART 1: IDENTIFYING COMMON COMPLETE SUBJECTS")
print("=" * 80)

tsnr_data = load_and_process_tsnr_data(tsnr_path, valid_subjects, headcoil_mapping)
if tsnr_data is None:
    raise Exception("Failed to load TSNR data")

# Use shared identify_complete_subjects
complete_tsnr = identify_complete_subjects(tsnr_data, IMG_VALUE_TSNR)
# complete_smoothness already available from Shared Deps cell

print(f"Complete TSNR subjects: {len(complete_tsnr)}")
print(f"Complete smoothness subjects: {len(complete_smoothness)}")

common_complete = sorted(list(set(complete_tsnr) & set(complete_smoothness)))
print(f"\nCommon complete subjects: {len(common_complete)}")

if not common_complete:
    raise Exception("No common complete subjects found")

# Filter and merge
tsnr_filtered = tsnr_data[tsnr_data['subject'].isin(common_complete)].copy()
smoothness_filtered = smoothness_data[smoothness_data['subject'].isin(common_complete)].copy()

merged_data = pd.merge(
    tsnr_filtered[['subject', 'headcoil', 'mb', 'me', IMG_VALUE_TSNR]],
    smoothness_filtered[['subject', 'mb', 'me', 'smoothness']],
    on=['subject', 'mb', 'me'], how='inner'
).dropna()

total_n = merged_data['subject'].nunique()
n_20ch = len(merged_data[merged_data['headcoil'] == '20']['subject'].unique())
n_64ch = len(merged_data[merged_data['headcoil'] == '64']['subject'].unique())

print(f"\nFinal dataset: {total_n} subjects, {len(merged_data)} observations")
print(f"  20-channel: {n_20ch}")
print(f"  64-channel: {n_64ch}")

# Save complete subjects table
tsnr_complete_table = tsnr_filtered.pivot_table(
    values='tsnrMean', index='subject', columns='acq_combined', aggfunc='mean'
).round(3)
hc_info = tsnr_filtered[['subject', 'headcoil']].drop_duplicates().set_index('subject')
tsnr_complete_table = pd.concat([hc_info, tsnr_complete_table], axis=1)
tsnr_complete_table.to_csv(complete_subjects_file)
print(f"\nComplete subjects table saved to '{complete_subjects_file.name}'")

# =============================================================================
# PART 2: LINEAR MIXED EFFECTS ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("PART 2: LINEAR MIXED EFFECTS ANALYSIS")
print("=" * 80)

# Sum-to-zero headcoil encoding
model_data = merged_data.copy()
model_data['headcoil_encoded'] = model_data['headcoil'].cat.codes - 0.5

model = Lmer(f'{IMG_VALUE_TSNR} ~ headcoil_encoded * mb * me + smoothness + (1 | subject)',
             data=model_data)
model.fit()
anova_results = model.anova()

# Build APA table
apa_data = []
for effect in anova_results.index:
    if effect in ['(Intercept)', 'Residuals']:
        continue
    effect_name = effect_names.get(effect, effect)
    apa_data.append({
        'Effect': effect_name,
        'Sum Sq': anova_results.loc[effect, 'SS'] if 'SS' in anova_results.columns else np.nan,
        'Mean Sq': anova_results.loc[effect, 'MS'] if 'MS' in anova_results.columns else np.nan,
        'Num df': df_dict.get(effect_name, np.nan),
        'Den df': anova_results.loc[effect, 'DenomDF'] if 'DenomDF' in anova_results.columns else np.nan,
        'F': anova_results.loc[effect, 'F-stat'] if 'F-stat' in anova_results.columns else np.nan,
        'p': anova_results.loc[effect, 'P-val'] if 'P-val' in anova_results.columns else np.nan,
        'Partial eta_sq': np.nan
    })

# Residual-SS-based eta-squared
try:
    residual_var = model.ranef_var.loc[model.ranef_var['Name'] == '', 'Var'].iloc[0]
except (KeyError, IndexError):
    residual_var = model.ranef_var.iloc[1]['Var']

ss_residual = residual_var * (len(model_data) - sum(df_dict.values()) - model_data['subject'].nunique())

for i, row in enumerate(apa_data):
    if pd.notna(row['Sum Sq']):
        apa_data[i]['Partial eta_sq'] = row['Sum Sq'] / (row['Sum Sq'] + ss_residual)

apa_table = pd.DataFrame(apa_data)
apa_table['Sum Sq'] = apa_table['Sum Sq'].round(2)
apa_table['Mean Sq'] = apa_table['Mean Sq'].round(2)
apa_table['Num df'] = apa_table['Num df'].astype('Int64').fillna(pd.NA)
apa_table['Den df'] = apa_table['Den df'].round(2)
apa_table['F'] = apa_table['F'].round(2)
apa_table['p'] = apa_table['p'].apply(
    lambda x: '< .001' if pd.notna(x) and x < 0.001 else f'{x:.3f}' if pd.notna(x) else 'N/A'
)
apa_table['Partial eta_sq'] = apa_table['Partial eta_sq'].round(3)

print(f"\nAPA-Style ANOVA Table:")
print(f"Model: {IMG_VALUE_TSNR} ~ headcoil * mb * me + smoothness + (1 | subject)")
print(f"Data: Common complete subjects (N = {total_n})\n")
print(apa_table.to_string(index=False))

apa_table.to_csv(anova_table_file, index=False)
print(f"\nSaved to '{anova_table_file.name}'")

# =============================================================================
# PART 3: COMPUTE EMMs AND GENERATE VISUALIZATION
# =============================================================================

print("\n" + "=" * 80)
print("PART 3: COMPUTING EMMs AND GENERATING NATURE-QUALITY PLOT")
print("=" * 80)

# Compute EMMs using shared function (via R's emmeans package)
emm_df = compute_emms(merged_data, IMG_VALUE_TSNR, 'wholebrain')

print(f"\nEstimated Marginal Means:")
print(emm_df.to_string(index=False))

# Generate plot
generate_wholebrain_tsnr_emm_plot(
    merged_data=merged_data,
    emm_df=emm_df,
    n_20ch=n_20ch,
    n_64ch=n_64ch,
    plot_filename=tsnr_plot_file,
    fig_ylabel='Whole-Brain TSNR (EMM)'
)

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{'=' * 80}")
print("ANALYSIS COMPLETE - SUMMARY")
print(f"{'=' * 80}")
print(f"Common complete subjects: {total_n}")
print(f"  - 20-channel: {n_20ch}")
print(f"  - 64-channel: {n_64ch}")
print(f"\nOutput files:")
print(f"  - {complete_subjects_file.name}")
print(f"  - {anova_table_file.name}")
print(f"  - {tsnr_plot_file.name}")
print("=" * 80)

# ROI-Based TSNR Analysis (Fig. 5)
Per-ROI TSNR analyses (VS, rFFA, Motor) plus combined ROI comparisons. 3×2 grid and VS-vs-Motor plots.

In [ ]:
# =============================================================================
# MULTI-ROI TSNR ANALYSIS WITH SMOOTHNESS COVARIATE (Fig. 5)
# =============================================================================
# Nature-quality 3×3 grid: brain image + 20-ch + 64-ch per row, 3 ROIs.
# EMM-style plots with individual subject points + model-based EMM means/SEs.
#
# Dependencies: project_root, output_dir, extractions_dir, valid_subjects, 
#               headcoil_mapping, smoothness_data, complete_smoothness (from Shared Deps),
#               extract_roi_data, identify_complete_subjects, compute_emms,
#               ROI_CONFIGS, MB_MAP, ME_MAP, HC_MAP (from Setup)

import matplotlib.image as mpimg

print("=" * 80)
print("MULTI-ROI TSNR ANALYSIS PIPELINE WITH SMOOTHNESS COVARIATE")
print("=" * 80)

# --- Config ---
IMG_VALUE_TSNR = "tsnr"
ROIS_TSNR = ROI_CONFIGS['3roi']['rois']
ROI_LABELS_TSNR = ROI_CONFIGS['3roi']['labels']

# Nature-quality color scheme
ME_COLORS_NATURE = {'me1': '#4A90A4', 'me4': '#E07B39'}

# Brain image configuration (same as activation figures)
BRAIN_IMG_DIR = project_root / 'derivatives' / 'figures' / 'brain_images'
ROI_BRAIN_IMAGES = {
    'VSconstrained': {'file': 'mask_VSconstrained.png', 'slice_label': 'y = 15'},
    'rFFA': {'file': 'mask_rFFA.png', 'slice_label': 'z = −18'},
    'bilateralMotor': {'file': 'mask_bilateralMotor.png', 'slice_label': 'y = −24'}
}

# =============================================================================
# LOCAL FUNCTIONS
# =============================================================================

def run_tsnr_lme(merged_data, roi_name, effect_names, df_dict):
    """Run LME for TSNR with sum-to-zero headcoil encoding."""
    model_data = merged_data.copy()
    model_data['headcoil_encoded'] = model_data['headcoil'].cat.codes - 0.5

    model = Lmer(f'{IMG_VALUE_TSNR} ~ headcoil_encoded * mb * me + smoothness + (1 | subject)',
                 data=model_data)
    model.fit()
    anova_results = model.anova()

    apa_data = []
    for effect in anova_results.index:
        if effect in ['(Intercept)', 'Residuals']:
            continue
        effect_name = effect_names.get(effect, effect)
        apa_data.append({
            'Effect': effect_name,
            'Sum Sq': anova_results.loc[effect, 'SS'] if 'SS' in anova_results.columns else np.nan,
            'Mean Sq': anova_results.loc[effect, 'MS'] if 'MS' in anova_results.columns else np.nan,
            'Num df': df_dict.get(effect_name, np.nan),
            'Den df': anova_results.loc[effect, 'DenomDF'] if 'DenomDF' in anova_results.columns else np.nan,
            'F': anova_results.loc[effect, 'F-stat'] if 'F-stat' in anova_results.columns else np.nan,
            'p': anova_results.loc[effect, 'P-val'] if 'P-val' in anova_results.columns else np.nan,
            'Partial eta_sq': np.nan
        })

    try:
        residual_var = model.ranef_var.loc[model.ranef_var['Name'] == '', 'Var'].iloc[0]
    except (KeyError, IndexError):
        residual_var = model.ranef_var.iloc[1]['Var']

    ss_residual = residual_var * (len(model_data) - sum(df_dict.values()) - model_data['subject'].nunique())

    for i, row in enumerate(apa_data):
        if pd.notna(row['Sum Sq']):
            apa_data[i]['Partial eta_sq'] = row['Sum Sq'] / (row['Sum Sq'] + ss_residual)

    apa_table = pd.DataFrame(apa_data)
    apa_table['Sum Sq'] = apa_table['Sum Sq'].round(2)
    apa_table['Mean Sq'] = apa_table['Mean Sq'].round(2)
    apa_table['Num df'] = apa_table['Num df'].astype('Int64').fillna(pd.NA)
    apa_table['Den df'] = apa_table['Den df'].round(2)
    apa_table['F'] = apa_table['F'].round(2)
    apa_table['p'] = apa_table['p'].apply(
        lambda x: '< .001' if pd.notna(x) and x < 0.001 else f'{x:.3f}' if pd.notna(x) else 'N/A'
    )
    apa_table['Partial eta_sq'] = apa_table['Partial eta_sq'].round(3)
    return apa_table


def plot_tsnr_emm_panel(ax, emm_df, raw_data, headcoil, n_subjects, me_colors=None):
    """
    Plot TSNR EMM data for a single headcoil with individual data points overlay.
    Matches activation figure style.
    """
    if me_colors is None:
        me_colors = ME_COLORS_NATURE
    
    coil_emm = emm_df[emm_df['HC'] == headcoil]
    coil_raw = raw_data[raw_data['headcoil'] == headcoil] if raw_data is not None else None
    
    if coil_emm.empty:
        ax.set_title(f"{headcoil}-ch (n=0)", fontsize=12)
        return
    
    x_positions = {'mb1': 0, 'mb3': 1, 'mb6': 2}
    jitter_emm = 0.08
    jitter_raw = 0.12
    me_offsets = {'me1': -jitter_emm, 'me4': jitter_emm}
    
    # --- Individual data points (behind EMMs) ---
    #if coil_raw is not None and len(coil_raw) > 0:
    #    np.random.seed(42)
    #    for me in ['me1', 'me4']:
    #        me_raw = coil_raw[coil_raw['me'] == me]
    #        for mb in ['mb1', 'mb3', 'mb6']:
    #            mb_me_data = me_raw[me_raw['mb'] == mb][IMG_VALUE_TSNR].dropna()
    #            if len(mb_me_data) > 0:
    #                x_base = x_positions[mb] + me_offsets[me]
    #                jitter_vals = np.random.uniform(-jitter_raw, jitter_raw, size=len(mb_me_data))
    #                ax.scatter(
    #                    np.full(len(mb_me_data), x_base) + jitter_vals,
    #                    mb_me_data.values,
    #                    color=me_colors[me], s=20, alpha=0.25,
    #                    edgecolors='none', zorder=1
    #                )
    
    # --- EMM lines and error bars ---
    for me in ['me1', 'me4']:
        me_data = coil_emm[coil_emm['ME'] == me].copy()
        if me_data.empty:
            continue
        me_data = me_data.sort_values('MB', key=lambda x: x.str.extract(r'mb(\d+)')[0].astype(int))
        x_vals = [x_positions[mb] + me_offsets[me] for mb in me_data['MB']]
        
        ax.plot(x_vals, me_data['emmean'], 
                color=me_colors[me], linewidth=2, linestyle='-', zorder=3)
        
        ax.errorbar(x_vals, me_data['emmean'], yerr=me_data['SE'],
                    fmt='o', color=me_colors[me], markersize=7,
                    capsize=3, capthick=1.5, elinewidth=1.5,
                    markeredgecolor='white', markeredgewidth=0.8, zorder=4)
    
    # --- Axis formatting ---
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['1', '3', '6'], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.set_title(f"{headcoil}-ch (n={n_subjects})", fontsize=12, fontweight='normal')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_linewidth(1)
    ax.grid(False)


def compute_tsnr_ylimits(emm_results_dict, raw_data_dict):
    """Compute shared y-limits across EMMs and raw data."""
    all_y, all_e = [], []
    
    for emm_df in emm_results_dict.values():
        all_y.extend(emm_df['emmean'].values)
        all_e.extend(emm_df['SE'].values)
    
    for raw_df in raw_data_dict.values():
        if IMG_VALUE_TSNR in raw_df.columns:
            vals = raw_df[IMG_VALUE_TSNR].dropna().values
            all_y.extend(vals)
            all_e.extend([0] * len(vals))
    
    if all_y:
        y_max = max(v + e for v, e in zip(all_y, all_e))
        y_min = min(v - e for v, e in zip(all_y, all_e))
        margin = (y_max - y_min) * 0.08
        return (y_min - margin, y_max + margin)
    return (0, 100)


def generate_tsnr_emm_grid_plot_with_brain_images(rois, roi_labels, roi_results, emm_results, 
                                                   plot_filename, fig_ylabel='TSNR (EMM)'):
    """
    Generate Nature-quality 3×3 grid: brain image + 20-ch + 64-ch per row.
    Matches activation figure layout exactly.
    """
    n_rois = len(rois)
    
    # Load brain images first to get aspect ratios
    brain_images = {}
    brain_aspects = []
    for roi in rois:
        if roi in ROI_BRAIN_IMAGES:
            img_path = BRAIN_IMG_DIR / ROI_BRAIN_IMAGES[roi]['file']
            if img_path.exists():
                brain_img = mpimg.imread(str(img_path))
                brain_images[roi] = brain_img
                img_h, img_w = brain_img.shape[:2]
                brain_aspects.append(img_w / img_h)
            else:
                print(f"  Warning: Brain image not found: {img_path}")
                brain_images[roi] = None
                brain_aspects.append(1.0)
        else:
            brain_images[roi] = None
            brain_aspects.append(1.0)
    
    # Use average brain aspect for consistent layout
    avg_brain_aspect = sum(brain_aspects) / len(brain_aspects) if brain_aspects else 1.0
    scatter_ratio = 1.2
    
    # Figure sizing: 3 columns (brain + 20ch + 64ch)
    fig_height = 3.2 * n_rois + 0.3
    fig, axes = plt.subplots(n_rois, 3, figsize=(13, fig_height),
                              gridspec_kw={'width_ratios': [avg_brain_aspect * 0.8, scatter_ratio, scatter_ratio],
                                          'wspace': 0.25, 'hspace': 0.4})
    
    if n_rois == 1:
        axes = axes.reshape(1, -1)
    
    # Compute y-limits including raw data
    raw_data_dict = {roi: roi_results[roi]['data'] for roi in rois if roi in roi_results}
    y_limits = compute_tsnr_ylimits(emm_results, raw_data_dict)
    
    # Panel labels
    panel_labels = 'abcdefghi'
    panel_idx = 0
    
    for roi_idx, roi in enumerate(rois):
        if roi not in emm_results or roi not in roi_results:
            continue
        
        emm_df = emm_results[roi]
        roi_data = roi_results[roi]
        raw_data = roi_data['data']
        
        ax_brain = axes[roi_idx, 0]
        ax_20ch = axes[roi_idx, 1]
        ax_64ch = axes[roi_idx, 2]
        
        # --- Brain image panel ---
        if brain_images[roi] is not None:
            ax_brain.imshow(brain_images[roi])
        ax_brain.axis('off')
        
        # --- Panel labels using ax.transAxes (BEFORE tight_layout) ---
        ax_brain.text(0.0, 1.08, panel_labels[panel_idx], transform=ax_brain.transAxes,
                      fontsize=16, fontweight='bold', va='bottom', ha='left')
        ax_20ch.text(-0.15, 1.08, panel_labels[panel_idx + 1], transform=ax_20ch.transAxes,
                     fontsize=16, fontweight='bold', va='bottom', ha='left')
        ax_64ch.text(-0.15, 1.08, panel_labels[panel_idx + 2], transform=ax_64ch.transAxes,
                     fontsize=16, fontweight='bold', va='bottom', ha='left')
        panel_idx += 3
        
        # --- Slice label below brain image ---
        if roi in ROI_BRAIN_IMAGES:
            ax_brain.text(0.5, -0.06, ROI_BRAIN_IMAGES[roi]['slice_label'],
                          transform=ax_brain.transAxes,
                          fontsize=11, ha='center', va='top')
        
        # --- 20-channel panel ---
        plot_tsnr_emm_panel(ax_20ch, emm_df, raw_data, '20', roi_data['n_20ch'])
        ax_20ch.set_ylim(y_limits)
        
        # --- 64-channel panel ---
        plot_tsnr_emm_panel(ax_64ch, emm_df, raw_data, '64', roi_data['n_64ch'])
        ax_64ch.set_ylim(y_limits)
        
        # Y-axis label on 20-ch panel only (ROI name)
        ax_20ch.set_ylabel(roi_labels.get(roi, roi), fontsize=13)
    
    # X-axis label only on bottom row
    for col in [1, 2]:
        axes[-1, col].set_xlabel('Multiband Factor', fontsize=13)
    
    # Legend in top-right panel
    handles = [
        plt.Line2D([0], [0], color=ME_COLORS_NATURE['me1'], marker='o', markersize=6, 
                   linewidth=2, markeredgecolor='white', markeredgewidth=0.5, label='ME=1'),
        plt.Line2D([0], [0], color=ME_COLORS_NATURE['me4'], marker='o', markersize=6, 
                   linewidth=2, markeredgecolor='white', markeredgewidth=0.5, label='ME=4')
    ]
    axes[0, 2].legend(handles=handles, fontsize=10, loc='upper right', frameon=False)
    
    plt.tight_layout()
    
    # Global y-axis label (AFTER tight_layout)
    fig.text(0.01, 0.5, fig_ylabel, va='center', rotation='vertical', 
             fontsize=14, transform=fig.transFigure)
    
    # Adjust layout
    fig.subplots_adjust(left=0.10, top=0.94, hspace=0.45)
    
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  TSNR EMM plot saved: {plot_filename}")
    plt.show()


# Effect name mappings
tsnr_effect_names = {
    'headcoil_encoded': 'Head Coil', 'mb': 'Multiband', 'me': 'Multi-echo',
    'smoothness': 'Smoothness',
    'headcoil_encoded:mb': 'Head Coil x Multiband', 'headcoil_encoded:me': 'Head Coil x Multi-echo',
    'mb:me': 'Multiband x Multi-echo', 'headcoil_encoded:mb:me': 'Head Coil x Multiband x Multi-echo'
}
tsnr_df_dict = {
    'Head Coil': 1, 'Multiband': 2, 'Multi-echo': 1, 'Smoothness': 1,
    'Head Coil x Multiband': 2, 'Head Coil x Multi-echo': 1,
    'Multiband x Multi-echo': 2, 'Head Coil x Multiband x Multi-echo': 2
}

# Extended for 4-factor ROI model
tsnr_roi_effect_names = {
    **tsnr_effect_names,
    'roi': 'ROI',
    'headcoil_encoded:roi': 'Head Coil x ROI', 'mb:roi': 'Multiband x ROI',
    'me:roi': 'Multi-echo x ROI',
    'headcoil_encoded:mb:roi': 'Head Coil x Multiband x ROI',
    'headcoil_encoded:me:roi': 'Head Coil x Multi-echo x ROI',
    'mb:me:roi': 'Multiband x Multi-echo x ROI',
    'headcoil_encoded:mb:me:roi': 'Head Coil x Multiband x Multi-echo x ROI'
}


def make_roi_df_dict(n_rois):
    """Build df_dict for ROI-factor model with given number of ROI levels."""
    k = n_rois - 1
    return {
        **tsnr_df_dict,
        'ROI': k,
        'Head Coil x ROI': k, 'Multiband x ROI': 2*k, 'Multi-echo x ROI': k,
        'Head Coil x Multiband x ROI': 2*k, 'Head Coil x Multi-echo x ROI': k,
        'Multiband x Multi-echo x ROI': 2*k, 'Head Coil x Multiband x Multi-echo x ROI': 2*k
    }


# =============================================================================
# PART 1: LOAD DATA AND IDENTIFY COMPLETE SUBJECTS
# =============================================================================

print("\n" + "=" * 80)
print("PART 1: LOADING DATA AND IDENTIFYING COMPLETE SUBJECTS")
print("=" * 80)

roi_results_tsnr = {}
all_roi_data_tsnr = {}

for roi in ROIS_TSNR:
    print(f"\nProcessing {roi}...")

    roi_data = extract_roi_data(extractions_dir, TYPE_VALUE, IMG_VALUE_TSNR,
                                 roi, DENOISE_VALUE, valid_subjects, headcoil_mapping)
    if roi_data is None:
        print(f"  Warning: No data found for {roi}")
        continue

    complete_roi = identify_complete_subjects(roi_data, IMG_VALUE_TSNR)
    print(f"  Complete {roi} subjects: {len(complete_roi)}")

    common_complete = sorted(list(set(complete_roi) & set(complete_smoothness)))
    print(f"  Common complete subjects: {len(common_complete)}")

    if not common_complete:
        print(f"  No common complete subjects for {roi}")
        continue

    roi_filtered = roi_data[roi_data['subject'].isin(common_complete)].copy()
    smoothness_filtered = smoothness_data[smoothness_data['subject'].isin(common_complete)].copy()

    merged = pd.merge(
        roi_filtered[['subject', 'headcoil', 'mb', 'me', IMG_VALUE_TSNR]],
        smoothness_filtered[['subject', 'mb', 'me', 'smoothness']],
        on=['subject', 'mb', 'me'], how='inner'
    ).dropna()

    roi_results_tsnr[roi] = {
        'data': merged,
        'n_subjects': merged['subject'].nunique(),
        'n_20ch': len(merged[merged['headcoil'] == '20']['subject'].unique()),
        'n_64ch': len(merged[merged['headcoil'] == '64']['subject'].unique())
    }
    all_roi_data_tsnr[roi] = roi_filtered

# =============================================================================
# PART 2: PER-ROI LME ANALYSES AND EMM COMPUTATION
# =============================================================================

print("\n" + "=" * 80)
print("PART 2: LINEAR MIXED EFFECTS ANALYSES")
print("=" * 80)

emm_results_tsnr = {}

for roi in ROIS_TSNR:
    if roi not in roi_results_tsnr:
        continue

    print(f"\n=== {ROI_LABELS_TSNR[roi]} ({roi}) ===")
    rd = roi_results_tsnr[roi]
    print(f"N = {rd['n_subjects']} subjects ({rd['n_20ch']} 20-ch, {rd['n_64ch']} 64-ch)")

    apa_table = run_tsnr_lme(rd['data'], roi, tsnr_effect_names, tsnr_df_dict)
    
    print(f"\nAPA-Style ANOVA Table:")
    print(f"Model: {IMG_VALUE_TSNR} ~ headcoil * mb * me + smoothness + (1 | subject)")
    print(f"Data: {ROI_LABELS_TSNR[roi]} ROI\n")
    print(apa_table.to_string(index=False))

    anova_file = output_dir / f'{roi}_{IMG_VALUE_TSNR}_lme_anova_with_smoothness.csv'
    apa_table.to_csv(anova_file, index=False)
    print(f"\nSaved to '{anova_file.name}'")
    
    # Compute EMMs using the shared function (via R's emmeans package)
    emm_df = compute_emms(rd['data'], IMG_VALUE_TSNR, roi)
    emm_results_tsnr[roi] = emm_df

# =============================================================================
# PART 2.5: COMBINED ROI ANALYSES (VS+Motor, All 3 ROIs)
# =============================================================================

for rois_to_compare, label in [
    (['VSconstrained', 'bilateralMotor'], 'VS vs Motor'),
    (ROIS_TSNR, f'All {len(ROIS_TSNR)} ROIs')
]:
    print(f"\n{'=' * 80}")
    print(f"COMBINED LME: {label}")
    print(f"{'=' * 80}")

    combined_roi_data = []
    for roi in rois_to_compare:
        if roi in roi_results_tsnr:
            rd = roi_results_tsnr[roi]['data'].copy()
            rd['roi'] = roi
            combined_roi_data.append(rd)

    if len(combined_roi_data) < 2:
        print(f"Not enough ROI data for {label} comparison")
        continue

    combined_df = pd.concat(combined_roi_data, ignore_index=True)
    combined_df['roi'] = pd.Categorical(combined_df['roi'], categories=rois_to_compare)

    print(f"\nCombined dataset: {len(combined_df)} observations")
    print(f"Subjects: {combined_df['subject'].nunique()}")
    print(f"ROI breakdown: {combined_df['roi'].value_counts().to_dict()}")

    model_data_c = combined_df.copy()
    model_data_c['headcoil_encoded'] = model_data_c['headcoil'].cat.codes - 0.5

    model_c = Lmer(f'{IMG_VALUE_TSNR} ~ headcoil_encoded * mb * me * roi + smoothness + (1 | subject)',
                   data=model_data_c)
    model_c.fit()
    anova_c = model_c.anova()

    roi_df_dict = make_roi_df_dict(len(rois_to_compare))

    apa_data_c = []
    for effect in anova_c.index:
        if effect in ['(Intercept)', 'Residuals']:
            continue
        effect_name = tsnr_roi_effect_names.get(effect, effect)
        apa_data_c.append({
            'Effect': effect_name,
            'Sum Sq': anova_c.loc[effect, 'SS'] if 'SS' in anova_c.columns else np.nan,
            'Mean Sq': anova_c.loc[effect, 'MS'] if 'MS' in anova_c.columns else np.nan,
            'Num df': roi_df_dict.get(effect_name, np.nan),
            'Den df': anova_c.loc[effect, 'DenomDF'] if 'DenomDF' in anova_c.columns else np.nan,
            'F': anova_c.loc[effect, 'F-stat'] if 'F-stat' in anova_c.columns else np.nan,
            'p': anova_c.loc[effect, 'P-val'] if 'P-val' in anova_c.columns else np.nan,
            'Partial eta_sq': np.nan
        })

    try:
        resvar_c = model_c.ranef_var.loc[model_c.ranef_var['Name'] == '', 'Var'].iloc[0]
    except (KeyError, IndexError):
        resvar_c = model_c.ranef_var.iloc[1]['Var']

    ss_res_c = resvar_c * (len(model_data_c) - sum(roi_df_dict.values()) - model_data_c['subject'].nunique())
    for i, row in enumerate(apa_data_c):
        if pd.notna(row['Sum Sq']):
            apa_data_c[i]['Partial eta_sq'] = row['Sum Sq'] / (row['Sum Sq'] + ss_res_c)

    apa_table_c = pd.DataFrame(apa_data_c)
    for col in ['Sum Sq', 'Mean Sq']:
        apa_table_c[col] = apa_table_c[col].round(2)
    apa_table_c['Num df'] = apa_table_c['Num df'].astype('Int64').fillna(pd.NA)
    apa_table_c['Den df'] = apa_table_c['Den df'].round(2)
    apa_table_c['F'] = apa_table_c['F'].round(2)
    apa_table_c['p'] = apa_table_c['p'].apply(
        lambda x: '< .001' if pd.notna(x) and x < 0.001 else f'{x:.3f}' if pd.notna(x) else 'N/A'
    )
    apa_table_c['Partial eta_sq'] = apa_table_c['Partial eta_sq'].round(3)

    print(f"\nAPA-Style ANOVA Table for {label}:")
    print(apa_table_c.to_string(index=False))

    fname = 'vs_motor' if len(rois_to_compare) == 2 else 'combined'
    anova_file_c = output_dir / f'{fname}_roi_{IMG_VALUE_TSNR}_lme_anova_with_smoothness.csv'
    apa_table_c.to_csv(anova_file_c, index=False)
    print(f"\nSaved to '{anova_file_c.name}'")

# =============================================================================
# PART 3: NATURE-QUALITY EMM GRID PLOT (3×3 with brain images)
# =============================================================================

print("\n" + "=" * 80)
print("PART 3: GENERATING NATURE-QUALITY EMM GRID PLOT WITH BRAIN IMAGES")
print("=" * 80)

tsnr_plot_file = output_dir / 'multi_roi_tsnr_emm_plot_publication.png'

generate_tsnr_emm_grid_plot_with_brain_images(
    rois=ROIS_TSNR,
    roi_labels=ROI_LABELS_TSNR,
    roi_results=roi_results_tsnr,
    emm_results=emm_results_tsnr,
    plot_filename=tsnr_plot_file,
    fig_ylabel='Temporal Signal-to-Noise Ratio (EMM)'
)

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{'=' * 80}")
print("ANALYSIS COMPLETE - SUMMARY")
print(f"{'=' * 80}")
print(f"ROIs analyzed: {', '.join(ROIS_TSNR)}")
for roi in ROIS_TSNR:
    if roi in roi_results_tsnr:
        r = roi_results_tsnr[roi]
        print(f"  {ROI_LABELS_TSNR[roi]}: {r['n_subjects']} total ({r['n_20ch']} 20-ch, {r['n_64ch']} 64-ch)")
print(f"\nOutput: {tsnr_plot_file.name}")
print("=" * 80)

# ROI Beta, Z-Stat & Weighted Beta Activation Analysis with EMMs (Fig. 6)

Analyzes activation estimates across Ventral Striatum, Right FFA, and Motor Cortex.
Runs **beta**, **z-stat**, and **weighted beta** (inverse-variance, weights = 1/varcope) analyses.
Weighted analysis uses R's lmer directly since pymer4 doesn't support observation weights.

In [ ]:
# =============================================================================
# ROI ACTIVATION ANALYSIS — BETA, Z-STAT, AND WEIGHTED BETA (Fig. 6 + Supplement)
# =============================================================================
# Consolidates unweighted (beta, zstat) and weighted (1/varcope) analyses for 3 ROIs.
# Nature-quality 3×3 grid: brain image + 20-ch + 64-ch per row.
#
# Dependencies: 
#   - project_root, output_dir, extractions_dir (from Setup)
#   - valid_subjects, headcoil_mapping (from Demographics)
#   - smoothness_data, complete_smoothness (from Shared Dependencies)
#   - extract_roi_data, identify_complete_subjects, run_lme_analysis, compute_emms (from Setup)

import matplotlib.image as mpimg

# =============================================================================
# NATURE-QUALITY PLOTTING FUNCTIONS (local to this cell)
# =============================================================================

# Color scheme for multi-echo
ME_COLORS_NATURE = {'me1': '#4A90A4', 'me4': '#E07B39'}

# Brain image configuration
BRAIN_IMG_DIR = project_root / 'derivatives' / 'figures' / 'brain_images'
ROI_BRAIN_IMAGES = {
    'VSconstrained': {
        'file': 'mask_VSconstrained.png',
        'slice_label': 'y = 15'
    },
    'rFFA': {
        'file': 'mask_rFFA.png',
        'slice_label': 'z = −18'
    },
    'bilateralMotor': {
        'file': 'mask_bilateralMotor.png',
        'slice_label': 'y = −24'
    }
}


def plot_emm_panel_nature(ax, emm_df, raw_data, headcoil, n_subjects, img_value, me_colors=None):
    """
    Plot EMM data for a single headcoil with individual data points overlay.
    """
    if me_colors is None:
        me_colors = ME_COLORS_NATURE
    
    coil_emm = emm_df[emm_df['HC'] == headcoil]
    coil_raw = raw_data[raw_data['headcoil'] == headcoil] if raw_data is not None else None
    
    if coil_emm.empty:
        ax.set_title(f"{headcoil}-ch (n=0)", fontsize=12)
        return
    
    x_positions = {'mb1': 0, 'mb3': 1, 'mb6': 2}
    jitter_emm = 0.08
    jitter_raw = 0.12
    me_offsets = {'me1': -jitter_emm, 'me4': jitter_emm}
    
    # --- Individual data points (behind EMMs) ---
    #if coil_raw is not None and len(coil_raw) > 0:
    #    np.random.seed(42)
    #    for me in ['me1', 'me4']:
    #        me_raw = coil_raw[coil_raw['me'] == me]
    #        for mb in ['mb1', 'mb3', 'mb6']:
    #            mb_me_data = me_raw[me_raw['mb'] == mb][img_value].dropna()
    #            if len(mb_me_data) > 0:
    #                x_base = x_positions[mb] + me_offsets[me]
    #                jitter_vals = np.random.uniform(-jitter_raw, jitter_raw, size=len(mb_me_data))
    #                ax.scatter(
    #                    np.full(len(mb_me_data), x_base) + jitter_vals,
    #                    mb_me_data.values,
    #                    color=me_colors[me], s=20, alpha=0.25,
    #                    edgecolors='none', zorder=1
    #                )
    
    # --- EMM lines and error bars ---
    for me in ['me1', 'me4']:
        me_data = coil_emm[coil_emm['ME'] == me].copy()
        if me_data.empty:
            continue
        me_data = me_data.sort_values('MB', key=lambda x: x.str.extract(r'mb(\d+)')[0].astype(int))
        x_vals = [x_positions[mb] + me_offsets[me] for mb in me_data['MB']]
        
        ax.plot(x_vals, me_data['emmean'], 
                color=me_colors[me], linewidth=2, linestyle='-', zorder=3)
        
        ax.errorbar(x_vals, me_data['emmean'], yerr=me_data['SE'],
                    fmt='o', color=me_colors[me], markersize=7,
                    capsize=3, capthick=1.5, elinewidth=1.5,
                    markeredgecolor='white', markeredgewidth=0.8, zorder=4)
    
    # --- Axis formatting ---
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['1', '3', '6'], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.set_title(f"{headcoil}-ch (n={n_subjects})", fontsize=12, fontweight='normal')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_linewidth(1)
    ax.grid(False)


def compute_emm_ylimits_nature(emm_results_dict, raw_data_dict=None, img_value=None):
    """Compute shared y-limits across EMMs and raw data."""
    all_y, all_e = [], []
    
    for emm_df in emm_results_dict.values():
        all_y.extend(emm_df['emmean'].values)
        all_e.extend(emm_df['SE'].values)
    
    if raw_data_dict is not None and img_value is not None:
        for raw_df in raw_data_dict.values():
            if img_value in raw_df.columns:
                vals = raw_df[img_value].dropna().values
                all_y.extend(vals)
                all_e.extend([0] * len(vals))
    
    if all_y:
        y_max = max(v + e for v, e in zip(all_y, all_e))
        y_min = min(v - e for v, e in zip(all_y, all_e))
        margin = (y_max - y_min) * 0.08
        return (y_min - margin, y_max + margin)
    return (-1, 1)


def generate_emm_grid_plot_with_brain_images(rois, roi_labels, roi_ylabels, roi_results, 
                                              emm_results, img_value, plot_filename, 
                                              fig_ylabel=None, legend_loc='upper right'):
    """
    Generate Nature-quality 3×3 grid: brain image + 20-ch + 64-ch per row.
    """
    n_rois = len(rois)
    
    # Load brain images first to get aspect ratios
    brain_images = {}
    brain_aspects = []
    for roi in rois:
        if roi in ROI_BRAIN_IMAGES:
            img_path = BRAIN_IMG_DIR / ROI_BRAIN_IMAGES[roi]['file']
            if img_path.exists():
                brain_img = mpimg.imread(str(img_path))
                brain_images[roi] = brain_img
                img_h, img_w = brain_img.shape[:2]
                brain_aspects.append(img_w / img_h)
            else:
                print(f"  Warning: Brain image not found: {img_path}")
                brain_images[roi] = None
                brain_aspects.append(1.0)
        else:
            brain_images[roi] = None
            brain_aspects.append(1.0)
    
    # Use average brain aspect for consistent layout
    avg_brain_aspect = sum(brain_aspects) / len(brain_aspects) if brain_aspects else 1.0
    scatter_ratio = 1.2
    
    # Figure sizing: 3 columns (brain + 20ch + 64ch)
    fig_height = 3.2 * n_rois + 0.3
    fig, axes = plt.subplots(n_rois, 3, figsize=(13, fig_height),
                              gridspec_kw={'width_ratios': [avg_brain_aspect * 0.8, scatter_ratio, scatter_ratio],
                                          'wspace': 0.25, 'hspace': 0.4})
    
    if n_rois == 1:
        axes = axes.reshape(1, -1)
    
    # Compute y-limits including raw data
    raw_data_dict = {roi: roi_results[roi]['data'] for roi in rois if roi in roi_results}
    y_limits = compute_emm_ylimits_nature(emm_results, raw_data_dict, img_value)
    
    # Panel label tracking
    panel_labels = 'abcdefghi'
    panel_idx = 0
    
    for roi_idx, roi in enumerate(rois):
        if roi not in emm_results or roi not in roi_results:
            continue
        
        emm_df = emm_results[roi]
        roi_data = roi_results[roi]
        raw_data = roi_data['data']
        
        ax_brain = axes[roi_idx, 0]
        ax_20ch = axes[roi_idx, 1]
        ax_64ch = axes[roi_idx, 2]
        
        # --- Brain image panel ---
        if brain_images[roi] is not None:
            ax_brain.imshow(brain_images[roi])
        ax_brain.axis('off')
        
        # --- Panel labels using ax.transAxes (BEFORE tight_layout) ---
        # These use axes-relative coords: (0,0)=bottom-left, (1,1)=top-right
        ax_brain.text(0.0, 1.08, panel_labels[panel_idx], transform=ax_brain.transAxes,
                      fontsize=16, fontweight='bold', va='bottom', ha='left')
        ax_20ch.text(-0.15, 1.08, panel_labels[panel_idx + 1], transform=ax_20ch.transAxes,
                     fontsize=16, fontweight='bold', va='bottom', ha='left')
        ax_64ch.text(-0.15, 1.08, panel_labels[panel_idx + 2], transform=ax_64ch.transAxes,
                     fontsize=16, fontweight='bold', va='bottom', ha='left')
        panel_idx += 3
        
        # --- Slice label below brain image ---
        if roi in ROI_BRAIN_IMAGES:
            ax_brain.text(0.5, -0.06, ROI_BRAIN_IMAGES[roi]['slice_label'],
                          transform=ax_brain.transAxes,
                          fontsize=11, ha='center', va='top')
        
        # --- 20-channel panel ---
        plot_emm_panel_nature(ax_20ch, emm_df, raw_data, '20', 
                              roi_data['n_20ch'], img_value)
        ax_20ch.set_ylim(y_limits)
        
        # --- 64-channel panel ---
        plot_emm_panel_nature(ax_64ch, emm_df, raw_data, '64', 
                              roi_data['n_64ch'], img_value)
        ax_64ch.set_ylim(y_limits)
        
        # Y-axis label on 20-ch panel only
        ax_20ch.set_ylabel(roi_ylabels.get(roi, ''), fontsize=13)
    
    # X-axis label only on bottom row
    for col in [1, 2]:
        axes[-1, col].set_xlabel('Multiband Factor', fontsize=13)
    
    # Legend in top-right panel (64-ch of first row)
    handles = [
        plt.Line2D([0], [0], color=ME_COLORS_NATURE['me1'], marker='o', markersize=6, 
                   linewidth=2, markeredgecolor='white', markeredgewidth=0.5, label='ME=1'),
        plt.Line2D([0], [0], color=ME_COLORS_NATURE['me4'], marker='o', markersize=6, 
                   linewidth=2, markeredgecolor='white', markeredgewidth=0.5, label='ME=4')
    ]
    axes[0, 2].legend(handles=handles, fontsize=10, loc=legend_loc, frameon=False)
    
    # Global y-axis label
    if fig_ylabel is None:
        label_map = {'beta': 'Beta', 'zstat': 'Z-Stat'}
        fig_ylabel = f'Activation ({label_map.get(img_value, img_value)} EMM)'
    
    plt.tight_layout()
    
    # Global y-axis label (rotated, on left side)
    fig.text(0.01, 0.5, fig_ylabel, va='center', rotation='vertical', 
             fontsize=14, transform=fig.transFigure)
    
    # Adjust layout to make room for labels
    fig.subplots_adjust(left=0.10, top=0.94, hspace=0.45)
    
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  EMM plot saved: {plot_filename}")
    plt.show()
    
    return fig


def run_full_roi_analysis_nature(img_value, roi_config_key, valid_subjects, headcoil_mapping,
                                  smoothness_data, complete_smoothness, legend_loc='upper right'):
    """
    Run complete ROI analysis pipeline with Nature-quality visualization.
    """
    config = ROI_CONFIGS[roi_config_key]
    rois = config['rois']
    roi_labels = config['labels']
    roi_ylabels = config['ylabels']
    
    label_map = {'beta': 'Beta', 'zstat': 'Z-Stat'}
    label = label_map.get(img_value, img_value)
    
    print("=" * 80)
    print(f"MULTI-ROI {label.upper()} ANALYSIS PIPELINE WITH SMOOTHNESS COVARIATE AND EMMs")
    print("=" * 80)
    print(f"Processing {img_value} values for ROIs: {', '.join(rois)}")
    
    # Part 1: Load data
    print("\n" + "=" * 80)
    print("PART 1: LOADING DATA AND IDENTIFYING COMPLETE SUBJECTS")
    print("=" * 80)
    
    roi_results = load_and_merge_roi_data(
        rois, img_value, smoothness_data, complete_smoothness,
        valid_subjects, headcoil_mapping
    )
    
    # Part 2: LME analyses
    print("\n" + "=" * 80)
    print("PART 2: LINEAR MIXED EFFECTS ANALYSES")
    print("=" * 80)
    
    emm_results = {}
    for roi in rois:
        if roi not in roi_results:
            continue
        
        print(f"\n=== {roi_labels[roi]} ({roi}) ===")
        rd = roi_results[roi]
        print(f"N = {rd['n_subjects']} subjects ({rd['n_20ch']} 20-ch, {rd['n_64ch']} 64-ch)")
        
        apa_table, _ = run_lme_analysis(rd['data'], img_value, roi)
        print(f"\nAPA-Style ANOVA Table:")
        print(f"Model: {img_value} ~ headcoil * mb * me + smoothness + (1 | subject)")
        print(f"Data: {roi_labels[roi]} ROI\n")
        print(apa_table.to_string(index=False))
        
        anova_file = output_dir / f'{roi}_{img_value}_lme_anova_with_smoothness.csv'
        apa_table.to_csv(anova_file, index=False)
        print(f"\nSaved to '{anova_file.name}'")
        
        emm_df = compute_emms(rd['data'], img_value, roi)
        emm_results[roi] = emm_df
    
    # Part 3: Nature-quality visualization with brain images
    print("\n" + "=" * 80)
    print("PART 3: GENERATING NATURE-QUALITY EMM PLOTS WITH BRAIN IMAGES")
    print("=" * 80)
    
    plot_file = output_dir / f'multi_roi_{img_value}_emm_plot_publication.png'
    generate_emm_grid_plot_with_brain_images(
        rois, roi_labels, roi_ylabels, roi_results, emm_results,
        img_value, plot_file, legend_loc=legend_loc
    )
    
    # Summary
    print("\n" + "=" * 80)
    print("ANALYSIS COMPLETE - SUMMARY")
    print("=" * 80)
    print(f"ROIs analyzed: {', '.join(rois)}")
    for roi in rois:
        if roi in roi_results:
            r = roi_results[roi]
            print(f"  {roi_labels[roi]}: {r['n_subjects']} total ({r['n_20ch']} 20-ch, {r['n_64ch']} 64-ch)")
    print("=" * 80)
    
    return roi_results, emm_results


# =============================================================================
# PART A: UNWEIGHTED ANALYSES (Beta and Z-Stat)
# =============================================================================

# Run beta analysis
print("\n" + "#" * 80)
print("# BETA ANALYSIS")
print("#" * 80 + "\n")

beta_roi_results, beta_emm_results = run_full_roi_analysis_nature(
    img_value='beta',
    roi_config_key='3roi',
    valid_subjects=valid_subjects,
    headcoil_mapping=headcoil_mapping,
    smoothness_data=smoothness_data,
    complete_smoothness=complete_smoothness,
    legend_loc='lower right'
)

# Run z-stat analysis (supplemental)
print("\n" + "#" * 80)
print("# Z-STAT ANALYSIS (SUPPLEMENT)")
print("#" * 80 + "\n")

zstat_roi_results, zstat_emm_results = run_full_roi_analysis_nature(
    img_value='zstat',
    roi_config_key='3roi',
    valid_subjects=valid_subjects,
    headcoil_mapping=headcoil_mapping,
    smoothness_data=smoothness_data,
    complete_smoothness=complete_smoothness,
    legend_loc='upper right'
)

# =============================================================================
# PART B: WEIGHTED BETA ANALYSIS (Inverse-Variance Weighting from Varcope)
# =============================================================================

print("\n" + "#" * 80)
print("# WEIGHTED BETA ANALYSIS (INVERSE-VARIANCE)")
print("#" * 80 + "\n")

print("=" * 80)
print("MULTI-ROI WEIGHTED BETA ANALYSIS PIPELINE")
print("(Inverse-Variance Weighting from Varcope Images)")
print("=" * 80)

# --- Config ---
ROIS_W = ROI_CONFIGS['3roi']['rois']
ROI_LABELS_W = ROI_CONFIGS['3roi']['labels']
ROI_YLABELS_W = ROI_CONFIGS['3roi']['ylabels']

combined_emm_plot_file = output_dir / 'multi_roi_weighted_beta_emm_plot_publication.png'

print(f"Processing weighted beta values for ROIs: {', '.join(ROIS_W)}")


def prepare_weights(merged_data, winsorize_percentile=99, normalize=True):
    """Compute inverse-variance weights from varcope values."""
    weights = 1.0 / merged_data['varcope']
    if winsorize_percentile < 100:
        weight_cap = np.percentile(weights, winsorize_percentile)
        weights = weights.clip(upper=weight_cap)
        n_capped = (weights == weight_cap).sum()
        if n_capped > 0:
            print(f"  Capped {n_capped} weights at {winsorize_percentile}th percentile")
    if normalize:
        weights = weights / weights.mean()
    return weights


def run_weighted_lme_analysis(merged_data, roi_name):
    """Fit weighted LME via lme4/lmerTest in R."""
    data_r = merged_data.rename(columns={
        'subject': 'Subj', 'headcoil': 'HC', 'mb': 'MB', 'me': 'ME',
        'beta': 'BetaValue', 'varcope': 'Varcope', 'weight': 'Weight'
    }).copy()
    
    data_r = data_r.dropna(subset=['BetaValue', 'Varcope', 'MB', 'ME', 'HC', 'Subj', 'smoothness', 'Weight'])
    for col in ['Subj', 'MB', 'ME', 'HC']:
        data_r[col] = data_r[col].astype(str)
    
    r_df = pandas2ri.py2rpy(data_r)
    ro.globalenv['model_data'] = r_df
    ro.globalenv['roi_name'] = roi_name
    
    ro.r('''
    library(lme4)
    library(lmerTest)
    
    model_data$MB <- factor(model_data$MB, levels = c("mb1", "mb3", "mb6"))
    model_data$ME <- factor(model_data$ME, levels = c("me1", "me4"))
    model_data$HC <- factor(model_data$HC, levels = c("20", "64"))
    
    weighted_model <- lmer(
        BetaValue ~ HC * MB * ME + smoothness + (1 | Subj),
        data = model_data,
        weights = Weight,
        REML = TRUE
    )
    
    assign(paste0("weighted_model_", roi_name), weighted_model, envir = .GlobalEnv)
    anova_result <- anova(weighted_model, type = 3, ddf = "Satterthwaite")
    ''')
    
    anova_r = ro.r('as.data.frame(anova_result)')
    anova_df = pandas2ri.rpy2py(anova_r)
    
    effect_names = {
        'HC': 'Head Coil', 'MB': 'Multiband', 'ME': 'Multi-echo',
        'smoothness': 'Smoothness',
        'HC:MB': 'Head Coil × Multiband', 'HC:ME': 'Head Coil × Multi-echo',
        'MB:ME': 'Multiband × Multi-echo', 'HC:MB:ME': 'Head Coil × Multiband × Multi-echo'
    }
    
    apa_data = []
    for effect in anova_df.index:
        effect_name = effect_names.get(effect, effect)
        apa_data.append({
            'Effect': effect_name,
            'Sum Sq': anova_df.loc[effect, 'Sum Sq'] if 'Sum Sq' in anova_df.columns else np.nan,
            'Mean Sq': anova_df.loc[effect, 'Mean Sq'] if 'Mean Sq' in anova_df.columns else np.nan,
            'Num df': anova_df.loc[effect, 'NumDF'] if 'NumDF' in anova_df.columns else np.nan,
            'Den df': anova_df.loc[effect, 'DenDF'] if 'DenDF' in anova_df.columns else np.nan,
            'F': anova_df.loc[effect, 'F value'] if 'F value' in anova_df.columns else np.nan,
            'p': anova_df.loc[effect, 'Pr(>F)'] if 'Pr(>F)' in anova_df.columns else np.nan
        })
    
    apa_table = pd.DataFrame(apa_data)
    for col in ['Sum Sq', 'Mean Sq']:
        apa_table[col] = apa_table[col].round(4)
    apa_table['Num df'] = apa_table['Num df'].round(0).astype('Int64')
    apa_table['Den df'] = apa_table['Den df'].round(2)
    apa_table['F'] = apa_table['F'].round(2)
    apa_table['p'] = apa_table['p'].apply(
        lambda x: '< .001' if pd.notna(x) and x < 0.001 else f'{x:.3f}' if pd.notna(x) else 'N/A'
    )
    
    return apa_table, data_r


def compute_weighted_emms_for_roi(roi_name):
    """Compute EMMs from the weighted model stored in R's global env."""
    ro.globalenv['roi_name'] = roi_name
    
    ro.r('''
    library(emmeans)
    weighted_model <- get(paste0("weighted_model_", roi_name))
    emm <- emmeans(weighted_model, "MB", by = c("HC", "ME"))
    emm_df <- as.data.frame(emm)
    ''')
    
    emm_df = pandas2ri.rpy2py(ro.r('emm_df'))
    
    if emm_df['MB'].dtype in ['int64', 'int32', 'float64']:
        emm_df['MB'] = emm_df['MB'].map(MB_MAP)
    if emm_df['ME'].dtype in ['int64', 'int32', 'float64']:
        emm_df['ME'] = emm_df['ME'].map(ME_MAP)
    if emm_df['HC'].dtype in ['int64', 'int32', 'float64']:
        emm_df['HC'] = emm_df['HC'].map(HC_MAP)
    
    return emm_df


# --- Load and process weighted data ---
print("\n" + "=" * 80)
print("WEIGHTED PART 1: LOADING DATA AND IDENTIFYING COMPLETE SUBJECTS")
print("=" * 80)

weighted_roi_results = {}
weighted_emm_results = {}

for roi in ROIS_W:
    print(f"\n{'='*60}")
    print(f"Processing {roi}...")
    print(f"{'='*60}")
    
    beta_data = extract_roi_data(extractions_dir, TYPE_VALUE, "beta",
                                  roi, DENOISE_VALUE, valid_subjects, headcoil_mapping)
    if beta_data is None:
        print(f"Warning: No beta data found for {roi}")
        continue
    
    varcope_data = extract_roi_data(extractions_dir, TYPE_VALUE, "varcope",
                                     roi, DENOISE_VALUE, valid_subjects, headcoil_mapping)
    if varcope_data is None:
        print(f"Warning: No varcope data found for {roi}")
        continue
    
    complete_beta = identify_complete_subjects(beta_data, 'beta')
    complete_varcope = identify_complete_subjects(varcope_data, 'varcope')
    print(f"Complete beta: {len(complete_beta)}, varcope: {len(complete_varcope)}")
    
    common_complete = sorted(list(
        set(complete_beta) & set(complete_varcope) & set(complete_smoothness)
    ))
    print(f"Common complete subjects: {len(common_complete)}")
    
    if not common_complete:
        print(f"No common complete subjects for {roi}")
        continue
    
    beta_filt = beta_data[beta_data['subject'].isin(common_complete)]
    varcope_filt = varcope_data[varcope_data['subject'].isin(common_complete)]
    smooth_filt = smoothness_data[smoothness_data['subject'].isin(common_complete)]
    
    merged = pd.merge(
        beta_filt[['subject', 'headcoil', 'mb', 'me', 'beta']],
        varcope_filt[['subject', 'mb', 'me', 'varcope']],
        on=['subject', 'mb', 'me'], how='inner'
    )
    merged = pd.merge(
        merged,
        smooth_filt[['subject', 'mb', 'me', 'smoothness']],
        on=['subject', 'mb', 'me'], how='inner'
    ).dropna()
    
    print(f"\nComputing inverse-variance weights...")
    merged['weight'] = prepare_weights(merged, winsorize_percentile=99, normalize=True)
    
    print(f"  Weight stats: min={merged['weight'].min():.3f}, "
          f"max={merged['weight'].max():.3f}, "
          f"mean={merged['weight'].mean():.3f}")
    
    weighted_roi_results[roi] = {
        'data': merged,
        'n_subjects': merged['subject'].nunique(),
        'n_20ch': len(merged[merged['headcoil'] == '20']['subject'].unique()),
        'n_64ch': len(merged[merged['headcoil'] == '64']['subject'].unique()),
        'n_observations': len(merged)
    }

# --- Run weighted LME analyses ---
print("\n" + "=" * 80)
print("WEIGHTED PART 2: WEIGHTED LINEAR MIXED EFFECTS ANALYSES")
print("=" * 80)

for roi in ROIS_W:
    if roi not in weighted_roi_results:
        continue
    
    print(f"\n{'='*60}")
    print(f"{ROI_LABELS_W[roi]} ({roi})")
    print(f"{'='*60}")
    
    rd = weighted_roi_results[roi]
    print(f"N = {rd['n_subjects']} ({rd['n_20ch']} 20-ch, {rd['n_64ch']} 64-ch), "
          f"{rd['n_observations']} observations")
    
    print("\nFitting weighted mixed model...")
    apa_table, model_data = run_weighted_lme_analysis(rd['data'], roi)
    
    print(f"\nAPA-Style ANOVA Table (Weighted LME):")
    print(f"Model: beta ~ HC * MB * ME + smoothness + (1 | Subj), weights = 1/varcope")
    print(f"Data: {ROI_LABELS_W[roi]} ROI\n")
    print(apa_table.to_string(index=False))
    
    anova_file = output_dir / f'{roi}_weighted_beta_lme_anova.csv'
    apa_table.to_csv(anova_file, index=False)
    print(f"Saved to '{anova_file.name}'")
    
    print("\nComputing estimated marginal means...")
    emm_df = compute_weighted_emms_for_roi(roi)
    weighted_emm_results[roi] = emm_df
    
    emm_file = output_dir / f'{roi}_weighted_beta_emm.csv'
    emm_df.to_csv(emm_file, index=False)
    print(f"EMMs saved to '{emm_file.name}'")

# --- Nature-quality weighted visualization ---
print("\n" + "=" * 80)
print("WEIGHTED PART 3: GENERATING NATURE-QUALITY EMM PLOTS WITH BRAIN IMAGES")
print("=" * 80)

generate_emm_grid_plot_with_brain_images(
    ROIS_W, ROI_LABELS_W, ROI_YLABELS_W, weighted_roi_results, weighted_emm_results,
    img_value='beta',
    plot_filename=combined_emm_plot_file,
    fig_ylabel='Weighted Activation (Beta EMM)',
    legend_loc='lower right'
)

# --- Weight diagnostics ---
print("\n" + "=" * 80)
print("WEIGHT DISTRIBUTION DIAGNOSTICS")
print("=" * 80)

for roi in ROIS_W:
    if roi not in weighted_roi_results:
        continue
    data = weighted_roi_results[roi]['data']
    print(f"\n{ROI_LABELS_W[roi]}:")
    print(f"  Varcope range: [{data['varcope'].min():.6f}, {data['varcope'].max():.6f}]")
    print(f"  Weight range: [{data['weight'].min():.3f}, {data['weight'].max():.3f}]")
    
    ratio = data['weight'].max() / data['weight'].min()
    if ratio > 100:
        print(f"  Warning: Large weight ratio ({ratio:.1f}x)")

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{'=' * 80}")
print("ROI ACTIVATION ANALYSIS COMPLETE - SUMMARY")
print(f"{'=' * 80}")
print("Analyses run: Beta (unweighted), Z-stat (unweighted), Weighted Beta (1/varcope)")
print(f"\nOutputs saved to: {output_dir}")
print("=" * 80)

# Motor vs Cerebellum ROI Comparison

Dedicated analysis comparing Motor Cortex and Cerebellum ROIs.
Includes a **combined 4-factor LME** model with ROI as a factor,
plus per-ROI EMM visualization. Runs for beta, z-stat, and weighted beta (1/varcope).

In [ ]:
# =============================================================================
# MOTOR VS CEREBELLUM ROI COMPARISON — BETA, Z-STAT, AND WEIGHTED BETA
# =============================================================================
# Nature-quality 3×3 grid: brain image + 20-ch + 64-ch per row.
#
# Dependencies:
#   - project_root, output_dir, extractions_dir (from Setup)
#   - valid_subjects, headcoil_mapping (from Demographics)
#   - smoothness_data, complete_smoothness (from Shared Dependencies)
#   - extract_roi_data, identify_complete_subjects, compute_emms (from Setup)
#   - load_and_merge_roi_data, build_apa_table (from Setup)

import matplotlib.image as mpimg

# =============================================================================
# NATURE-QUALITY PLOTTING FUNCTIONS (local to this cell)
# =============================================================================

# Color scheme for multi-echo
ME_COLORS_NATURE = {'me1': '#4A90A4', 'me4': '#E07B39'}

# Brain image configuration for Motor vs Cerebellum
BRAIN_IMG_DIR_MC = project_root / 'derivatives' / 'figures' / 'brain_images'
ROI_BRAIN_IMAGES_MC = {
    'bilateralMotor': {
        'file': 'mask_bilateralMotor.png',
        'slice_label': 'y = −24'
    },
    'bilateralCerebellum': {
        'file': 'mask_bilateralCerebellum.png',
        'slice_label': 'y = −52'
    }
}


def plot_emm_panel_mc(ax, emm_df, raw_data, headcoil, n_subjects, img_value, me_colors=None):
    """
    Plot EMM data for a single headcoil with individual data points overlay.
    """
    if me_colors is None:
        me_colors = ME_COLORS_NATURE
    
    coil_emm = emm_df[emm_df['HC'] == headcoil]
    coil_raw = raw_data[raw_data['headcoil'] == headcoil] if raw_data is not None else None
    
    if coil_emm.empty:
        ax.set_title(f"{headcoil}-ch (n=0)", fontsize=12)
        return
    
    x_positions = {'mb1': 0, 'mb3': 1, 'mb6': 2}
    jitter_emm = 0.08
    jitter_raw = 0.12
    me_offsets = {'me1': -jitter_emm, 'me4': jitter_emm}
    
    # --- Individual data points (behind EMMs) ---
    #if coil_raw is not None and len(coil_raw) > 0:
    #    np.random.seed(42)
    #    for me in ['me1', 'me4']:
    #        me_raw = coil_raw[coil_raw['me'] == me]
    #        for mb in ['mb1', 'mb3', 'mb6']:
    #            mb_me_data = me_raw[me_raw['mb'] == mb][img_value].dropna()
    #            if len(mb_me_data) > 0:
    #                x_base = x_positions[mb] + me_offsets[me]
    #                jitter_vals = np.random.uniform(-jitter_raw, jitter_raw, size=len(mb_me_data))
    #                ax.scatter(
    #                    np.full(len(mb_me_data), x_base) + jitter_vals,
    #                    mb_me_data.values,
    #                    color=me_colors[me], s=20, alpha=0.25,
    #                    edgecolors='none', zorder=1
    #                )
    
    # --- EMM lines and error bars ---
    for me in ['me1', 'me4']:
        me_data = coil_emm[coil_emm['ME'] == me].copy()
        if me_data.empty:
            continue
        me_data = me_data.sort_values('MB', key=lambda x: x.str.extract(r'mb(\d+)')[0].astype(int))
        x_vals = [x_positions[mb] + me_offsets[me] for mb in me_data['MB']]
        
        ax.plot(x_vals, me_data['emmean'], 
                color=me_colors[me], linewidth=2, linestyle='-', zorder=3)
        
        ax.errorbar(x_vals, me_data['emmean'], yerr=me_data['SE'],
                    fmt='o', color=me_colors[me], markersize=7,
                    capsize=3, capthick=1.5, elinewidth=1.5,
                    markeredgecolor='white', markeredgewidth=0.8, zorder=4)
    
    # --- Axis formatting ---
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['1', '3', '6'], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.set_title(f"{headcoil}-ch (n={n_subjects})", fontsize=12, fontweight='normal')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_linewidth(1)
    ax.grid(False)


def compute_emm_ylimits_mc(emm_results_dict, raw_data_dict=None, img_value=None):
    """Compute shared y-limits across EMMs and raw data."""
    all_y, all_e = [], []
    
    for emm_df in emm_results_dict.values():
        all_y.extend(emm_df['emmean'].values)
        all_e.extend(emm_df['SE'].values)
    
    if raw_data_dict is not None and img_value is not None:
        for raw_df in raw_data_dict.values():
            if img_value in raw_df.columns:
                vals = raw_df[img_value].dropna().values
                all_y.extend(vals)
                all_e.extend([0] * len(vals))
    
    if all_y:
        y_max = max(v + e for v, e in zip(all_y, all_e))
        y_min = min(v - e for v, e in zip(all_y, all_e))
        margin = (y_max - y_min) * 0.08
        return (y_min - margin, y_max + margin)
    return (-1, 1)


def generate_mc_grid_plot_with_brain_images(rois, roi_labels, roi_ylabels, roi_results, 
                                             emm_results, img_value, plot_filename, 
                                             fig_ylabel=None, legend_loc='upper right'):
    """
    Generate Nature-quality 2×3 grid for Motor vs Cerebellum: brain image + 20-ch + 64-ch per row.
    """
    n_rois = len(rois)
    
    # Load brain images first to get aspect ratios
    brain_images = {}
    brain_aspects = []
    for roi in rois:
        if roi in ROI_BRAIN_IMAGES_MC:
            img_path = BRAIN_IMG_DIR_MC / ROI_BRAIN_IMAGES_MC[roi]['file']
            if img_path.exists():
                brain_img = mpimg.imread(str(img_path))
                brain_images[roi] = brain_img
                img_h, img_w = brain_img.shape[:2]
                brain_aspects.append(img_w / img_h)
            else:
                print(f"  Warning: Brain image not found: {img_path}")
                brain_images[roi] = None
                brain_aspects.append(1.0)
        else:
            brain_images[roi] = None
            brain_aspects.append(1.0)
    
    # Use average brain aspect for consistent layout
    avg_brain_aspect = sum(brain_aspects) / len(brain_aspects) if brain_aspects else 1.0
    scatter_ratio = 1.2
    
    # Figure sizing: 3 columns (brain + 20ch + 64ch)
    fig_height = 3.5 * n_rois + 0.3
    fig, axes = plt.subplots(n_rois, 3, figsize=(13, fig_height),
                              gridspec_kw={'width_ratios': [avg_brain_aspect * 0.8, scatter_ratio, scatter_ratio],
                                          'wspace': 0.25, 'hspace': 0.45})
    
    if n_rois == 1:
        axes = axes.reshape(1, -1)
    
    # Compute y-limits including raw data
    raw_data_dict = {roi: roi_results[roi]['data'] for roi in rois if roi in roi_results}
    y_limits = compute_emm_ylimits_mc(emm_results, raw_data_dict, img_value)
    
    # Panel label tracking
    panel_labels = 'abcdef'
    panel_idx = 0
    
    for roi_idx, roi in enumerate(rois):
        if roi not in emm_results or roi not in roi_results:
            continue
        
        emm_df = emm_results[roi]
        roi_data = roi_results[roi]
        raw_data = roi_data['data']
        
        ax_brain = axes[roi_idx, 0]
        ax_20ch = axes[roi_idx, 1]
        ax_64ch = axes[roi_idx, 2]
        
        # --- Brain image panel ---
        if brain_images[roi] is not None:
            ax_brain.imshow(brain_images[roi])
        ax_brain.axis('off')
        
        # --- Panel labels using ax.transAxes (BEFORE tight_layout) ---
        ax_brain.text(0.0, 1.08, panel_labels[panel_idx], transform=ax_brain.transAxes,
                      fontsize=16, fontweight='bold', va='bottom', ha='left')
        ax_20ch.text(-0.15, 1.08, panel_labels[panel_idx + 1], transform=ax_20ch.transAxes,
                     fontsize=16, fontweight='bold', va='bottom', ha='left')
        ax_64ch.text(-0.15, 1.08, panel_labels[panel_idx + 2], transform=ax_64ch.transAxes,
                     fontsize=16, fontweight='bold', va='bottom', ha='left')
        panel_idx += 3
        
        # --- Slice label below brain image ---
        if roi in ROI_BRAIN_IMAGES_MC:
            ax_brain.text(0.5, -0.06, ROI_BRAIN_IMAGES_MC[roi]['slice_label'],
                          transform=ax_brain.transAxes,
                          fontsize=11, ha='center', va='top')
        
        # --- 20-channel panel ---
        plot_emm_panel_mc(ax_20ch, emm_df, raw_data, '20', 
                          roi_data['n_20ch'], img_value)
        ax_20ch.set_ylim(y_limits)
        
        # --- 64-channel panel ---
        plot_emm_panel_mc(ax_64ch, emm_df, raw_data, '64', 
                          roi_data['n_64ch'], img_value)
        ax_64ch.set_ylim(y_limits)
        
        # Y-axis label on 20-ch panel only
        ax_20ch.set_ylabel(roi_ylabels.get(roi, ''), fontsize=13)
    
    # X-axis label only on bottom row
    for col in [1, 2]:
        axes[-1, col].set_xlabel('Multiband Factor', fontsize=13)
    
    # Legend in top-right panel (64-ch of first row)
    handles = [
        plt.Line2D([0], [0], color=ME_COLORS_NATURE['me1'], marker='o', markersize=6, 
                   linewidth=2, markeredgecolor='white', markeredgewidth=0.5, label='ME=1'),
        plt.Line2D([0], [0], color=ME_COLORS_NATURE['me4'], marker='o', markersize=6, 
                   linewidth=2, markeredgecolor='white', markeredgewidth=0.5, label='ME=4')
    ]
    axes[0, 2].legend(handles=handles, fontsize=10, loc=legend_loc, frameon=False)
    
    # Global y-axis label
    if fig_ylabel is None:
        label_map = {'beta': 'Beta', 'zstat': 'Z-Stat'}
        fig_ylabel = f'Activation ({label_map.get(img_value, img_value)} EMM)'
    
    plt.tight_layout()
    
    # Global y-axis label (rotated, on left side)
    fig.text(0.01, 0.5, fig_ylabel, va='center', rotation='vertical', 
             fontsize=14, transform=fig.transFigure)
    
    # Adjust layout to make room for labels
    fig.subplots_adjust(left=0.10, top=0.94, hspace=0.45)
    
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  EMM plot saved: {plot_filename}")
    plt.show()
    
    return fig


# =============================================================================
# MAIN ANALYSIS FUNCTION
# =============================================================================

def run_motor_cerebellum_comparison_nature(img_value, valid_subjects, headcoil_mapping,
                                            smoothness_data, complete_smoothness):
    """
    Run the Motor vs Cerebellum comparison analysis with Nature-quality visualization.
    """
    config = ROI_CONFIGS['motor_cerebellum']
    rois = config['rois']
    roi_labels = config['labels']
    roi_ylabels = config['ylabels']

    label_map = {'beta': 'Beta', 'zstat': 'Z-Stat'}
    label = label_map.get(img_value, img_value)

    print("=" * 80)
    print(f"MOTOR VS CEREBELLUM ROI COMPARISON — {label.upper()} ANALYSIS WITH EMMs")
    print("=" * 80)

    # Part 1: Load data and compute per-ROI EMMs
    print("\n" + "=" * 80)
    print("PART 1: LOADING DATA AND IDENTIFYING COMPLETE SUBJECTS")
    print("=" * 80)

    roi_results = load_and_merge_roi_data(
        rois, img_value, smoothness_data, complete_smoothness,
        valid_subjects, headcoil_mapping
    )

    emm_results = {}
    for roi in rois:
        if roi in roi_results:
            print(f"Computing EMMs for {roi}...")
            emm_results[roi] = compute_emms(roi_results[roi]['data'], img_value, roi)

    # Part 2: Combined ROI LME analysis
    print("\n" + "=" * 80)
    print("PART 2: COMBINED LINEAR MIXED EFFECTS ANALYSIS (MOTOR VS CEREBELLUM)")
    print("=" * 80)

    combined_roi_data = []
    for roi in rois:
        if roi in roi_results:
            rd = roi_results[roi]['data'].copy()
            rd['roi'] = roi
            combined_roi_data.append(rd)

    if len(combined_roi_data) == 2:
        combined_df = pd.concat(combined_roi_data, ignore_index=True)
        combined_df['roi'] = pd.Categorical(combined_df['roi'], categories=rois)

        print(f"\nCombined dataset: {len(combined_df)} observations")
        print(f"Subjects: {combined_df['subject'].nunique()}")
        print(f"ROI breakdown: {combined_df['roi'].value_counts().to_dict()}")

        # Fit 4-factor model
        model = Lmer(f'{img_value} ~ headcoil * mb * me * roi + smoothness + (1 | subject)',
                     data=combined_df)
        model.fit()
        anova_results = model.anova()

        apa_table = build_apa_table(model, anova_results,
                                     ROI_COMPARISON_EFFECT_NAMES, ROI_COMPARISON_DF_DICT)

        print(f"\nAPA-Style ANOVA Table for Motor vs Cerebellum ROI Analysis:")
        print(f"Model: {img_value} ~ headcoil * mb * me * roi + smoothness + (1 | subject)\n")
        print(apa_table.to_string(index=False))

        anova_file = output_dir / f'motor_cerebellum_roi_{img_value}_lme_anova_with_smoothness.csv'
        apa_table.to_csv(anova_file, index=False)
        print(f"\nSaved to '{anova_file.name}'")
    else:
        print("Could not find both Motor and Cerebellum ROI data for combined analysis")

    # Part 3: Nature-quality visualization with brain images
    print("\n" + "=" * 80)
    print("PART 3: GENERATING NATURE-QUALITY EMM PLOTS WITH BRAIN IMAGES")
    print("=" * 80)

    plot_file = output_dir / f'motor_cerebellum_{img_value}_emm_plot_publication.png'
    generate_mc_grid_plot_with_brain_images(
        rois, roi_labels, roi_ylabels, roi_results, emm_results,
        img_value, plot_file, legend_loc='lower right'
    )

    # Summary
    print("\n" + "=" * 80)
    print("ANALYSIS COMPLETE — SUMMARY")
    print("=" * 80)
    for roi in rois:
        if roi in roi_results:
            r = roi_results[roi]
            print(f"  {roi_labels[roi]}: {r['n_subjects']} total ({r['n_20ch']} 20-ch, {r['n_64ch']} 64-ch)")
    print("=" * 80)

    return roi_results, emm_results


# --- Run for Beta ---
print("\n" + "#" * 80)
print("# MOTOR VS CEREBELLUM — BETA")
print("#" * 80 + "\n")

mc_beta_results, mc_beta_emms = run_motor_cerebellum_comparison_nature(
    'beta', valid_subjects, headcoil_mapping, smoothness_data, complete_smoothness
)

# --- Run for Z-Stat (Supplement) ---
print("\n" + "#" * 80)
print("# MOTOR VS CEREBELLUM — Z-STAT (SUPPLEMENT)")
print("#" * 80 + "\n")

mc_zstat_results, mc_zstat_emms = run_motor_cerebellum_comparison_nature(
    'zstat', valid_subjects, headcoil_mapping, smoothness_data, complete_smoothness
)


# =============================================================================
# WEIGHTED MOTOR VS CEREBELLUM ANALYSIS (Inverse-Variance Weighting)
# =============================================================================

print("\n" + "#" * 80)
print("# MOTOR VS CEREBELLUM — WEIGHTED BETA (INVERSE-VARIANCE)")
print("#" * 80 + "\n")


def prepare_weights_mc(merged_data, winsorize_percentile=99, normalize=True):
    """Compute inverse-variance weights from varcope values."""
    weights = 1.0 / merged_data['varcope']
    if winsorize_percentile < 100:
        weight_cap = np.percentile(weights, winsorize_percentile)
        weights = weights.clip(upper=weight_cap)
        n_capped = (weights == weight_cap).sum()
        if n_capped > 0:
            print(f"  Capped {n_capped} weights at {winsorize_percentile}th percentile")
    if normalize:
        weights = weights / weights.mean()
    return weights


def run_weighted_lme_mc(merged_data, roi_name):
    """Fit weighted LME via lme4/lmerTest in R. Returns APA table."""
    data_r = merged_data.rename(columns={
        'subject': 'Subj', 'headcoil': 'HC', 'mb': 'MB', 'me': 'ME',
        'beta': 'BetaValue', 'varcope': 'Varcope', 'weight': 'Weight'
    }).copy()
    data_r = data_r.dropna(subset=['BetaValue', 'Varcope', 'MB', 'ME', 'HC', 'Subj', 'smoothness', 'Weight'])
    for col in ['Subj', 'MB', 'ME', 'HC']:
        data_r[col] = data_r[col].astype(str)

    r_df = pandas2ri.py2rpy(data_r)
    ro.globalenv['model_data'] = r_df
    ro.globalenv['roi_name'] = roi_name

    ro.r('''
    library(lme4); library(lmerTest)
    model_data$MB <- factor(model_data$MB, levels = c("mb1", "mb3", "mb6"))
    model_data$ME <- factor(model_data$ME, levels = c("me1", "me4"))
    model_data$HC <- factor(model_data$HC, levels = c("20", "64"))
    weighted_model <- lmer(
        BetaValue ~ HC * MB * ME + smoothness + (1 | Subj),
        data = model_data, weights = Weight, REML = TRUE
    )
    assign(paste0("weighted_model_", roi_name), weighted_model, envir = .GlobalEnv)
    anova_result <- anova(weighted_model, type = 3, ddf = "Satterthwaite")
    ''')

    anova_r = ro.r('as.data.frame(anova_result)')
    anova_df = pandas2ri.rpy2py(anova_r)

    effect_names = {
        'HC': 'Head Coil', 'MB': 'Multiband', 'ME': 'Multi-echo',
        'smoothness': 'Smoothness',
        'HC:MB': 'Head Coil × Multiband', 'HC:ME': 'Head Coil × Multi-echo',
        'MB:ME': 'Multiband × Multi-echo', 'HC:MB:ME': 'Head Coil × Multiband × Multi-echo'
    }

    apa_data = []
    for effect in anova_df.index:
        apa_data.append({
            'Effect': effect_names.get(effect, effect),
            'Sum Sq': anova_df.loc[effect, 'Sum Sq'] if 'Sum Sq' in anova_df.columns else np.nan,
            'Mean Sq': anova_df.loc[effect, 'Mean Sq'] if 'Mean Sq' in anova_df.columns else np.nan,
            'Num df': anova_df.loc[effect, 'NumDF'] if 'NumDF' in anova_df.columns else (
                      anova_df.loc[effect, 'Df'] if 'Df' in anova_df.columns else np.nan),
            'Den df': anova_df.loc[effect, 'DenDF'] if 'DenDF' in anova_df.columns else np.nan,
            'F': anova_df.loc[effect, 'F value'] if 'F value' in anova_df.columns else (
                 anova_df.loc[effect, 'F.value'] if 'F.value' in anova_df.columns else np.nan),
            'p': anova_df.loc[effect, 'Pr(>F)'] if 'Pr(>F)' in anova_df.columns else (
                 anova_df.loc[effect, 'Pr..F.'] if 'Pr..F.' in anova_df.columns else np.nan)
        })

    apa_table = pd.DataFrame(apa_data)
    for col in ['Sum Sq', 'Mean Sq']:
        apa_table[col] = apa_table[col].round(4)
    apa_table['Num df'] = apa_table['Num df'].round(0).astype('Int64')
    apa_table['Den df'] = apa_table['Den df'].round(2)
    apa_table['F'] = apa_table['F'].round(2)
    apa_table['p'] = apa_table['p'].apply(
        lambda x: '< .001' if pd.notna(x) and x < 0.001 else f'{x:.3f}' if pd.notna(x) else 'N/A'
    )
    return apa_table, data_r


def compute_weighted_emms_mc(roi_name):
    """Compute EMMs from the weighted model stored in R's global env."""
    ro.globalenv['roi_name'] = roi_name
    ro.r('''
    library(emmeans)
    weighted_model <- get(paste0("weighted_model_", roi_name))
    emm <- emmeans(weighted_model, "MB", by = c("HC", "ME"))
    emm_df <- as.data.frame(emm)
    ''')
    emm_df = pandas2ri.rpy2py(ro.r('emm_df'))
    if emm_df['MB'].dtype in ['int64', 'int32', 'float64']:
        emm_df['MB'] = emm_df['MB'].map(MB_MAP)
    if emm_df['ME'].dtype in ['int64', 'int32', 'float64']:
        emm_df['ME'] = emm_df['ME'].map(ME_MAP)
    if emm_df['HC'].dtype in ['int64', 'int32', 'float64']:
        emm_df['HC'] = emm_df['HC'].map(HC_MAP)
    return emm_df


# --- Load data and run weighted analysis for Motor & Cerebellum ---
mc_config = ROI_CONFIGS['motor_cerebellum']
mc_rois_w = mc_config['rois']
mc_labels_w = mc_config['labels']
mc_ylabels_w = mc_config['ylabels']

mc_weighted_results = {}
mc_weighted_emms = {}

for roi in mc_rois_w:
    print(f"\n{'='*60}")
    print(f"Processing {roi} (weighted)...")
    print(f"{'='*60}")

    beta_data = extract_roi_data(extractions_dir, TYPE_VALUE, "beta",
                                  roi, DENOISE_VALUE, valid_subjects, headcoil_mapping)
    if beta_data is None:
        print(f"Warning: No beta data for {roi}")
        continue

    varcope_data = extract_roi_data(extractions_dir, TYPE_VALUE, "varcope",
                                     roi, DENOISE_VALUE, valid_subjects, headcoil_mapping)
    if varcope_data is None:
        print(f"Warning: No varcope data for {roi}")
        continue

    complete_beta = identify_complete_subjects(beta_data, 'beta')
    complete_varcope = identify_complete_subjects(varcope_data, 'varcope')
    common = sorted(list(set(complete_beta) & set(complete_varcope) & set(complete_smoothness)))
    print(f"Complete subjects — beta: {len(complete_beta)}, varcope: {len(complete_varcope)}, common: {len(common)}")

    if not common:
        print(f"No common complete subjects for {roi}")
        continue

    merged = pd.merge(
        beta_data[beta_data['subject'].isin(common)][['subject', 'headcoil', 'mb', 'me', 'beta']],
        varcope_data[varcope_data['subject'].isin(common)][['subject', 'mb', 'me', 'varcope']],
        on=['subject', 'mb', 'me'], how='inner'
    )
    merged = pd.merge(
        merged,
        smoothness_data[smoothness_data['subject'].isin(common)][['subject', 'mb', 'me', 'smoothness']],
        on=['subject', 'mb', 'me'], how='inner'
    ).dropna()

    merged['weight'] = prepare_weights_mc(merged)
    print(f"  Weight stats: min={merged['weight'].min():.3f}, max={merged['weight'].max():.3f}, "
          f"mean={merged['weight'].mean():.3f}")

    mc_weighted_results[roi] = {
        'data': merged,
        'n_subjects': merged['subject'].nunique(),
        'n_20ch': len(merged[merged['headcoil'] == '20']['subject'].unique()),
        'n_64ch': len(merged[merged['headcoil'] == '64']['subject'].unique()),
    }

    # Fit weighted model
    print(f"\nFitting weighted LME for {roi}...")
    apa_table, _ = run_weighted_lme_mc(merged, roi)
    print(f"\nWeighted ANOVA ({mc_labels_w[roi]}):")
    print(apa_table.to_string(index=False))

    anova_file = output_dir / f'{roi}_weighted_beta_lme_anova.csv'
    apa_table.to_csv(anova_file, index=False)

    # Compute weighted EMMs
    emm_df = compute_weighted_emms_mc(roi)
    mc_weighted_emms[roi] = emm_df
    emm_file = output_dir / f'{roi}_weighted_beta_emm.csv'
    emm_df.to_csv(emm_file, index=False)
    print(f"Saved ANOVA and EMMs for {roi}")

# --- Weighted EMM Plot (Nature-quality) ---
if mc_weighted_emms:
    print("\n" + "=" * 80)
    print("GENERATING WEIGHTED EMM PLOT WITH BRAIN IMAGES")
    print("=" * 80)
    
    plot_file_w = output_dir / 'motor_cerebellum_weighted_beta_emm_plot_publication.png'
    generate_mc_grid_plot_with_brain_images(
        mc_rois_w, mc_labels_w, mc_ylabels_w, mc_weighted_results, mc_weighted_emms,
        img_value='beta',
        plot_filename=plot_file_w,
        fig_ylabel='Weighted Activation (Beta EMM)',
        legend_loc='lower right'
    )

    # Weight diagnostics
    print("\n" + "=" * 80)
    print("WEIGHT DISTRIBUTION DIAGNOSTICS")
    print("=" * 80)
    for roi in mc_rois_w:
        if roi not in mc_weighted_results:
            continue
        data = mc_weighted_results[roi]['data']
        print(f"\n{mc_labels_w[roi]}:")
        print(f"  Varcope range: [{data['varcope'].min():.6f}, {data['varcope'].max():.6f}]")
        print(f"  Weight range: [{data['weight'].min():.3f}, {data['weight'].max():.3f}]")

print("\n" + "=" * 80)
print("MOTOR VS CEREBELLUM — ALL ANALYSES COMPLETE")
print("=" * 80)

# Statistical Mask Analyses — PPI Beta & Z-stat (9 ROIs)
LME on PPI connectivity (seed: VS) within 9 statistical mask regions. Both beta and z-stat versions.

### MB×ME interaction on VS-seeded PPI connectivity (zstat17)

In [ ]:
# =============================================================================
# PPI CONNECTIVITY: MB×ME Interaction — Left Ventral Striatum (Fig. 7)
# =============================================================================
# This mask shows a significant MB×ME interaction effect on VS-seeded PPI 
# connectivity (zstat17 = physiological regressor). Single clean cluster in 
# left VS (140 voxels, centroid: x=-10, y=12, z=-4 MNI).
#
# Generates a combined figure with rows for each image type (beta, zstat, varcope)
# and columns for brain image + 20-ch + 64-ch.
#
# Dependencies: project_root, output_dir, valid_subjects, headcoil_mapping,
#               ME_COLORS_NATURE (from Setup)

import matplotlib.image as mpimg

print("=" * 80)
print("PPI CONNECTIVITY: MB×ME Interaction — Left Ventral Striatum")
print("=" * 80)

# --- Config ---
MASK_NAME = "MBxME_ppi_zstat17_thresh"
MASK_LABEL = "MB×ME (Left VS)"
IMG_TYPES = ['beta', 'zstat', 'weighted_beta']  # weighted_beta requires both beta and varcope extractions

extractions_dir = project_root / 'derivatives' / 'extractions'
brain_img_dir = project_root / 'derivatives' / 'figures' / 'brain_images'

# Brain images: Seed (VS) and Target (MBxME cluster)
SEED_IMAGE = {'file': 'mask_VSconstrained.png', 'slice_label': 'y = 15'}
TARGET_IMAGE = {'file': 'ppi_MBxME_leftVS.png', 'slice_label': 'y = 15'}

# Nature-quality color scheme
ME_COLORS_NATURE = {'me1': '#4A90A4', 'me4': '#E07B39'}

# Y-axis labels for each image type
YLABEL_MAP = {'beta': 'PPI Beta', 'zstat': 'PPI Z-stat', 'weighted_beta': 'PPI Beta (weighted)'}


# =============================================================================
# LOCAL FUNCTIONS
# =============================================================================

def load_ppi_extractions(extractions_dir, mask_name, img_type, valid_subjects, headcoil_mapping):
    """
    Load PPI extraction files for a given mask and image type.
    Returns DataFrame with subject, mb, me, headcoil, and value columns.
    """
    pattern = f"ts_sub-*_acq_*_type-ppi_seed-VS_thr5_img-{img_type}_mask-{mask_name}_denoise_base.txt"
    files = list(extractions_dir.glob(pattern))
    
    if not files:
        return None
    
    data_rows = []
    for file_path in files:
        parts = file_path.name.split('_')
        try:
            subject = parts[1].replace('sub-', '')
            acq_part = parts[3].replace('acq-', '')
            
            mb = next((f'mb{i}' for i in [1, 3, 6] if f'mb{i}' in acq_part), None)
            me = next((f'me{i}' for i in [1, 4] if f'me{i}' in acq_part), None)
            if not mb or not me:
                continue
            
            if subject not in valid_subjects:
                continue
            
            headcoil = headcoil_mapping.get(subject)
            if headcoil is None:
                continue
            
            with open(file_path, 'r') as f:
                value = float(f.read().strip())
            
            data_rows.append({
                'subject': subject, 'mb': mb, 'me': me,
                'headcoil': str(headcoil), 'value': value
            })
        except Exception as e:
            continue
    
    if not data_rows:
        return None
    
    df = pd.DataFrame(data_rows)
    df['mb'] = pd.Categorical(df['mb'], categories=['mb1', 'mb3', 'mb6'], ordered=True)
    df['me'] = pd.Categorical(df['me'], categories=['me1', 'me4'], ordered=True)
    df['headcoil'] = pd.Categorical(df['headcoil'], categories=['20', '64'], ordered=True)
    
    return df


def run_ppi_lme(data, img_type, mask_name, output_dir):
    """Run LME analysis for PPI data and save results."""
    df_lme = data.copy()
    
    model = Lmer(f'value ~ headcoil * mb * me + (1|subject)', data=df_lme)
    model.fit(summarize=False)
    anova_results = model.anova()
    
    print(f"\nANOVA Results ({img_type}):")
    print(anova_results)
    
    # Save ANOVA table
    anova_file = output_dir / f"ppi_{mask_name}_{img_type}_anova.csv"
    anova_results.to_csv(anova_file)
    print(f"Saved: {anova_file.name}")
    
    return model, anova_results


def compute_weights(varcope_values):
    """Compute inverse-variance weights from varcope values."""
    weights = 1.0 / varcope_values
    # Winsorize at 99th percentile
    cap = np.percentile(weights, 99)
    weights = np.clip(weights, None, cap)
    # Normalize to mean=1
    weights = weights / weights.mean()
    return weights


def run_weighted_ppi_lme(data, mask_name, output_dir):
    """
    Fit weighted LME via lme4/lmerTest in R.
    Weights = 1/varcope (inverse variance weighting).
    """
    from rpy2.robjects import r, pandas2ri, globalenv
    from rpy2.robjects.packages import importr
    
    lme4 = importr('lme4')
    lmerTest = importr('lmerTest')
    
    # Prepare data for R
    df_r = data.copy()
    df_r = df_r.rename(columns={
        'value': 'BetaValue', 'varcope': 'Varcope', 'weight': 'Weight',
        'headcoil': 'HC', 'mb': 'MB', 'me': 'ME', 'subject': 'Subj'
    })
    
    for col in ['HC', 'MB', 'ME', 'Subj']:
        df_r[col] = df_r[col].astype(str)
    
    # Convert to R dataframe
    with (ro.default_converter + pandas2ri.converter).context():
        r_df = ro.conversion.py2rpy(df_r)
    globalenv['ppi_data'] = r_df
    
    # Fit weighted model in R
    r('''
    ppi_data$HC <- factor(ppi_data$HC)
    ppi_data$MB <- factor(ppi_data$MB)
    ppi_data$ME <- factor(ppi_data$ME)
    ppi_data$Subj <- factor(ppi_data$Subj)
    
    weighted_model <- lmer(
        BetaValue ~ HC * MB * ME + (1 | Subj),
        data = ppi_data,
        weights = Weight
    )
    
    anova_result <- anova(weighted_model, type = 3, ddf = "Satterthwaite")
    ''')
    
    # Extract ANOVA results
    anova_r = r('anova_result')
    with (ro.default_converter + pandas2ri.converter).context():
        anova_df = ro.conversion.rpy2py(anova_r)
    
    print(f"\nANOVA Results (weighted_beta):")
    print(anova_df)
    
    # Save ANOVA table
    anova_file = output_dir / f"ppi_{mask_name}_weighted_beta_anova.csv"
    anova_df.to_csv(anova_file)
    print(f"Saved: {anova_file.name}")
    
    return None, anova_df  # model object stays in R


def compute_ppi_cell_stats(data, img_type):
    """Compute cell means and SEs for plotting (no smoothness covariate for PPI)."""
    summary = data.groupby(['headcoil', 'mb', 'me']).agg(
        mean_val=('value', 'mean'),
        se_val=('value', lambda x: x.std() / np.sqrt(len(x))),
        n=('value', 'count')
    ).reset_index()
    return summary


def plot_ppi_emm_style(ax, raw_data, summary_data, headcoil, y_limits, me_colors, 
                        img_type, show_legend=False):
    """
    Plot PPI values with jittered individual points + cell mean markers + SE error bars.
    """
    coil_raw = raw_data[raw_data['headcoil'] == headcoil] if raw_data is not None else None
    coil_summary = summary_data[summary_data['headcoil'] == headcoil] if summary_data is not None else None
    n_subjects = coil_raw['subject'].nunique() if coil_raw is not None else 0
    
    if coil_summary is None or coil_summary.empty:
        ax.set_title(f"{headcoil}-ch (n=0)", fontsize=12)
        ax.set_ylim(y_limits)
        return
    
    x_positions = {'mb1': 0, 'mb3': 1, 'mb6': 2}
    jitter_mean = 0.08
    jitter_raw = 0.12
    me_offsets = {'me1': -jitter_mean, 'me4': jitter_mean}
    
    # --- Individual data points (behind means) ---
    #if coil_raw is not None and len(coil_raw) > 0:
    #    np.random.seed(42)
    #    for me in ['me1', 'me4']:
    #        me_raw = coil_raw[coil_raw['me'] == me]
    #        for mb in ['mb1', 'mb3', 'mb6']:
    #            mb_me_data = me_raw[me_raw['mb'] == mb]['value'].dropna()
    #            if len(mb_me_data) > 0:
    #                x_base = x_positions[mb] + me_offsets[me]
    #                jitter_vals = np.random.uniform(-jitter_raw, jitter_raw, size=len(mb_me_data))
    #                ax.scatter(
    #                    np.full(len(mb_me_data), x_base) + jitter_vals,
    #                    mb_me_data.values,
    #                    color=me_colors[me], s=20, alpha=0.25,
    #                    edgecolors='none', zorder=1
    #                )
    
    # --- Mean lines connecting MB levels ---
    for me in ['me1', 'me4']:
        me_data = coil_summary[coil_summary['me'] == me].copy()
        if me_data.empty:
            continue
        me_data = me_data.sort_values('mb', key=lambda x: x.astype(str).str.extract(r'mb(\d+)')[0].astype(int))
        x_vals = [x_positions[mb] + me_offsets[me] for mb in me_data['mb']]
        
        ax.plot(x_vals, me_data['mean_val'], 
                color=me_colors[me], linewidth=2, linestyle='-', zorder=3,
                label=me if show_legend else None)
        
        ax.errorbar(x_vals, me_data['mean_val'], yerr=me_data['se_val'],
                    fmt='o', color=me_colors[me], markersize=7,
                    capsize=3, capthick=1.5, elinewidth=1.5,
                    markeredgecolor='white', markeredgewidth=0.8, zorder=4)
    
    # --- Axis formatting ---
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['1', '3', '6'], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.set_title(f"{headcoil}-ch (n={n_subjects})", fontsize=12, fontweight='normal')
    ax.set_ylim(y_limits)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_linewidth(1)
    ax.grid(False)
    
    if show_legend:
        ax.legend(title='ME', fontsize=9, title_fontsize=9, loc='upper right',
                  framealpha=0.9, edgecolor='none')


def generate_ppi_figure(data, summary, img_type, seed_img_path, target_img_path,
                         seed_slice_label, target_slice_label, output_dir, mask_name):
    """
    Generate Nature-quality figure for one image type.
    Layout: Seed (VS) + Target (cluster) + 20-ch plot + 64-ch plot.
    """
    # Compute y-limits
    all_means = summary['mean_val'].values
    all_ses = summary['se_val'].values
    y_max = max(all_means + all_ses)
    y_min = min(all_means - all_ses)
    margin = (y_max - y_min) * 0.15
    y_limits = (y_min - margin, y_max + margin)
    
    # Load brain images and compute aspect ratios
    seed_img = mpimg.imread(seed_img_path)
    target_img = mpimg.imread(target_img_path)
    seed_aspect = seed_img.shape[1] / seed_img.shape[0]
    target_aspect = target_img.shape[1] / target_img.shape[0]
    
    # Create figure: Seed + Target + 20-ch + 64-ch
    scatter_ratio = 1.2
    brain_ratio = 0.8
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5),
                              gridspec_kw={'width_ratios': [seed_aspect * brain_ratio, 
                                                            target_aspect * brain_ratio, 
                                                            scatter_ratio, scatter_ratio],
                                           'wspace': 0.25})
    
    ax_seed, ax_target, ax_20ch, ax_64ch = axes
    
    # Panel labels
    panel_labels = ['a', 'b', 'c', 'd']
    ax_seed.text(0.0, 1.08, panel_labels[0], transform=ax_seed.transAxes,
                 fontsize=16, fontweight='bold', va='bottom', ha='left')
    ax_target.text(0.0, 1.08, panel_labels[1], transform=ax_target.transAxes,
                   fontsize=16, fontweight='bold', va='bottom', ha='left')
    ax_20ch.text(-0.15, 1.08, panel_labels[2], transform=ax_20ch.transAxes,
                 fontsize=16, fontweight='bold', va='bottom', ha='left')
    ax_64ch.text(-0.15, 1.08, panel_labels[3], transform=ax_64ch.transAxes,
                 fontsize=16, fontweight='bold', va='bottom', ha='left')
    
    # Seed image panel
    ax_seed.imshow(seed_img)
    ax_seed.axis('off')
    ax_seed.text(0.5, -0.02, 'Seed', transform=ax_seed.transAxes,
                 fontsize=11, ha='center', va='top', fontweight='bold')
    ax_seed.text(0.5, -0.12, seed_slice_label, transform=ax_seed.transAxes,
                 fontsize=10, ha='center', va='top')
    
    # Target image panel
    ax_target.imshow(target_img)
    ax_target.axis('off')
    ax_target.text(0.5, -0.02, 'Target', transform=ax_target.transAxes,
                   fontsize=11, ha='center', va='top', fontweight='bold')
    ax_target.text(0.5, -0.12, target_slice_label, transform=ax_target.transAxes,
                   fontsize=10, ha='center', va='top')
    
    # EMM-style plots
    plot_ppi_emm_style(ax_20ch, data, summary, '20', y_limits, 
                        ME_COLORS_NATURE, img_type, show_legend=False)
    plot_ppi_emm_style(ax_64ch, data, summary, '64', y_limits, 
                        ME_COLORS_NATURE, img_type, show_legend=True)
    
    # Y-axis label on left plot only
    ax_20ch.set_ylabel(YLABEL_MAP.get(img_type, img_type), fontsize=13)
    
    plt.tight_layout()
    fig.subplots_adjust(top=0.90, bottom=0.15)
    fig.text(0.75, 0.02, 'Multiband Factor', ha='center', fontsize=13)
    
    # Save
    plot_file = output_dir / f"ppi_{mask_name}_{img_type}_emm_plot.png"
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"Saved: {plot_file.name}")
    plt.show()
    
    return plot_file


# =============================================================================
# MAIN ANALYSIS
# =============================================================================

# Check for brain images
seed_img_path = brain_img_dir / SEED_IMAGE['file']
target_img_path = brain_img_dir / TARGET_IMAGE['file']

images_available = True
if not seed_img_path.exists():
    print(f"WARNING: Seed image not found: {seed_img_path}")
    images_available = False
else:
    print(f"Seed image found: {SEED_IMAGE['file']}")

if not target_img_path.exists():
    print(f"WARNING: Target image not found: {target_img_path}")
    images_available = False
else:
    print(f"Target image found: {TARGET_IMAGE['file']}")

if not images_available:
    print("Figures will be generated once brain images are added.")

# Process each image type and collect results
results = {}

for img_type in IMG_TYPES:
    print(f"\n{'-'*60}")
    print(f"Processing: {img_type}")
    print(f"{'-'*60}")
    
    if img_type == 'weighted_beta':
        # Load both beta and varcope extractions
        beta_data = load_ppi_extractions(extractions_dir, MASK_NAME, 'beta', 
                                          valid_subjects, headcoil_mapping)
        varcope_data = load_ppi_extractions(extractions_dir, MASK_NAME, 'varcope', 
                                             valid_subjects, headcoil_mapping)
        
        if beta_data is None:
            print(f"No beta extractions found — skipping weighted_beta")
            continue
        if varcope_data is None:
            print(f"No varcope extractions found — skipping weighted_beta")
            print(f"  Expected pattern: ts_sub-*_acq_*_type-ppi_seed-VS_thr5_img-varcope_mask-{MASK_NAME}_denoise_base.txt")
            continue
        
        # Merge beta and varcope
        varcope_data = varcope_data.rename(columns={'value': 'varcope'})
        data = beta_data.merge(
            varcope_data[['subject', 'mb', 'me', 'headcoil', 'varcope']],
            on=['subject', 'mb', 'me', 'headcoil'],
            how='inner'
        )
        
        # Compute weights
        data['weight'] = compute_weights(data['varcope'])
        
        print(f"Loaded {len(data)} observations from {data['subject'].nunique()} subjects")
        print(f"  20-ch: {(data['headcoil'] == '20').sum()} obs, "
              f"64-ch: {(data['headcoil'] == '64').sum()} obs")
        
        # Run weighted LME
        model, anova = run_weighted_ppi_lme(data, MASK_NAME, output_dir)
        
    else:
        # Standard unweighted analysis (beta, zstat)
        data = load_ppi_extractions(extractions_dir, MASK_NAME, img_type, 
                                     valid_subjects, headcoil_mapping)
        
        if data is None:
            print(f"No extractions found for {img_type} — skipping")
            print(f"  Expected pattern: ts_sub-*_acq_*_type-ppi_seed-VS_thr5_img-{img_type}_mask-{MASK_NAME}_denoise_base.txt")
            continue
        
        print(f"Loaded {len(data)} observations from {data['subject'].nunique()} subjects")
        print(f"  20-ch: {(data['headcoil'] == '20').sum()} obs, "
              f"64-ch: {(data['headcoil'] == '64').sum()} obs")
        
        # Run LME
        model, anova = run_ppi_lme(data, img_type, MASK_NAME, output_dir)
    
    # Compute cell stats (for plotting — uses raw values, not weighted)
    summary = compute_ppi_cell_stats(data, img_type)
    
    # Save raw data
    data_file = output_dir / f"ppi_{MASK_NAME}_{img_type}_data.csv"
    data.to_csv(data_file, index=False)
    print(f"Saved: {data_file.name}")
    
    # Generate figure (if images available)
    if images_available:
        plot_file = generate_ppi_figure(
            data, summary, img_type, 
            seed_img_path, target_img_path,
            SEED_IMAGE['slice_label'], TARGET_IMAGE['slice_label'],
            output_dir, MASK_NAME
        )
    else:
        print(f"Skipping figure for {img_type} — add brain images first")
        plot_file = None
    
    results[img_type] = {'data': data, 'summary': summary, 'model': model, 'anova': anova, 'plot': plot_file}

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{'='*80}")
print("ANALYSIS COMPLETE")
print(f"{'='*80}")
print(f"\nMask: {MASK_NAME}")
print(f"Processed image types: {list(results.keys())}")
print(f"\nOutput files in {output_dir}:")
for img_type in results.keys():
    print(f"  - ppi_{MASK_NAME}_{img_type}_anova.csv")
    print(f"  - ppi_{MASK_NAME}_{img_type}_data.csv")
    if images_available:
        print(f"  - ppi_{MASK_NAME}_{img_type}_emm_plot.png")

The MB×ME interaction on PPI connectivity (zstat17) tells us that the effect of multi-echo acquisition on VS connectivity depends on the multiband factor (or vice versa).
Looking at the figure:
20-ch coil (clearer pattern):

At MB1: ME4 >> ME1 (multi-echo gives stronger connectivity)
At MB6: ME1 ≈ ME4 or ME1 > ME4 (the multi-echo advantage disappears or reverses)

Interpretation:
Multi-echo acquisition (ME4) improves PPI connectivity estimates, but only when TR is long enough (low multiband). At high multiband (MB6), TRs are short, and the benefit of multi-echo diminishes — possibly because:

- Echo time constraints: With short TRs (MB6), there's less time to acquire multiple echoes optimally, so ME4 can't fully leverage its denoising advantage
- Signal dynamics: PPI relies on detecting covariance between seed timeseries and task × connectivity regressors. Multi-echo helps by reducing thermal noise, but at very short TRs the physiological signal itself may be undersampled
- BOLD sensitivity tradeoff: ME4 combines echoes to optimize T2* weighting, but this optimization may interact with the temporal dynamics at different TRs

Bottom line for the manuscript:
If you want optimal PPI connectivity from VS, use MB1 or MB3 with multi-echo. MB6 eliminates the multi-echo advantage for connectivity analyses — the speed gain comes at a cost.

# Design Matrix Preparation (Cells 11–14)
FD loading and echo consolidation, smoothness table generation, per-acquisition design matrices with proper de-meaning, and pairwise contrast matrices for FSL randomise.

In [ ]:
# =============================================================================
# FD MEAN DATA LOADING & MULTI-ECHO CONSOLIDATION
# =============================================================================
# Merges original Cells 23 + 24 (202 lines -> ~90 lines by removing duplicate loads)
# Produces: filtered_fd_df, consolidated_df (used by downstream design matrix cells)
# Dependencies: project_root (from Setup)

print("=" * 80)
print("LOADING FD_MEAN DATA AND CONSOLIDATING MULTI-ECHO ECHOES")
print("=" * 80)

BASE_CODE_DIR = project_root / 'code'
SUBLIST_PATH = BASE_CODE_DIR / 'sublist-complete.txt'
FD_TSV_PATH = project_root / 'derivatives' / 'Task-sharedreward_Level-Acq_Outlier-info_mriqc-0.16.1.tsv'

print(f"FD TSV data path: {FD_TSV_PATH}")
print(f"Subject list path: {SUBLIST_PATH}")

# --- Step 1: Load Master Subject List ---
with open(SUBLIST_PATH, 'r') as f:
    all_subjects_raw = sorted([line.strip() for line in f if line.strip()])
all_subjects_formatted = [f"sub-{s}" for s in all_subjects_raw]
print(f"Loaded {len(all_subjects_raw)} subjects")

# --- Step 2: Load and Truncate FD TSV ---
fd_new_df_raw = pd.read_csv(FD_TSV_PATH, sep=r'\s+')
print(f"Loaded {len(fd_new_df_raw)} records from FD TSV")

filtered_fd_df = fd_new_df_raw[fd_new_df_raw['Sub'].isin(all_subjects_formatted)].copy()
print(f"Truncated to {len(filtered_fd_df)} records for our subjects")

# Clean outlier column
filtered_fd_df['outlier_acq_Custom1'] = pd.to_numeric(
    filtered_fd_df['outlier_acq_Custom1'], errors='coerce'
).fillna(False).astype(bool)

# --- Step 3: Outlier Summary ---
outlier_runs = filtered_fd_df[filtered_fd_df['outlier_acq_Custom1'] == True]
print(f"\nOutlier runs: {len(outlier_runs)}")
if len(outlier_runs) > 0:
    print(outlier_runs)

# --- Step 4: Consolidate Multi-Echo Data ---
print("\n--- Consolidating Multi-Echo Data ---")

def get_consolidated_acq(acq_str):
    match = re.match(r'(mb\dme\d+)(_echo-\d+_part-mag)?', acq_str)
    return match.group(1) if match else acq_str

filtered_fd_df['consolidated_acq'] = filtered_fd_df['acq'].apply(get_consolidated_acq)

consolidated_df = filtered_fd_df.groupby(
    ['Sub', 'task', 'consolidated_acq'], as_index=False
).agg({
    'tsnr': 'mean',
    'fd_mean': 'mean',
    'outlier_acq_Custom1': 'any'
})
consolidated_df = consolidated_df.rename(columns={'consolidated_acq': 'acq'})
consolidated_df = consolidated_df[['Sub', 'task', 'acq', 'tsnr', 'fd_mean', 'outlier_acq_Custom1']]

print(f"Original rows: {len(filtered_fd_df)}")
print(f"Consolidated rows: {len(consolidated_df)}")

print("\n" + "=" * 80)
print("FD DATA LOADING AND CONSOLIDATION COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# SMOOTHNESS TABLE FOR MULTI-ECHO ACQUISITIONS
# =============================================================================
# Original Cell 25 (258 lines -> ~180 lines, stripped duplicate imports/config)
# Produces: smoothness_df (used by design matrix cells)
# Dependencies: project_root (from Setup)

print("=" * 80)
print("GENERATING SMOOTHNESS TABLE FOR MULTI-ECHO ACQUISITIONS")
print("=" * 80)

# --- Config ---
csv_path_smoothness_table = project_root / 'code' / 'smoothness-all.csv'
SUBLIST_PATH_ON = project_root / 'code' / 'sublist-openneuro.txt'

HEADCOIL_64_SUBJECTS = [
    "10015", "10017", "10024", "10028", "10035", "10041", "10043", "10046",
    "10054", "10059", "10069", "10074", "10078", "10080", "10085", "10094",
    "10108", "10125", "10130", "10136", "10137", "10142", "10150", "10154",
    "10186", "10188", "10221"
]

MULTI_ECHO_ACQS = ['mb1me4', 'mb3me4', 'mb6me4']
SMOOTHNESS_OUTPUT_CSV = project_root / 'code' / 'smoothness_multi_echo_table.csv'

print(f"Input smoothness file: {csv_path_smoothness_table}")
print(f"Subject list path: {SUBLIST_PATH_ON}")

# --- Step 1: Load Subject List (excluding SP) ---
with open(SUBLIST_PATH_ON, 'r') as f:
    all_subjects_raw_on = [line.strip() for line in f if line.strip()]

all_subjects_on = sorted([s for s in all_subjects_raw_on if not s.endswith('sp')])
print(f"Loaded {len(all_subjects_raw_on)} subjects, filtered to {len(all_subjects_on)} (excluding 'sp')")

# --- Step 2: Load and Process Smoothness Data ---
print("\n--- Processing Smoothness Data ---")

def get_consolidated_acq_name(acq_str):
    match = re.match(r'(mb\dme\d+)(?:_echo-\d+_part-mag)?', acq_str)
    return match.group(1) if match else acq_str

def load_and_process_smoothness_for_table(csv_path, subjects_list):
    """Load smoothness CSV with shift-up correction, filter to subjects, deduplicate."""
    raw_data = pd.read_csv(csv_path)
    data = raw_data.rename(columns={raw_data.columns[0]: 'path', 'Unnamed: 3': 'smoothness'})
    data['file_path'] = data['path'].shift(1)
    data['smoothness'] = pd.to_numeric(data['smoothness'], errors='coerce')
    data = data[data['smoothness'].notna() & data['file_path'].notna()].copy()
    data['file_path'] = data['file_path'].astype(str)

    path_pattern = r'sub-(\d+).*acq-([a-zA-Z0-9_.-]+)'
    records = []
    for _, row in data.iterrows():
        if row['file_path'] == 'nan' or not row['file_path'].strip():
            continue
        match = re.search(path_pattern, row['file_path'])
        if match:
            subject = match.group(1)
            acq = get_consolidated_acq_name(match.group(2))
            if subject in subjects_list:
                records.append({'subject': subject, 'acq': acq, 'smoothness': row['smoothness']})

    if not records:
        print("  No valid smoothness records found.")
        return pd.DataFrame(columns=['subject', 'headcoil', 'acq', 'smoothness'])

    df = pd.DataFrame(records)
    df['subject'] = df['subject'].astype(str)
    df['acq'] = df['acq'].astype(str)
    df['headcoil'] = df['subject'].apply(lambda x: '64' if x in HEADCOIL_64_SUBJECTS else '20')
    df = df[['subject', 'headcoil', 'acq', 'smoothness']].copy()

    # Deduplicate by averaging
    initial = len(df)
    df = df.groupby(['subject', 'headcoil', 'acq'], as_index=False)['smoothness'].mean()
    print(f"  Extracted {initial} records, deduplicated to {len(df)}")
    return df

smoothness_df = load_and_process_smoothness_for_table(csv_path_smoothness_table, all_subjects_on)

# --- Step 3: Create Display Table ---
print("\n--- Creating Final Smoothness Table ---")

if not smoothness_df.empty:
    smoothness_filtered_display = smoothness_df[smoothness_df['acq'].isin(MULTI_ECHO_ACQS)].copy()

    base_combos = pd.MultiIndex.from_product(
        [all_subjects_on, MULTI_ECHO_ACQS], names=['subject', 'acq']
    ).to_frame(index=False)

    final_display = pd.merge(
        base_combos,
        smoothness_filtered_display[['subject', 'acq', 'smoothness']],
        on=['subject', 'acq'], how='left'
    )

    smoothness_pivot_display = final_display.pivot_table(
        values='smoothness', index='subject', columns='acq', aggfunc='first'
    ).reindex(columns=MULTI_ECHO_ACQS)

    hc_info = pd.DataFrame({
        'subject': all_subjects_on,
        'headcoil': ['64' if s in HEADCOIL_64_SUBJECTS else '20' for s in all_subjects_on]
    }).set_index('subject')

    smoothness_pivot_display = smoothness_pivot_display.merge(hc_info, left_index=True, right_index=True, how='left')
    smoothness_pivot_display = smoothness_pivot_display[['headcoil'] + MULTI_ECHO_ACQS].sort_index()
    smoothness_pivot_display[MULTI_ECHO_ACQS] = smoothness_pivot_display[MULTI_ECHO_ACQS].round(3)

    pd.set_option('display.max_rows', None)
    print(f"\nFinal Smoothness Table:")
    print(smoothness_pivot_display.to_string())
    pd.reset_option('display.max_rows')

    smoothness_pivot_display.to_csv(SMOOTHNESS_OUTPUT_CSV)
    print(f"\nSaved to '{SMOOTHNESS_OUTPUT_CSV}'")

    # Missing data summary
    print(f"\nMissing smoothness data:")
    for acq in MULTI_ECHO_ACQS:
        missing = smoothness_pivot_display[acq].isna().sum()
        if missing > 0:
            subs = smoothness_pivot_display[smoothness_pivot_display[acq].isna()].index.tolist()
            print(f"  {acq}: {missing} subjects: {subs}")
        else:
            print(f"  {acq}: None missing")
else:
    print("Error: Smoothness data could not be loaded.")

print("\n" + "=" * 80)
print("SMOOTHNESS TABLE GENERATION COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# GROUP-LEVEL DESIGN MATRICES FOR RANDOMISE
# =============================================================================
# Original Cell 27 (411 lines -> ~340 lines, stripped duplicate imports/config)
# This is the CORRECTED version with proper de-meaning after all filtering.
# Dependencies: project_root, smoothness_df (from Cell 12), consolidated_df (from Cell 11)
# Produces: design_matrices dict (used by TEDANA cells)

print("=" * 80)
print("GENERATING GROUP-LEVEL DESIGN MATRICES FOR RANDOMISE")
print("  (Using Consolidated FD_MEAN and Pre-Processed Smoothness Table)")
print("  *** Ensured proper de-meaning after all filtering ***")
print("=" * 80)

# --- Configuration ---
BASE_CODE_DIR = project_root / 'code'
SUBLIST_PATH_DM = BASE_CODE_DIR / 'sublist-openneuro.txt'
FD_TSV_PATH_DM = project_root / 'derivatives' / 'Task-sharedreward_Level-Acq_Outlier-info_mriqc-0.16.1.tsv'
RECALCULATED_FD_PATH = BASE_CODE_DIR / 'missing_fd_mean_recalculated.csv'
SMOOTHNESS_TABLE_PATH = BASE_CODE_DIR / 'smoothness_multi_echo_table.csv'

HEADCOIL_64_SUBJECTS_DM = [
    "10015", "10017", "10024", "10028", "10035", "10041", "10043", "10046",
    "10054", "10059", "10069", "10074", "10078", "10080", "10085", "10094",
    "10108", "10125", "10130", "10136", "10137", "10142", "10150", "10154",
    "10186", "10188", "10221"
]

ACQ_TYPES_DM = ["mb1me4", "mb3me4", "mb6me4"]
DESIGN_MATRIX_OUTPUT_DIR = BASE_CODE_DIR / 'design_matrices'
DESIGN_MATRIX_OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Subject list: {SUBLIST_PATH_DM}")
print(f"FD TSV: {FD_TSV_PATH_DM}")
print(f"Recalculated FD: {RECALCULATED_FD_PATH}")
print(f"Smoothness table: {SMOOTHNESS_TABLE_PATH}")
print(f"Output: {DESIGN_MATRIX_OUTPUT_DIR}")

# --- Step 1: Load Subject List (excluding SP) ---
with open(SUBLIST_PATH_DM, 'r') as f:
    all_subjects_dm = sorted([s.strip() for s in f if s.strip() and not s.strip().endswith('sp')])
print(f"Loaded {len(all_subjects_dm)} subjects (excluding SP)")

# --- Step 2: Load, Truncate, and Consolidate FD Mean Data ---
print("\n--- Processing FD Mean Data ---")
fd_raw = pd.read_csv(FD_TSV_PATH_DM, sep=r'\s+')
fd_filtered = fd_raw[fd_raw['Sub'].isin([f"sub-{s}" for s in all_subjects_dm])].copy()
fd_filtered['subject'] = fd_filtered['Sub'].str.replace('sub-', '').astype(str)
fd_filtered['outlier_acq_Custom1'] = pd.to_numeric(
    fd_filtered['outlier_acq_Custom1'], errors='coerce'
).fillna(False).astype(bool)

def get_consolidated_acq_dm(acq_str):
    match = re.match(r'(mb\dme\d+)(_echo-\d+_part-mag)?', acq_str)
    return match.group(1) if match else acq_str

fd_filtered['consolidated_acq'] = fd_filtered['acq'].apply(get_consolidated_acq_dm)

temp_consolidated = fd_filtered.groupby(['subject', 'consolidated_acq'], as_index=False).agg({
    'tsnr': 'mean', 'fd_mean': 'mean', 'outlier_acq_Custom1': 'any'
}).rename(columns={'consolidated_acq': 'acq'})

base_combos = pd.MultiIndex.from_product(
    [all_subjects_dm, ACQ_TYPES_DM], names=['subject', 'acq']
).to_frame(index=False)
consolidated_fd_dm = pd.merge(base_combos, temp_consolidated, on=['subject', 'acq'], how='left')

print(f"Consolidated FD: {len(temp_consolidated)} records, padded to {len(consolidated_fd_dm)}")

# --- Step 2a: Incorporate Recalculated Missing FD Mean ---
print("\n--- Incorporating Recalculated FD Mean Data ---")
try:
    recalc_fd = pd.read_csv(RECALCULATED_FD_PATH)
    recalc_fd = recalc_fd.rename(columns={'acquisition': 'acq'})
    recalc_fd['subject'] = recalc_fd['subject'].astype(str)
    recalc_fd['fd_mean_recalc'] = pd.to_numeric(recalc_fd['fd_mean'], errors='coerce')

    nan_before = consolidated_fd_dm['fd_mean'].isna().sum()
    consolidated_fd_dm = pd.merge(
        consolidated_fd_dm,
        recalc_fd[['subject', 'acq', 'fd_mean_recalc']],
        on=['subject', 'acq'], how='left'
    )
    consolidated_fd_dm['fd_mean'] = consolidated_fd_dm['fd_mean'].fillna(consolidated_fd_dm['fd_mean_recalc'])
    consolidated_fd_dm = consolidated_fd_dm.drop(columns=['fd_mean_recalc'])
    nan_after = consolidated_fd_dm['fd_mean'].isna().sum()
    print(f"  NaNs before: {nan_before}, after: {nan_after}, filled: {nan_before - nan_after}")
except FileNotFoundError:
    print(f"  Recalculated FD file not found. Skipping.")

# --- Step 3: Load Pre-Processed Smoothness Table ---
print("\n--- Loading Pre-Processed Smoothness Table ---")
try:
    smoothness_table_dm = pd.read_csv(SMOOTHNESS_TABLE_PATH, index_col=0)
    smoothness_table_dm.index.name = 'subject'
    smoothness_table_dm = smoothness_table_dm.reset_index()
    smoothness_table_dm['subject'] = smoothness_table_dm['subject'].astype(str)

    smoothness_long_dm = smoothness_table_dm.melt(
        id_vars=['subject', 'headcoil'],
        value_vars=[c for c in smoothness_table_dm.columns if c.startswith('mb')],
        var_name='acq', value_name='smoothness'
    )
    smoothness_long_dm['smoothness'] = pd.to_numeric(smoothness_long_dm['smoothness'], errors='coerce')
    print(f"  Loaded {len(smoothness_long_dm)} smoothness records")
except FileNotFoundError:
    print(f"  Smoothness table not found at {SMOOTHNESS_TABLE_PATH}")
    smoothness_long_dm = pd.DataFrame(columns=['subject', 'acq', 'smoothness'])

# --- Step 4: Generate Headcoil Encoded Data ---
print("\n--- Generating Headcoil Data ---")
hc_map_dm = {s: 1 if s in HEADCOIL_64_SUBJECTS_DM else 0 for s in all_subjects_dm}
hc_series_dm = pd.DataFrame({'subject': list(hc_map_dm.keys()), 'headcoil_encoded': list(hc_map_dm.values())})
hc_series_dm['subject'] = hc_series_dm['subject'].astype(str)
print(f"Headcoil encoding for {len(hc_series_dm)} subjects")

# --- Step 5: Create Design Matrices ---
design_matrices = {}

for acq_type in ACQ_TYPES_DM:
    print(f"\n--- Constructing Design Matrix for {acq_type} ---")

    df = pd.DataFrame({'subject': all_subjects_dm})
    df['ones'] = 1

    # Merge FD
    fd_for_acq = consolidated_fd_dm[consolidated_fd_dm['acq'] == acq_type][['subject', 'fd_mean']].copy()
    df = pd.merge(df, fd_for_acq, on='subject', how='left')

    # Merge headcoil
    df = pd.merge(df, hc_series_dm, on='subject', how='left')

    df['fd_mean'] = pd.to_numeric(df['fd_mean'], errors='coerce')
    df['headcoil_encoded'] = pd.to_numeric(df['headcoil_encoded'], errors='coerce')

    # Merge smoothness
    if not smoothness_long_dm.empty:
        smooth_for_acq = smoothness_long_dm[smoothness_long_dm['acq'] == acq_type][['subject', 'smoothness']].copy()
        df = pd.merge(df, smooth_for_acq, on='subject', how='left')
        df['smoothness'] = pd.to_numeric(df['smoothness'], errors='coerce')
    else:
        df['smoothness'] = np.nan

    # Drop NaNs in FD
    missing_fd = df[df['fd_mean'].isna()]['subject'].tolist()
    if missing_fd:
        print(f"  Dropping {len(missing_fd)} subjects with missing FD: {missing_fd}")
        df = df.dropna(subset=['fd_mean']).copy()

    # Drop NaNs in smoothness
    missing_smooth = df[df['smoothness'].isna()]['subject'].tolist()
    if missing_smooth:
        print(f"  Dropping {len(missing_smooth)} subjects with missing smoothness: {missing_smooth}")
        df = df.dropna(subset=['smoothness']).copy()

    # De-mean on FINAL filtered set
    df['fd_mean_demeaned'] = df['fd_mean'] - df['fd_mean'].mean()
    df['headcoil_demeaned'] = df['headcoil_encoded'] - df['headcoil_encoded'].mean()
    df['smoothness_demeaned'] = df['smoothness'] - df['smoothness'].mean()

    interaction_raw = df['fd_mean_demeaned'] * df['headcoil_demeaned']
    df['fd_mean_x_headcoil_demeaned'] = interaction_raw - interaction_raw.mean()

    design_matrix_df = df[['subject', 'ones', 'fd_mean_demeaned', 'headcoil_demeaned',
                            'fd_mean_x_headcoil_demeaned', 'smoothness_demeaned']].copy()

    design_matrices[acq_type] = design_matrix_df
    print(f"  Created: {len(design_matrix_df)} subjects")
    print(f"  NaN check — FD: {design_matrix_df['fd_mean_demeaned'].isna().sum()}, "
          f"Smoothness: {design_matrix_df['smoothness_demeaned'].isna().sum()}")

# --- Display Sample ---
print("\n" + "=" * 80)
print("SAMPLE DESIGN MATRICES")
print("=" * 80)
for acq_type, df in design_matrices.items():
    print(f"\n--- {acq_type} (first 5 rows) ---")
    print(df.head().to_string())

# --- Save ---
print("\n" + "=" * 80)
print(f"SAVING TO: {DESIGN_MATRIX_OUTPUT_DIR}")
print("=" * 80)
for acq_type, df in design_matrices.items():
    out_file = DESIGN_MATRIX_OUTPUT_DIR / f'design_matrix_{acq_type}.csv'
    df.to_csv(out_file, index=False)
    print(f"Saved '{out_file.name}' ({len(df)} rows)")

print("\nAll design matrices generated and saved.")
print("=" * 80)

In [ ]:
# =============================================================================
# PAIRWISE CONTRAST DESIGN MATRICES (FROM SCRATCH)
# =============================================================================
# Original Cell 35 (347 lines -> ~280 lines, stripped duplicate imports)
# Produces: Per-acq design matrices AND pairwise contrast matrices for randomise
# Dependencies: project_root (from Setup)
# NOTE: This cell is self-contained — loads its own subject list, FD, smoothness,
#       and headcoil data independently of the LME pipeline cells.

print("=" * 80)
print("GENERATING ALL DESIGN MATRICES FROM SCRATCH")
print("=" * 80)

CODE_DIR = project_root / 'code'
OUTPUT_DIR_PW = Path.home() / 'Desktop' / 'projects' / 'multiecho-pilot' / 'randomise' / 'models'

SUBJECT_LIST = CODE_DIR / 'sublist-randomise.txt'
FD_EXCEL = CODE_DIR / 'outputs' / 'tables' / 'fd_mean_averages.xlsx'
FD_MISSING = CODE_DIR / 'outputs' / 'tables' / 'missing_fd_mean_recalculated.csv'
SMOOTHNESS_CSV_PW = CODE_DIR / 'smoothness-all.csv'
HEADCOIL_20CH = CODE_DIR / 'sublist-20ch.txt'
HEADCOIL_64CH = CODE_DIR / 'sublist-64ch.txt'

print(f"Code directory: {CODE_DIR}")
print(f"Output directory: {OUTPUT_DIR_PW}")

# --- Step 1: Load Subject List ---
print("\n--- Loading Subject List ---")
with open(SUBJECT_LIST, 'r') as f:
    subjects_pw = [line.strip() for line in f if line.strip()]
print(f"Loaded {len(subjects_pw)} subjects")
subjects_formatted_pw = [f'sub-{s}' for s in subjects_pw]

# --- Step 2: Load and Combine FD Mean Data ---
print("\n--- Loading FD Mean Data ---")
fd_excel = pd.read_excel(FD_EXCEL)
fd_excel.columns = fd_excel.columns.str.strip()
print(f"  Excel: {len(fd_excel)} rows")

fd_missing = pd.read_csv(FD_MISSING)
fd_missing.columns = fd_missing.columns.str.strip()
fd_missing = fd_missing.rename(columns={'subject': 'Sub', 'acquisition': 'acq_base'})
fd_missing['Sub'] = 'sub-' + fd_missing['Sub'].astype(str)
fd_missing = fd_missing[fd_missing['fd_mean'].notna()]
print(f"  CSV (missing): {len(fd_missing)} rows")

fd_combined = pd.concat([fd_excel, fd_missing], ignore_index=True)
fd_combined = fd_combined.sort_values('Sub').drop_duplicates(subset=['Sub', 'acq_base'], keep='first')

fd_filtered_pw = fd_combined[fd_combined['Sub'].isin(subjects_formatted_pw)]
fd_pivot_pw = fd_filtered_pw.pivot(index='Sub', columns='acq_base', values='fd_mean')
fd_pivot_pw = fd_pivot_pw.reindex(subjects_formatted_pw)
print(f"  Combined: {len(fd_pivot_pw)} subjects x {len(fd_pivot_pw.columns)} acquisitions")

# --- Step 3: Load Smoothness Data ---
print("\n--- Loading Smoothness Data ---")
smoothness_raw_pw = pd.read_csv(SMOOTHNESS_CSV_PW)
smoothness_raw_pw = smoothness_raw_pw.rename(columns={
    smoothness_raw_pw.columns[0]: 'path', 'Unnamed: 3': 'smoothness'
})
smoothness_raw_pw['file_path'] = smoothness_raw_pw['path'].shift(1)
smoothness_data_pw = smoothness_raw_pw[
    smoothness_raw_pw['smoothness'].notna() & smoothness_raw_pw['file_path'].notna()
].reset_index(drop=True).copy()
print(f"  Loaded {len(smoothness_data_pw)} rows after shift-up correction")

def parse_smoothness_path_pw(path):
    try:
        if not isinstance(path, str):
            return pd.Series({'subject': None, 'acq': None})
        sub_match = re.search(r'sub-(\d+)', path)
        acq_match = re.search(r'acq-(mb\dme\d)', path)
        return pd.Series({
            'subject': sub_match.group(1) if sub_match else None,
            'acq': acq_match.group(1) if acq_match else None
        })
    except Exception:
        return pd.Series({'subject': None, 'acq': None})

parsed_pw = smoothness_data_pw['file_path'].apply(parse_smoothness_path_pw)
smoothness_data_pw[['subject', 'acq']] = parsed_pw
smoothness_data_pw = smoothness_data_pw.dropna(subset=['subject', 'acq']).reset_index(drop=True)
smoothness_data_pw = smoothness_data_pw[smoothness_data_pw['subject'].isin(subjects_pw)].reset_index(drop=True)
smoothness_data_pw = smoothness_data_pw[
    smoothness_data_pw['acq'].isin(['mb1me4', 'mb3me4', 'mb6me4'])
].reset_index(drop=True)

smoothness_data_pw['subject_formatted'] = 'sub-' + smoothness_data_pw['subject']
smoothness_data_pw = smoothness_data_pw.drop_duplicates(subset=['subject_formatted', 'acq'], keep='first')

smoothness_pivot_pw = smoothness_data_pw.pivot(
    index='subject_formatted', columns='acq', values='smoothness'
).reindex(subjects_formatted_pw)
print(f"  Pivoted: {len(smoothness_pivot_pw)} subjects x {len(smoothness_pivot_pw.columns)} acquisitions")

missing_count_pw = smoothness_pivot_pw.isna().sum().sum()
if missing_count_pw > 0:
    print(f"  WARNING: {missing_count_pw} missing smoothness values")

# --- Step 4: Load Headcoil Assignments ---
print("\n--- Loading Headcoil Assignments ---")
with open(HEADCOIL_20CH, 'r') as f:
    subjects_20ch = set(line.strip() for line in f if line.strip())
with open(HEADCOIL_64CH, 'r') as f:
    subjects_64ch = set(line.strip() for line in f if line.strip())
print(f"  20-channel: {len(subjects_20ch)}, 64-channel: {len(subjects_64ch)}")

headcoil_map_pw = {}
for sub in subjects_pw:
    if sub in subjects_64ch:
        headcoil_map_pw[f'sub-{sub}'] = 1
    elif sub in subjects_20ch:
        headcoil_map_pw[f'sub-{sub}'] = 0
    else:
        print(f"  WARNING: {sub} not in either headcoil list!")
        headcoil_map_pw[f'sub-{sub}'] = np.nan

headcoil_series_pw = pd.Series(headcoil_map_pw, name='hc').reindex(subjects_formatted_pw)

# --- Helper: Save FSL .mat ---
def save_fsl_mat(df, filepath):
    df_mat = df.drop(columns=['subject']) if 'subject' in df.columns else df
    with open(filepath, 'w') as f:
        f.write(f'/NumWaves\t{len(df_mat.columns)}\n')
        f.write(f'/NumPoints\t{len(df_mat)}\n')
        f.write('/Matrix\n')
        for _, row in df_mat.iterrows():
            f.write(' '.join(f'{val:.10f}' for val in row.values) + '\n')

# --- Step 5: Create Original Design Matrices ---
print("\n" + "=" * 80)
print("CREATING ORIGINAL DESIGN MATRICES")
print("=" * 80)

def create_design_matrix_pw(acq_name, fd_values, hc_values, smoothness_values):
    df = pd.DataFrame({'subject': subjects_formatted_pw})
    df['intercept'] = 1.0
    fd_dm = fd_values - fd_values.mean()
    df['fd_mean'] = fd_dm.values
    hc_dm = hc_values - hc_values.mean()
    df['hc'] = hc_dm.values
    interaction = df['fd_mean'] * df['hc']
    df['fd_mean_x_hc'] = (interaction - interaction.mean()).values
    smooth_dm = smoothness_values - smoothness_values.mean()
    df['smoothness'] = smooth_dm.values
    return df

acquisition_matrices_pw = {}
for acq in ['mb1me4', 'mb3me4', 'mb6me4']:
    print(f"\n{acq.upper()}:")
    fd_data = fd_pivot_pw[acq]
    smooth_data = smoothness_pivot_pw[acq]
    print(f"  FD: {fd_data.notna().sum()}/{len(fd_data)}, Smoothness: {smooth_data.notna().sum()}/{len(smooth_data)}")

    df = create_design_matrix_pw(acq, fd_data, headcoil_series_pw, smooth_data)
    acquisition_matrices_pw[acq] = df

    print(f"  Mean checks: fd={df['fd_mean'].mean():.10f}, hc={df['hc'].mean():.10f}, "
          f"interaction={df['fd_mean_x_hc'].mean():.10f}, smooth={df['smoothness'].mean():.10f}")

    acq_dir = OUTPUT_DIR_PW / acq
    acq_dir.mkdir(exist_ok=True)

    tsv_file = acq_dir / f'{acq}_design_matrix.tsv'
    df.drop(columns=['subject']).to_csv(tsv_file, sep='\t', index=False)
    print(f"  Saved TSV: {tsv_file.name}")

    mat_file = acq_dir / f'{acq}_design_matrix.mat'
    save_fsl_mat(df, mat_file)
    print(f"  Saved MAT: {mat_file.name}")

# --- Step 6: Create Contrast Design Matrices ---
print("\n" + "=" * 80)
print("CREATING CONTRAST DESIGN MATRICES")
print("=" * 80)

def create_contrast_matrix_pw(df_high, df_low, contrast_name):
    print(f"\n{contrast_name}:")
    n = len(df_high)
    cdf = pd.DataFrame({'intercept': np.ones(n)})
    cdf['fd_mean'] = df_high['fd_mean'].values - df_low['fd_mean'].values
    cdf['hc'] = df_high['hc'].values  # Same across acquisitions
    cdf['fd_mean_x_hc'] = df_high['fd_mean_x_hc'].values - df_low['fd_mean_x_hc'].values
    cdf['smoothness'] = df_high['smoothness'].values - df_low['smoothness'].values

    print(f"  fd_mean std: {cdf['fd_mean'].std():.6f}, smoothness std: {cdf['smoothness'].std():.6f}")
    return cdf

contrasts_pw = {
    'mb6me4_vs_mb1me4': (acquisition_matrices_pw['mb6me4'], acquisition_matrices_pw['mb1me4']),
    'mb3me4_vs_mb1me4': (acquisition_matrices_pw['mb3me4'], acquisition_matrices_pw['mb1me4']),
    'mb6me4_vs_mb3me4': (acquisition_matrices_pw['mb6me4'], acquisition_matrices_pw['mb3me4'])
}

for contrast_name, (df_high, df_low) in contrasts_pw.items():
    cdf = create_contrast_matrix_pw(df_high, df_low, contrast_name)
    contrast_dir = OUTPUT_DIR_PW / contrast_name
    contrast_dir.mkdir(exist_ok=True)

    tsv_file = contrast_dir / f'{contrast_name}_design_matrix.tsv'
    cdf.to_csv(tsv_file, sep='\t', index=False)
    print(f"  Saved TSV: {tsv_file.name}")

    mat_file = contrast_dir / f'{contrast_name}_design_matrix.mat'
    save_fsl_mat(cdf, mat_file)
    print(f"  Saved MAT: {mat_file.name}")

print("\n" + "=" * 80)
print("COMPLETE — 12 files total (6 TSV + 6 MAT)")
print("=" * 80)

# TEDANA Denoising Efficacy — Per-Acquisition ROI Analyses
ΔR² (tedana > base) vs. head motion for three ROIs, separately for MB1ME4, MB3ME4, and MB6ME4.

In [ ]:
# =============================================================================
# TEDANA BENEFIT ANALYSIS — DELTA R² BY ROI (PER ACQUISITION)
# =============================================================================
# Publication-quality figures with brain images for each ROI.
# Dependencies: project_root (from Setup)

from scipy import stats as sp_stats
import matplotlib.image as mpimg

plt.close('all')

print("=" * 80)
print("TEDANA BENEFIT ANALYSIS — DELTA R² BY ROI (PER ACQUISITION)")
print("=" * 80)

# --- Config ---
MODELS_DIR = Path.home() / 'Desktop' / 'projects' / 'multiecho-pilot' / 'randomise' / 'models'
BRAIN_IMG_DIR = project_root / 'derivatives' / 'figures' / 'brain_images'

TEDANA_ACQS = ['mb1me4', 'mb3me4', 'mb6me4']

TEDANA_ROIS = {
    'VS': {
        'label': 'Ventral Striatum',
        'brain_img': 'mask_VSconstrained.png',
        'slice_label': 'y = 15'
    },
    'rFFA': {
        'label': 'Right FFA',
        'brain_img': 'mask_rFFA.png',
        'slice_label': 'z = −18'
    },
    'bilateralMotor': {
        'label': 'Motor Cortex',
        'brain_img': 'mask_bilateralMotor.png',
        'slice_label': 'y = −24'
    }
}

# Nature-style settings
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 14,
    'axes.linewidth': 1,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
})

point_color = '#2c3e50'

# --- Process each acquisition ---
for acq in TEDANA_ACQS:
    print(f"\n{'='*60}")
    print(f"Processing {acq.upper()}")
    print(f"{'='*60}")

    data_dir = MODELS_DIR / acq
    design_matrix_path = data_dir / f'{acq}_design_matrix.tsv'

    print(f"Loading design matrix: {design_matrix_path}")
    design_matrix = pd.read_csv(design_matrix_path, sep='\t')
    design_matrix.columns = design_matrix.columns.str.strip()
    print(f"  {len(design_matrix)} observations")

    # Load signal data for each ROI
    roi_data_tedana = {}
    for roi_name, roi_info in TEDANA_ROIS.items():
        signal_file = data_dir / f'combined_r_square_diff_{acq}_mask-{roi_name}.txt'
        print(f"  Loading {roi_name}: {signal_file}")
        with open(signal_file, 'r') as f:
            signal_values = [float(line.strip()) for line in f if line.strip()]
        print(f"    {len(signal_values)} values")
        df = design_matrix.copy()
        df[f'{roi_name}_signal'] = signal_values
        roi_data_tedana[roi_name] = df

    # Load brain images and calculate aspect ratios
    brain_images = {}
    brain_aspects = {}
    for roi_name, roi_info in TEDANA_ROIS.items():
        brain_img_path = BRAIN_IMG_DIR / roi_info['brain_img']
        brain_img = mpimg.imread(str(brain_img_path))
        brain_images[roi_name] = brain_img
        img_h, img_w = brain_img.shape[:2]
        brain_aspects[roi_name] = img_w / img_h

    # Create 3-row figure: each row = brain image + scatterplot for one ROI
    # Use the average brain aspect ratio for consistent layout
    avg_brain_aspect = sum(brain_aspects.values()) / len(brain_aspects)
    scatter_ratio = 1.4

    fig, axes = plt.subplots(3, 2, figsize=(12, 15),
                              gridspec_kw={'width_ratios': [avg_brain_aspect, scatter_ratio], 
                                           'wspace': 0.25, 'hspace': 0.35})

    panel_labels = ['a', 'b', 'c', 'd', 'e', 'f']
    panel_idx = 0

    for idx, (roi_name, roi_info) in enumerate(TEDANA_ROIS.items()):
        ax_brain = axes[idx, 0]
        ax_scatter = axes[idx, 1]

        # --- Brain image panel ---
        ax_brain.imshow(brain_images[roi_name])
        ax_brain.axis('off')

        # --- Scatterplot panel ---
        df = roi_data_tedana[roi_name]
        sig_col = f'{roi_name}_signal'
        signal_inverted = df[sig_col] * -1

        ax_scatter.scatter(df['fd_mean'], signal_inverted,
                           color=point_color, s=50, alpha=0.7,
                           edgecolors='white', linewidth=0.5)

        # Regression line with confidence band
        slope, intercept, r_val, p_val, se = sp_stats.linregress(
            df['fd_mean'], signal_inverted)
        x_range = np.linspace(df['fd_mean'].min(), df['fd_mean'].max(), 100)
        y_pred = slope * x_range + intercept

        # 95% confidence interval
        n = len(df)
        x_mean = df['fd_mean'].mean()
        se_line = se * np.sqrt(1/n + (x_range - x_mean)**2 / np.sum((df['fd_mean'] - x_mean)**2))
        ci = 1.96 * se_line

        ax_scatter.fill_between(x_range, y_pred - ci, y_pred + ci,
                                color=point_color, alpha=0.15, linewidth=0)
        ax_scatter.plot(x_range, y_pred, color=point_color, linewidth=2, linestyle='-')

        # Axis labels
        ax_scatter.set_xlabel('Mean FD', fontsize=16)
        ax_scatter.set_ylabel(f'ΔR² (tedana > base, {acq.upper()})', fontsize=16)

        # Clean up axes
        ax_scatter.spines['top'].set_visible(False)
        ax_scatter.spines['right'].set_visible(False)
        ax_scatter.spines['left'].set_linewidth(1)
        ax_scatter.spines['bottom'].set_linewidth(1)

        # Tighter axis limits
        x_pad = (df['fd_mean'].max() - df['fd_mean'].min()) * 0.08
        y_pad = (signal_inverted.max() - signal_inverted.min()) * 0.08
        ax_scatter.set_xlim(df['fd_mean'].min() - x_pad, df['fd_mean'].max() + x_pad)
        ax_scatter.set_ylim(signal_inverted.min() - y_pad, signal_inverted.max() + y_pad)

        ax_scatter.grid(False)

        print(f"  {roi_info['label']}: r={r_val:.4f}, p={p_val:.4f}")

    # Add overall title for the acquisition
    fig.suptitle(acq.upper(), fontsize=20, fontweight='bold', y=0.98)

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Get axes positions AFTER tight_layout and add labels
    for idx, (roi_name, roi_info) in enumerate(TEDANA_ROIS.items()):
        ax_brain = axes[idx, 0]
        ax_scatter = axes[idx, 1]

        brain_bbox = ax_brain.get_position()
        scatter_bbox = ax_scatter.get_position()

        # Use the higher of the two tops for consistent y-coordinate
        top_y = max(brain_bbox.y1, scatter_bbox.y1) + 0.01

        # Place both labels at the same height
        fig.text(brain_bbox.x0, top_y, panel_labels[panel_idx], 
                 fontsize=18, fontweight='bold', va='bottom', ha='left')
        fig.text(scatter_bbox.x0 - 0.02, top_y, panel_labels[panel_idx + 1], 
                 fontsize=18, fontweight='bold', va='bottom', ha='left')

        # Add slice label for brain image
        fig.text(brain_bbox.x0 + brain_bbox.width/2, brain_bbox.y0 - 0.01, 
                 roi_info['slice_label'], fontsize=14, ha='center', va='top')

        panel_idx += 2

    out_file = data_dir / f'tedana_benefit_multi_roi_{acq}_publication.png'
    plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  Saved: {out_file}")
    plt.show()

print("\n" + "=" * 80)
print("TEDANA PER-ACQUISITION ANALYSIS COMPLETE")
print("=" * 80)

# TEDANA Whole-Brain Exploratory: dlPFC (MB6ME4)
Main effect of motion on TEDANA denoising efficacy in dlPFC, identified via whole-brain randomise analysis.

In [ ]:
# =============================================================================
# TEDANA BENEFIT — dlPFC (MB6ME4) — MAIN EFFECT OF MOTION
# =============================================================================
# Dependencies: project_root (from Setup)

from scipy import stats as sp_stats
import matplotlib.image as mpimg

plt.close('all')

print("=" * 80)
print("TEDANA BENEFIT — dlPFC (MB6ME4)")
print("=" * 80)

RANDOMISE_DIR = Path.home() / 'Desktop' / 'projects' / 'multiecho-pilot' / 'randomise' / 'models'
BRAIN_IMG_DIR = project_root / 'derivatives' / 'figures' / 'brain_images'

# Load design matrix
acq_dir = RANDOMISE_DIR / 'mb6me4'
dm_path = acq_dir / 'mb6me4_design_matrix.tsv'
print(f"Design matrix: {dm_path}")
df = pd.read_csv(dm_path, sep='\t')
df.columns = df.columns.str.strip()
print(f"  {len(df)} observations")

# Load signal
signal_path = acq_dir / 'dlpfc_cluster_signal.txt'
print(f"Signal file: {signal_path}")
with open(signal_path, 'r') as f:
    signal_values = [float(line.strip()) for line in f if line.strip()]
print(f"  {len(signal_values)} values loaded")

df['signal'] = signal_values
signal_inverted = df['signal'] * -1

# Load brain image FIRST to get its aspect ratio
brain_img_path = BRAIN_IMG_DIR / 'tedana_mb6me4_dlPFC.png'
print(f"Loading brain image: {brain_img_path}")
brain_img = mpimg.imread(str(brain_img_path))

# Calculate brain image aspect ratio (width / height)
img_h, img_w = brain_img.shape[:2]
brain_aspect = img_w / img_h
print(f"  Brain image dimensions: {img_w} x {img_h}, aspect ratio: {brain_aspect:.2f}")

# Nature-style settings
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 18,
    'axes.linewidth': 1,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
})

# Create figure with DYNAMIC width ratio based on brain image aspect
scatter_ratio = 1.4
fig, (ax_brain, ax_scatter) = plt.subplots(
    1, 2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [brain_aspect, scatter_ratio], 'wspace': 0.3}
)

# --- Panel A: Brain image ---
ax_brain.imshow(brain_img)
ax_brain.axis('off')

# --- Panel B: Scatterplot ---
point_color = '#2c3e50'

ax_scatter.scatter(df['fd_mean'], signal_inverted,
                   color=point_color, s=60, alpha=0.7,
                   edgecolors='white', linewidth=0.5)

# Regression line with confidence band
slope, intercept, r_val, p_val, se = sp_stats.linregress(df['fd_mean'], signal_inverted)
x_range = np.linspace(df['fd_mean'].min(), df['fd_mean'].max(), 100)
y_pred = slope * x_range + intercept

# 95% confidence interval
n = len(df)
x_mean = df['fd_mean'].mean()
se_line = se * np.sqrt(1/n + (x_range - x_mean)**2 / np.sum((df['fd_mean'] - x_mean)**2))
ci = 1.96 * se_line

ax_scatter.fill_between(x_range, y_pred - ci, y_pred + ci,
                        color=point_color, alpha=0.15, linewidth=0)
ax_scatter.plot(x_range, y_pred, color=point_color, linewidth=2, linestyle='-')

# Axis labels
ax_scatter.set_xlabel('Mean FD', fontsize=20)
ax_scatter.set_ylabel('ΔR² (tedana > base, MB6ME4)', fontsize=20)

# Clean up axes
ax_scatter.spines['top'].set_visible(False)
ax_scatter.spines['right'].set_visible(False)
ax_scatter.spines['left'].set_linewidth(1)
ax_scatter.spines['bottom'].set_linewidth(1)

# Tighter axis limits
x_pad = (df['fd_mean'].max() - df['fd_mean'].min()) * 0.08
y_pad = (signal_inverted.max() - signal_inverted.min()) * 0.08
ax_scatter.set_xlim(df['fd_mean'].min() - x_pad, df['fd_mean'].max() + x_pad)
ax_scatter.set_ylim(signal_inverted.min() - y_pad, signal_inverted.max() + y_pad)

ax_scatter.grid(False)

print(f"  Correlation: r = {r_val:.4f}, p = {p_val:.4f}")

plt.tight_layout()

# Get axes positions in figure coordinates AFTER tight_layout
brain_bbox = ax_brain.get_position()
scatter_bbox = ax_scatter.get_position()

# Use the higher of the two tops for consistent y-coordinate
top_y = max(brain_bbox.y1, scatter_bbox.y1) + 0.02

# Place both labels at the same height
fig.text(brain_bbox.x0, top_y, 'a', fontsize=24, fontweight='bold', va='bottom', ha='left')
fig.text(scatter_bbox.x0 - 0.03, top_y, 'b', fontsize=24, fontweight='bold', va='bottom', ha='left')

# Add slice label for brain image
fig.text(brain_bbox.x0 + brain_bbox.width/2, brain_bbox.y0 - 0.02, 'x = 44',
         fontsize=18, ha='center', va='top')

out_file = acq_dir / 'dlpfc_motion_tedana_mb6me4_publication.png'
plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {out_file}")
plt.show()

print("\n" + "=" * 80)
print("dlPFC ANALYSIS COMPLETE")
print("=" * 80)

# TEDANA Whole-Brain Exploratory: Left IPL (MB1ME4)
Motion × headcoil interaction on TEDANA denoising efficacy in left IPL, identified via whole-brain randomise analysis.

In [ ]:
# =============================================================================
# TEDANA BENEFIT — LEFT IPL (MB1ME4) — MOTION × HEADCOIL INTERACTION
# =============================================================================
# Dependencies: project_root (from Setup)

from scipy import stats as sp_stats
import matplotlib.image as mpimg

plt.close('all')

print("=" * 80)
print("TEDANA BENEFIT — LEFT IPL (MB1ME4)")
print("=" * 80)

RANDOMISE_DIR = Path.home() / 'Desktop' / 'projects' / 'multiecho-pilot' / 'randomise' / 'models'
BRAIN_IMG_DIR = project_root / 'derivatives' / 'figures' / 'brain_images'
HC_COLORS = {'64': '#2E86AB', '20': '#A23B72'}

# Load design matrix
acq_dir = RANDOMISE_DIR / 'mb1me4'
dm_path = acq_dir / 'mb1me4_design_matrix.tsv'
print(f"Design matrix: {dm_path}")
df = pd.read_csv(dm_path, sep='\t')
df.columns = df.columns.str.strip()
print(f"  {len(df)} observations")

# Load signal
signal_path = acq_dir / 'leftIPL_cluster_signal.txt'
print(f"Signal file: {signal_path}")
with open(signal_path, 'r') as f:
    signal_values = [float(line.strip()) for line in f if line.strip()]
print(f"  {len(signal_values)} values loaded")

df['signal'] = signal_values
signal_inverted = df['signal'] * -1

# Load brain image FIRST to get its aspect ratio
brain_img_path = BRAIN_IMG_DIR / 'tedana_mb1me4_leftIPL_motion_x_headcoil.png'
print(f"Loading brain image: {brain_img_path}")
brain_img = mpimg.imread(str(brain_img_path))

# Calculate brain image aspect ratio (width / height)
img_h, img_w = brain_img.shape[:2]
brain_aspect = img_w / img_h
print(f"  Brain image dimensions: {img_w} x {img_h}, aspect ratio: {brain_aspect:.2f}")

# Split by headcoil
hc_64 = df[df['hc'] > 0]
hc_20 = df[df['hc'] < 0]

# Nature-style settings
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 18,
    'axes.linewidth': 1,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
})

# Create figure with DYNAMIC width ratio based on brain image aspect
scatter_ratio = 1.4
fig, (ax_brain, ax_scatter) = plt.subplots(
    1, 2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [brain_aspect, scatter_ratio], 'wspace': 0.3}
)

# --- Panel A: Brain image ---
ax_brain.imshow(brain_img)
ax_brain.axis('off')

# --- Panel B: Scatterplot ---
ax_scatter.scatter(hc_64['fd_mean'], hc_64['signal'] * -1,
                   color=HC_COLORS['64'], s=60, alpha=0.7,
                   edgecolors='white', linewidth=0.5, label='64-channel')
ax_scatter.scatter(hc_20['fd_mean'], hc_20['signal'] * -1,
                   color=HC_COLORS['20'], s=60, alpha=0.7,
                   edgecolors='white', linewidth=0.5, label='20-channel')

# Separate regression lines with confidence bands
for hc_label, hc_subset, color in [('64', hc_64, HC_COLORS['64']),
                                    ('20', hc_20, HC_COLORS['20'])]:
    if len(hc_subset) >= 2:
        hc_signal = hc_subset['signal'] * -1
        slope, intercept, r_val, p_val, se = sp_stats.linregress(
            hc_subset['fd_mean'], hc_signal)
        x_range = np.linspace(hc_subset['fd_mean'].min(), hc_subset['fd_mean'].max(), 100)
        y_pred = slope * x_range + intercept

        # 95% confidence interval
        n = len(hc_subset)
        x_mean = hc_subset['fd_mean'].mean()
        se_line = se * np.sqrt(1/n + (x_range - x_mean)**2 / np.sum((hc_subset['fd_mean'] - x_mean)**2))
        ci = 1.96 * se_line

        ax_scatter.fill_between(x_range, y_pred - ci, y_pred + ci,
                                color=color, alpha=0.15, linewidth=0)
        ax_scatter.plot(x_range, y_pred, color=color, linewidth=2, linestyle='-')

        print(f"  {hc_label}-channel: r = {r_val:.4f}, p = {p_val:.4f}")

# Legend
ax_scatter.legend(frameon=True, fontsize=14, loc='best',
                  edgecolor='none', facecolor='white', framealpha=0.8)

# Axis labels
ax_scatter.set_xlabel('Mean FD', fontsize=20)
ax_scatter.set_ylabel('ΔR² (tedana > base, MB1ME4)', fontsize=20)

# Clean up axes
ax_scatter.spines['top'].set_visible(False)
ax_scatter.spines['right'].set_visible(False)
ax_scatter.spines['left'].set_linewidth(1)
ax_scatter.spines['bottom'].set_linewidth(1)

# Tighter axis limits
x_pad = (df['fd_mean'].max() - df['fd_mean'].min()) * 0.08
y_pad = (signal_inverted.max() - signal_inverted.min()) * 0.08
ax_scatter.set_xlim(df['fd_mean'].min() - x_pad, df['fd_mean'].max() + x_pad)
ax_scatter.set_ylim(signal_inverted.min() - y_pad, signal_inverted.max() + y_pad)

ax_scatter.grid(False)

plt.tight_layout()

# Get axes positions in figure coordinates AFTER tight_layout
brain_bbox = ax_brain.get_position()
scatter_bbox = ax_scatter.get_position()

# Use the higher of the two tops for consistent y-coordinate
top_y = max(brain_bbox.y1, scatter_bbox.y1) + 0.02

# Place both labels at the same height
fig.text(brain_bbox.x0, top_y, 'a', fontsize=24, fontweight='bold', va='bottom', ha='left')
fig.text(scatter_bbox.x0 - 0.03, top_y, 'b', fontsize=24, fontweight='bold', va='bottom', ha='left')

# Add slice label for brain image
fig.text(brain_bbox.x0 + brain_bbox.width/2, brain_bbox.y0 - 0.02, 'y = −45',
         fontsize=18, ha='center', va='top')

out_file = acq_dir / 'leftIPL_motion_tedana_mb1me4_publication.png'
plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {out_file}")
plt.show()

print("\n" + "=" * 80)
print("LEFT IPL ANALYSIS COMPLETE")
print("=" * 80)

# TEDANA Denoising Efficacy — Pairwise Acquisition Comparisons
ΔΔR² vs. Δ Mean FD for pairwise contrasts (MB3−MB1, MB6−MB3) across three ROIs.

In [ ]:
# =============================================================================
# TEDANA BENEFIT ANALYSIS — DELTA R² PAIRWISE COMPARISONS
# =============================================================================
# Publication-quality figures with brain images for each ROI.
# Dependencies: project_root (from Setup)

from scipy import stats as sp_stats
import matplotlib.image as mpimg

plt.close('all')

print("=" * 80)
print("TEDANA BENEFIT ANALYSIS — PAIRWISE COMPARISONS")
print("=" * 80)

MODELS_DIR_PW = Path.home() / 'Desktop' / 'projects' / 'multiecho-pilot' / 'randomise' / 'models'
BRAIN_IMG_DIR = project_root / 'derivatives' / 'figures' / 'brain_images'

PAIRWISE_COMPARISONS = [
    {'dir': 'mb3me4_vs_mb1me4', 'label': 'MB3ME4 vs MB1ME4',
     'x_label': 'Δ Mean FD'},
    {'dir': 'mb6me4_vs_mb3me4', 'label': 'MB6ME4 vs MB3ME4',
     'x_label': 'Δ Mean FD'},
    {'dir': 'mb6me4_vs_mb1me4', 'label': 'MB6ME4 vs MB1ME4',
     'x_label': 'Δ Mean FD'},
]

TEDANA_ROIS_PW = {
    'VS': {
        'label': 'Ventral Striatum',
        'brain_img': 'mask_VSconstrained.png',
        'slice_label': 'y = 15'
    },
    'rFFA': {
        'label': 'Right FFA',
        'brain_img': 'mask_rFFA.png',
        'slice_label': 'z = −18'
    },
    'bilateralMotor': {
        'label': 'Motor Cortex',
        'brain_img': 'mask_bilateralMotor.png',
        'slice_label': 'y = −24'
    }
}

# Nature-style settings
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 14,
    'axes.linewidth': 1,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
})

point_color = '#2c3e50'

for comp in PAIRWISE_COMPARISONS:
    print(f"\n{'='*60}")
    print(f"Processing {comp['label']}")
    print(f"{'='*60}")

    data_dir = MODELS_DIR_PW / comp['dir']
    dm_path = data_dir / f"design_matrix_{comp['dir']}.csv"

    design_matrix = pd.read_csv(dm_path)
    design_matrix.columns = design_matrix.columns.str.strip()
    print(f"  {len(design_matrix)} observations")

    # Load signal data for all ROIs
    roi_data = {}
    for roi_name, roi_info in TEDANA_ROIS_PW.items():
        sig_file = data_dir / f"combined_r_square_diff_{comp['dir']}_mask-{roi_name}.txt"
        with open(sig_file, 'r') as f:
            vals = [float(line.strip()) for line in f if line.strip()]
        df = design_matrix.copy()
        df[f'{roi_name}_signal'] = vals
        roi_data[roi_name] = df

    # Create 3-row figure: each row = brain image + scatterplot for one ROI
    fig, axes = plt.subplots(3, 2, figsize=(12, 15),
                              gridspec_kw={'width_ratios': [1, 1.4], 'wspace': 0.25, 'hspace': 0.35})

    panel_labels = ['a', 'b', 'c', 'd', 'e', 'f']
    panel_idx = 0

    for idx, (roi_name, roi_info) in enumerate(TEDANA_ROIS_PW.items()):
        ax_brain = axes[idx, 0]
        ax_scatter = axes[idx, 1]

        # --- Brain image panel ---
        brain_img_path = BRAIN_IMG_DIR / roi_info['brain_img']
        if brain_img_path.exists():
            brain_img = mpimg.imread(str(brain_img_path))
            ax_brain.imshow(brain_img)
        ax_brain.axis('off')
        ax_brain.text(0.5, -0.06, roi_info['slice_label'], transform=ax_brain.transAxes,
                      fontsize=14, ha='center', va='top')
        ax_brain.text(-0.05, 1.05, panel_labels[panel_idx], transform=ax_brain.transAxes,
                      fontsize=18, fontweight='bold', va='bottom', ha='left')
        panel_idx += 1

        # --- Scatterplot panel ---
        df = roi_data[roi_name]
        sig_col = f'{roi_name}_signal'
        signal_inverted = df[sig_col] * -1

        ax_scatter.scatter(df['fd_mean_demeaned'], signal_inverted,
                           color=point_color, s=50, alpha=0.7,
                           edgecolors='white', linewidth=0.5)

        # Regression line with confidence band
        slope, intercept, r_val, p_val, se = sp_stats.linregress(
            df['fd_mean_demeaned'], signal_inverted)
        x_range = np.linspace(df['fd_mean_demeaned'].min(), df['fd_mean_demeaned'].max(), 100)
        y_pred = slope * x_range + intercept

        # 95% confidence interval
        n = len(df)
        x_mean = df['fd_mean_demeaned'].mean()
        se_line = se * np.sqrt(1/n + (x_range - x_mean)**2 / np.sum((df['fd_mean_demeaned'] - x_mean)**2))
        ci = 1.96 * se_line

        ax_scatter.fill_between(x_range, y_pred - ci, y_pred + ci,
                                color=point_color, alpha=0.15, linewidth=0)
        ax_scatter.plot(x_range, y_pred, color=point_color, linewidth=2, linestyle='-')

        # Axis labels
        ax_scatter.set_xlabel(comp['x_label'], fontsize=16)
        ax_scatter.set_ylabel('ΔΔR² (tedana > base)', fontsize=16)

        # Clean up axes
        ax_scatter.spines['top'].set_visible(False)
        ax_scatter.spines['right'].set_visible(False)
        ax_scatter.spines['left'].set_linewidth(1)
        ax_scatter.spines['bottom'].set_linewidth(1)

        # Tighter axis limits
        x_pad = (df['fd_mean_demeaned'].max() - df['fd_mean_demeaned'].min()) * 0.08
        y_pad = (signal_inverted.max() - signal_inverted.min()) * 0.08
        ax_scatter.set_xlim(df['fd_mean_demeaned'].min() - x_pad, df['fd_mean_demeaned'].max() + x_pad)
        ax_scatter.set_ylim(signal_inverted.min() - y_pad, signal_inverted.max() + y_pad)

        ax_scatter.grid(False)
        ax_scatter.text(-0.12, 1.05, panel_labels[panel_idx], transform=ax_scatter.transAxes,
                        fontsize=18, fontweight='bold', va='bottom', ha='left')
        panel_idx += 1

        print(f"  {roi_info['label']}: r={r_val:.4f}, p={p_val:.4f}")

    # Add overall title for the comparison
    fig.suptitle(comp['label'], fontsize=20, fontweight='bold', y=0.98)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    out_file = data_dir / f"tedana_benefit_multi_roi_{comp['dir']}_publication.png"
    plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  Saved: {out_file}")
    plt.show()

print("\n" + "=" * 80)
print("TEDANA PAIRWISE ANALYSIS COMPLETE")
print("=" * 80)

# Whole-Brain Cluster Analysis: MB6ME4 vs MB1ME4 (Amygdala)
ΔΔR² extracted from Cluster 2 (~amygdala, cluster-corrected) plotted against Δ Mean FD, with brain image showing cluster location.

In [ ]:
# =============================================================================
# TEDANA BENEFIT — WHOLE-BRAIN CLUSTER 2 / AMYGDALA (MB6ME4 vs MB1ME4)
# =============================================================================
# Plots ΔΔR² (tedana > base, mb6me4 − mb1me4) against Δfd_mean for Cluster 2
# (amygdala) identified from whole-brain randomise analysis of tstat1.
# Includes brain image showing cluster location alongside scatterplot.
# Dependencies: project_root (from Setup)

from scipy import stats as sp_stats
import matplotlib.image as mpimg

plt.close('all')

print("=" * 80)
print("TEDANA BENEFIT — AMYGDALA CLUSTER (MB6ME4 vs MB1ME4)")
print("=" * 80)

CONTRAST_DIR = Path.home() / 'Desktop' / 'projects' / 'multiecho-pilot' / 'randomise' / 'models' / 'mb6me4_vs_mb1me4'
BRAIN_IMG_PATH = project_root / 'derivatives' / 'figures' / 'brain_images' / 'tedana_mb6me4_vs_mb1me4_cluster2_amygdala.png'

# Load pairwise design matrix
dm_path = CONTRAST_DIR / 'mb6me4_vs_mb1me4_design_matrix.tsv'
print(f"Design matrix: {dm_path}")
dm = pd.read_csv(dm_path, sep='\t')
dm.columns = dm.columns.str.strip()
print(f"  {len(dm)} observations")

# Load Cluster 2 (amygdala) signal
CLUSTER_ID = 2
sig_path = CONTRAST_DIR / f'combined_r_square_diff_mb6me4_vs_mb1me4_cluster{CLUSTER_ID}.txt'
print(f"Loading Cluster {CLUSTER_ID} (Amygdala): {sig_path}")
with open(sig_path, 'r') as f:
    cluster_signal = [float(line.strip()) for line in f if line.strip()]
print(f"  {len(cluster_signal)} values")

# Load brain image FIRST to get its aspect ratio
print(f"Loading brain image: {BRAIN_IMG_PATH}")
brain_img = mpimg.imread(str(BRAIN_IMG_PATH))

# Calculate brain image aspect ratio (width / height)
img_h, img_w = brain_img.shape[:2]
brain_aspect = img_w / img_h
print(f"  Brain image dimensions: {img_w} x {img_h}, aspect ratio: {brain_aspect:.2f}")

# =============================================================================
# CREATE PUBLICATION-QUALITY FIGURE
# =============================================================================

# Nature-style settings
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 18,
    'axes.linewidth': 1,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
})

# Create figure with DYNAMIC width ratio based on brain image aspect
scatter_ratio = 1.4
fig, (ax_brain, ax_scatter) = plt.subplots(
    1, 2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [brain_aspect, scatter_ratio], 'wspace': 0.3}
)

# --- Panel A: Brain image ---
ax_brain.imshow(brain_img)
ax_brain.axis('off')

# --- Panel B: Scatterplot ---
df = dm.copy()
df['signal'] = cluster_signal
signal_inverted = df['signal'] * -1

# Single color for all points (clean navy)
point_color = '#2c3e50'

ax_scatter.scatter(df['fd_mean'], signal_inverted,
                   color=point_color, s=60, alpha=0.7,
                   edgecolors='white', linewidth=0.5)

# Regression line with confidence band
slope, intercept, r_val, p_val, se = sp_stats.linregress(df['fd_mean'], signal_inverted)
x_range = np.linspace(df['fd_mean'].min(), df['fd_mean'].max(), 100)
y_pred = slope * x_range + intercept

# Confidence interval (95%)
n = len(df)
x_mean = df['fd_mean'].mean()
se_line = se * np.sqrt(1/n + (x_range - x_mean)**2 / np.sum((df['fd_mean'] - x_mean)**2))
ci = 1.96 * se_line

ax_scatter.fill_between(x_range, y_pred - ci, y_pred + ci,
                        color=point_color, alpha=0.15, linewidth=0)
ax_scatter.plot(x_range, y_pred, color=point_color, linewidth=2, linestyle='-')

# Axis labels
ax_scatter.set_xlabel('Δ Mean FD', fontsize=20)
ax_scatter.set_ylabel('ΔΔR² (tedana > base, MB6−MB1)', fontsize=20)

# Clean up axes
ax_scatter.spines['top'].set_visible(False)
ax_scatter.spines['right'].set_visible(False)
ax_scatter.spines['left'].set_linewidth(1)
ax_scatter.spines['bottom'].set_linewidth(1)

# Tighter axis limits
x_pad = (df['fd_mean'].max() - df['fd_mean'].min()) * 0.08
y_pad = (signal_inverted.max() - signal_inverted.min()) * 0.08
ax_scatter.set_xlim(df['fd_mean'].min() - x_pad, df['fd_mean'].max() + x_pad)
ax_scatter.set_ylim(signal_inverted.min() - y_pad, signal_inverted.max() + y_pad)

ax_scatter.grid(False)

print(f"  Amygdala: r = {r_val:.4f}, p = {p_val:.4f}")

plt.tight_layout()

# Get axes positions in figure coordinates AFTER tight_layout
brain_bbox = ax_brain.get_position()
scatter_bbox = ax_scatter.get_position()

# Use the higher of the two tops for consistent y-coordinate
top_y = max(brain_bbox.y1, scatter_bbox.y1) + 0.02

# Place both labels at the same height
fig.text(brain_bbox.x0, top_y, 'a', fontsize=24, fontweight='bold', va='bottom', ha='left')
fig.text(scatter_bbox.x0 - 0.03, top_y, 'b', fontsize=24, fontweight='bold', va='bottom', ha='left')

# Add slice label for brain image
fig.text(brain_bbox.x0 + brain_bbox.width/2, brain_bbox.y0 - 0.02, 'z = −18',
         fontsize=18, ha='center', va='top')

out_file = CONTRAST_DIR / 'amygdala_cluster2_tedana_mb6me4_vs_mb1me4_with_brain.png'
plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\nSaved: {out_file}")
plt.show()

print("\n" + "=" * 80)
print("AMYGDALA CLUSTER ANALYSIS COMPLETE")
print("=" * 80)

# TEDANA Component Rejection vs. Motion
 
Assessing whether head motion (fd_mean) predicts the number or proportion of rejected ICA components from TEDANA.
Analysis is conducted at the subject × MB level, treating each acquisition as a separate observation.
Mixed-effects model accounts for repeated measures within subjects.

In [ ]:
# =============================================================================
# TEDANA COMPONENT REJECTION VS. MOTION ANALYSIS
# =============================================================================
# Tests whether fd_mean predicts the number/proportion of rejected components.
# Dependencies: project_root, design_matrices, pandas2ri, ro, lme4, lmerTest (from Setup)
 
from scipy import stats as sp_stats
import statsmodels.api as sm
 
plt.close('all')
 
print("=" * 80)
print("TEDANA COMPONENT REJECTION VS. MOTION ANALYSIS")
print("=" * 80)
 
# --- Configuration ---
TEDANA_DERIV_DIR = Path.home() / 'Desktop' / 'projects' / 'multiecho-pilot' / 'derivatives' / 'tedana'
MODELS_DIR = Path.home() / 'Desktop' / 'projects' / 'multiecho-pilot' / 'randomise' / 'models'
TEDANA_ACQS_REJ = ['mb1me4', 'mb3me4', 'mb6me4']
 
# Nature-style settings (consistent with other TEDANA plots)
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 14,
    'axes.linewidth': 1,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
})
 
point_color = '#2c3e50'
 
# --- Step 1: Parse TEDANA metrics files ---
print("\n--- Parsing TEDANA metrics files ---")
 
records = []
 
for acq in TEDANA_ACQS_REJ:
    print(f"\nProcessing {acq}...")
    
    # Load design matrix to get subjects with fd_mean
    design_matrix_path = MODELS_DIR / acq / f'{acq}_design_matrix.tsv'
    if not design_matrix_path.exists():
        print(f"  Warning: Design matrix not found: {design_matrix_path}")
        continue
    
    dm = pd.read_csv(design_matrix_path, sep='\t')
    dm.columns = dm.columns.str.strip()
    
    # Iterate over subjects in the design matrix
    for _, row in dm.iterrows():
        subj = str(int(row['subject'])).zfill(5)  # Ensure 5-digit format
        subj_dir = TEDANA_DERIV_DIR / f'sub-{subj}'
        
        # Look for the tedana metrics file
        metrics_pattern = f'sub-{subj}_task-sharedreward_acq-{acq}_desc-tedana_metrics.tsv'
        metrics_file = subj_dir / metrics_pattern
        
        if not metrics_file.exists():
            print(f"  Warning: Metrics file not found for sub-{subj} {acq}")
            continue
        
        # Parse metrics file
        try:
            metrics_df = pd.read_csv(metrics_file, sep='\t')
            
            # Count accepted and rejected components
            n_total = len(metrics_df)
            n_accepted = (metrics_df['classification'] == 'accepted').sum()
            n_rejected = (metrics_df['classification'] == 'rejected').sum()
            prop_rejected = n_rejected / n_total if n_total > 0 else np.nan
            
            records.append({
                'subject': subj,
                'acq': acq,
                'mb': acq[:3],  # 'mb1', 'mb3', or 'mb6'
                'n_total': n_total,
                'n_accepted': n_accepted,
                'n_rejected': n_rejected,
                'prop_rejected': prop_rejected,
                'fd_mean': row['fd_mean']
            })
        except Exception as e:
            print(f"  Error parsing {metrics_file}: {e}")
 
df_rej = pd.DataFrame(records)
print(f"\n{len(df_rej)} observations loaded ({len(df_rej['subject'].unique())} subjects × {len(TEDANA_ACQS_REJ)} acquisitions)")
 
# --- Step 2: Descriptive statistics ---
print("\n--- Descriptive Statistics ---")
print(df_rej.groupby('acq')[['n_total', 'n_accepted', 'n_rejected', 'prop_rejected', 'fd_mean']].describe().round(3))
 
# --- Step 3: Mixed-effects model (proportion rejected ~ fd_mean + (1|subject)) ---
print("\n--- Mixed-Effects Model: Proportion Rejected ~ FD Mean ---")
 
# Prepare data for R
df_for_r = df_rej.dropna(subset=['prop_rejected', 'fd_mean']).copy()
df_for_r['subject'] = df_for_r['subject'].astype(str)
df_for_r['mb'] = df_for_r['mb'].astype(str)
 
r_df = pandas2ri.py2rpy(df_for_r)
ro.globalenv['df_rej'] = r_df
 
ro.r('''
library(lme4)
library(lmerTest)
 
df_rej$subject <- as.factor(df_rej$subject)
df_rej$mb <- factor(df_rej$mb, levels = c("mb1", "mb3", "mb6"))
 
# Model: proportion rejected ~ fd_mean + (1|subject)
model_prop <- lmer(prop_rejected ~ fd_mean + (1 | subject), data = df_rej, REML = TRUE)
summary_prop <- summary(model_prop)
anova_prop <- anova(model_prop, type = 3, ddf = "Satterthwaite")
 
# Model with MB as covariate
model_prop_mb <- lmer(prop_rejected ~ fd_mean + mb + (1 | subject), data = df_rej, REML = TRUE)
summary_prop_mb <- summary(model_prop_mb)
anova_prop_mb <- anova(model_prop_mb, type = 3, ddf = "Satterthwaite")
 
# Model: n_rejected ~ fd_mean + (1|subject)
model_n <- lmer(n_rejected ~ fd_mean + (1 | subject), data = df_rej, REML = TRUE)
summary_n <- summary(model_n)
anova_n <- anova(model_n, type = 3, ddf = "Satterthwaite")
 
# Model with MB as covariate
model_n_mb <- lmer(n_rejected ~ fd_mean + mb + (1 | subject), data = df_rej, REML = TRUE)
summary_n_mb <- summary(model_n_mb)
anova_n_mb <- anova(model_n_mb, type = 3, ddf = "Satterthwaite")
''')
 
# Print results
print("\n=== PROPORTION REJECTED ~ FD_MEAN ===")
print(ro.r('summary_prop'))
print("\nANOVA (Type III):")
anova_prop = pandas2ri.rpy2py(ro.r('as.data.frame(anova_prop)'))
print(anova_prop.to_string())
 
print("\n=== PROPORTION REJECTED ~ FD_MEAN + MB ===")
print(ro.r('summary_prop_mb'))
print("\nANOVA (Type III):")
anova_prop_mb = pandas2ri.rpy2py(ro.r('as.data.frame(anova_prop_mb)'))
print(anova_prop_mb.to_string())
 
print("\n=== N_REJECTED ~ FD_MEAN ===")
print(ro.r('summary_n'))
print("\nANOVA (Type III):")
anova_n = pandas2ri.rpy2py(ro.r('as.data.frame(anova_n)'))
print(anova_n.to_string())
 
print("\n=== N_REJECTED ~ FD_MEAN + MB ===")
print(ro.r('summary_n_mb'))
print("\nANOVA (Type III):")
anova_n_mb = pandas2ri.rpy2py(ro.r('as.data.frame(anova_n_mb)'))
print(anova_n_mb.to_string())
 
# --- Step 4: Visualization ---
print("\n--- Generating Figures ---")
 
# Color palette for MB levels
mb_colors = {'mb1': '#1f77b4', 'mb3': '#ff7f0e', 'mb6': '#2ca02c'}
 
# Figure 1: Proportion rejected vs fd_mean (colored by MB)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
 
# Panel A: Proportion rejected
ax = axes[0]
for mb in ['mb1', 'mb3', 'mb6']:
    subset = df_rej[df_rej['mb'] == mb]
    ax.scatter(subset['fd_mean'], subset['prop_rejected'],
               color=mb_colors[mb], s=60, alpha=0.7,
               edgecolors='white', linewidth=0.5, label=mb.upper())
 
# Overall regression line
slope, intercept, r_val, p_val, se = sp_stats.linregress(
    df_rej['fd_mean'].dropna(), df_rej['prop_rejected'].dropna())
x_range = np.linspace(df_rej['fd_mean'].min(), df_rej['fd_mean'].max(), 100)
y_pred = slope * x_range + intercept
 
# 95% CI band
n = len(df_rej.dropna(subset=['fd_mean', 'prop_rejected']))
x_mean = df_rej['fd_mean'].mean()
se_line = se * np.sqrt(1/n + (x_range - x_mean)**2 / np.sum((df_rej['fd_mean'] - x_mean)**2))
ci = 1.96 * se_line
 
ax.fill_between(x_range, y_pred - ci, y_pred + ci,
                color='gray', alpha=0.2, linewidth=0)
ax.plot(x_range, y_pred, color='black', linewidth=2, linestyle='-')
 
ax.set_xlabel('Mean FD', fontsize=16)
ax.set_ylabel('Proportion Rejected', fontsize=16)
ax.set_title(f'r = {r_val:.3f}, p = {p_val:.4f}', fontsize=14, style='italic')
 
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(title='MB', loc='upper left', frameon=False)
 
# Tighter axis limits
x_pad = (df_rej['fd_mean'].max() - df_rej['fd_mean'].min()) * 0.08
y_pad = (df_rej['prop_rejected'].max() - df_rej['prop_rejected'].min()) * 0.08
ax.set_xlim(df_rej['fd_mean'].min() - x_pad, df_rej['fd_mean'].max() + x_pad)
ax.set_ylim(df_rej['prop_rejected'].min() - y_pad, df_rej['prop_rejected'].max() + y_pad)
 
# Panel B: Number rejected
ax = axes[1]
for mb in ['mb1', 'mb3', 'mb6']:
    subset = df_rej[df_rej['mb'] == mb]
    ax.scatter(subset['fd_mean'], subset['n_rejected'],
               color=mb_colors[mb], s=60, alpha=0.7,
               edgecolors='white', linewidth=0.5, label=mb.upper())
 
# Overall regression line
slope_n, intercept_n, r_val_n, p_val_n, se_n = sp_stats.linregress(
    df_rej['fd_mean'].dropna(), df_rej['n_rejected'].dropna())
y_pred_n = slope_n * x_range + intercept_n
 
# 95% CI band
se_line_n = se_n * np.sqrt(1/n + (x_range - x_mean)**2 / np.sum((df_rej['fd_mean'] - x_mean)**2))
ci_n = 1.96 * se_line_n
 
ax.fill_between(x_range, y_pred_n - ci_n, y_pred_n + ci_n,
                color='gray', alpha=0.2, linewidth=0)
ax.plot(x_range, y_pred_n, color='black', linewidth=2, linestyle='-')
 
ax.set_xlabel('Mean FD', fontsize=16)
ax.set_ylabel('N Components Rejected', fontsize=16)
ax.set_title(f'r = {r_val_n:.3f}, p = {p_val_n:.4f}', fontsize=14, style='italic')
 
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(title='MB', loc='upper left', frameon=False)
 
# Tighter axis limits
y_pad_n = (df_rej['n_rejected'].max() - df_rej['n_rejected'].min()) * 0.08
ax.set_xlim(df_rej['fd_mean'].min() - x_pad, df_rej['fd_mean'].max() + x_pad)
ax.set_ylim(df_rej['n_rejected'].min() - y_pad_n, df_rej['n_rejected'].max() + y_pad_n)
 
# Panel labels
fig.text(0.02, 0.95, 'a', fontsize=18, fontweight='bold', va='top', ha='left')
fig.text(0.51, 0.95, 'b', fontsize=18, fontweight='bold', va='top', ha='left')
 
plt.tight_layout()
 
out_file = output_dir / 'tedana_component_rejection_vs_motion.png'
plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {out_file}")
plt.show()
 
# --- Step 5: Summary statistics table ---
print("\n--- Summary by Acquisition ---")
summary_by_acq = df_rej.groupby('acq').agg({
    'n_total': ['mean', 'std'],
    'n_rejected': ['mean', 'std'],
    'prop_rejected': ['mean', 'std'],
    'fd_mean': ['mean', 'std']
}).round(3)
print(summary_by_acq)
 
# Correlations within each MB level
print("\n--- Correlations by MB Level ---")
for mb in ['mb1', 'mb3', 'mb6']:
    subset = df_rej[df_rej['mb'] == mb].dropna(subset=['fd_mean', 'prop_rejected'])
    if len(subset) > 2:
        r, p = sp_stats.pearsonr(subset['fd_mean'], subset['prop_rejected'])
        print(f"  {mb.upper()}: r = {r:.4f}, p = {p:.4f}, n = {len(subset)}")
 
print("\n" + "=" * 80)
print("TEDANA COMPONENT REJECTION ANALYSIS COMPLETE")
print("=" * 80)

# Special Protocol (SP) Subject Analysis (Fig. 9)

Beta, z-stat, and weighted beta activation in VS for SP subjects across 6 special acquisitions.
LMM, Tukey comparisons, and EMM plots. Weighted beta uses inverse-variance weighting (1/varcope).

In [ ]:
# =============================================================================
# SP SUBJECT ANALYSIS — FIGURE 9 (Beta, Zstat, and Weighted Beta)
# =============================================================================
# Dependencies: project_root, output_dir, pandas2ri, ro (from Setup)

import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

print("=" * 80)
print("SP SUBJECT ANALYSIS — FIGURE 9 (Beta, Zstat, and Weighted Beta)")
print("=" * 80)

# --- Config ---
SP_BASE_DIR = project_root / 'derivatives' / 'extractions'
SP_ACQ_PARAMS = ["mb3me1fa50", "mb3me3", "mb3me3ip0", "mb2me4", "mb3me4", "mb3me4fa50"]
SP_CUSTOM_LABELS = ["MB3ME1\nFA50", "MB3ME3", "MB3ME3\nIP0", "MB2ME4", "MB3ME4", "MB3ME4\nFA50"]
POINT_COLOR = '#2c3e50'

# R factor integer codes → acquisition names mapping
# (R factors get converted to integers when passing through pandas2ri)
SP_ACQ_FACTOR_MAP = {
    '1': 'mb3me1fa50', '2': 'mb3me3', '3': 'mb3me3ip0',
    '4': 'mb2me4', '5': 'mb3me4', '6': 'mb3me4fa50'
}

# =============================================================================
# FUNCTIONS
# =============================================================================

def extract_sp_file_data(base_dir, acq_params, type_value, img_value, mask_value, denoise_value):
    """Extract data from text files for SP subjects only (subject ID contains 'sp')."""
    pattern = re.compile(
        r"ts_sub-(\d+[a-zA-Z]*)_acq_([^_]+)_type-(.+?)_img-([^_]+)_mask-([^_]+)_denoise_([^\.]+)\.txt"
    )
    data_by_subject = {}
    file_paths = glob.glob(str(base_dir / "*.txt"))
    matched = 0

    for fp in file_paths:
        filename = os.path.basename(fp)
        match = pattern.match(filename)
        if match:
            sub_id, acq, file_type, img, mask, denoise = match.groups()
            if 'sp' not in sub_id:
                continue
            if (file_type.lower() == type_value.lower() and
                img.lower() == img_value.lower() and
                mask.lower() == mask_value.lower() and
                denoise.lower() == denoise_value.lower() and
                acq in acq_params):
                try:
                    with open(fp, 'r') as f:
                        value = float(f.read().strip())
                    if sub_id not in data_by_subject:
                        data_by_subject[sub_id] = {a: np.nan for a in acq_params}
                    data_by_subject[sub_id][acq] = value
                    matched += 1
                except (ValueError, IOError) as e:
                    print(f"Error processing {filename}: {e}")

    print(f"  Matched {matched} files")
    return data_by_subject


def create_sp_dataframe(data_by_subject, acq_params):
    """Convert extracted data dict into a tidy DataFrame."""
    df = pd.DataFrame.from_dict(data_by_subject, orient='index').reset_index()
    df = df.rename(columns={'index': 'subject'})
    return df[['subject'] + acq_params].sort_values('subject')


def create_sp_emm_plot(emm_df, df_long, acq_params, img_value, mask_value, out_dir, 
                        type_value, denoise_value):
    """
    Publication-quality EMM plot with individual data points overlaid.
    Falls back to raw means ± SEM if EMMs are not available.
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x_positions = np.arange(len(acq_params))
    
    # --- Individual subject points (jittered) ---
    #jitter_width = 0.15
    #np.random.seed(42)
    #for idx, acq in enumerate(acq_params):
    #    acq_data = df_long[df_long['acq'] == acq][img_value].dropna()
    #    if len(acq_data) > 0:
    #        jitter = np.random.uniform(-jitter_width, jitter_width, size=len(acq_data))
    #        ax.scatter(np.full(len(acq_data), idx) + jitter, acq_data,
    #                   color=POINT_COLOR, s=40, alpha=0.4,
    #                   edgecolors='white', linewidth=0.5, zorder=2)
    
    # --- EMM or fallback to raw means ---
    means_vals = []
    se_vals = []
    
    # Check if we have valid EMM data with correct columns
    use_emm = False
    if emm_df is not None and len(emm_df) > 0:
        # Make a copy to avoid modifying the original
        emm_df = emm_df.copy()
        
        # Normalize column names to lowercase
        emm_df.columns = emm_df.columns.str.strip().str.lower()
        
        # Convert acq values to strings
        if 'acq' in emm_df.columns:
            emm_df['acq'] = emm_df['acq'].astype(str).str.strip()
            
            # CRITICAL FIX: Remap R factor integers back to acquisition names
            # R factors get converted to integer codes ('1','2','3'...) when 
            # passing through pandas2ri, not the actual level labels
            first_val = emm_df['acq'].iloc[0]
            if first_val in SP_ACQ_FACTOR_MAP:
                emm_df['acq'] = emm_df['acq'].map(SP_ACQ_FACTOR_MAP)
                print(f"  Remapped factor codes to acquisition names")
        
        if 'emmean' in emm_df.columns and 'se' in emm_df.columns:
            use_emm = True
            print(f"  Using EMM data: {len(emm_df)} rows")
            print(f"  EMM acq values after remapping: {emm_df['acq'].tolist()}")
    
    if use_emm:
        for acq in acq_params:
            acq_emm = emm_df[emm_df['acq'] == acq]
            if not acq_emm.empty:
                means_vals.append(acq_emm['emmean'].values[0])
                se_vals.append(acq_emm['se'].values[0])
            else:
                print(f"  WARNING: No EMM data for {acq}")
                means_vals.append(np.nan)
                se_vals.append(np.nan)
        ylabel_prefix = f'{img_value.capitalize()} EMM'
    else:
        # Fallback: compute raw means and SEM
        print("  EMM data not available, using raw means ± SEM")
        for acq in acq_params:
            acq_data = df_long[df_long['acq'] == acq][img_value].dropna()
            if len(acq_data) > 0:
                means_vals.append(acq_data.mean())
                se_vals.append(acq_data.sem())
            else:
                means_vals.append(np.nan)
                se_vals.append(np.nan)
        ylabel_prefix = f'{img_value.capitalize()} Mean'
    
    # Filter valid points for plotting
    valid_indices = [i for i, v in enumerate(means_vals) if not np.isnan(v)]
    valid_x = [x_positions[i] for i in valid_indices]
    valid_means = [means_vals[i] for i in valid_indices]
    valid_se = [se_vals[i] for i in valid_indices]
    
    print(f"  Valid points to plot: {len(valid_x)}")
    
    # Plot line connecting means (only valid points)
    if len(valid_x) > 1:
        ax.plot(valid_x, valid_means, color=POINT_COLOR, linewidth=2, 
                linestyle='-', zorder=3)
    
    # Plot mean points with error bars
    if len(valid_x) > 0:
        ax.errorbar(valid_x, valid_means, yerr=valid_se,
                    fmt='o', color=POINT_COLOR, markersize=10,
                    capsize=4, capthick=1.5, elinewidth=1.5,
                    markeredgecolor='white', markeredgewidth=1.5, zorder=4)
    
    # --- Axis labels ---
    ax.set_xlabel('Acquisition', fontsize=16)
    ax.set_ylabel(f'{ylabel_prefix} ({mask_value})', fontsize=16)
    
    # --- X-axis setup ---
    ax.set_xticks(x_positions)
    ax.set_xticklabels(SP_CUSTOM_LABELS, rotation=45, ha='right', fontsize=12)
    ax.tick_params(axis='y', labelsize=12)
    
    # --- Clean up axes ---
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_linewidth(1)
    ax.grid(False)
    
    # --- Tighter y-axis limits ---
    all_vals = df_long[img_value].dropna().values
    valid_pairs = [(v, e) for v, e in zip(means_vals, se_vals) 
                   if not np.isnan(v) and not np.isnan(e)]
    
    if len(all_vals) > 0 and len(valid_pairs) > 0:
        emm_mins = [v - e for v, e in valid_pairs]
        emm_maxs = [v + e for v, e in valid_pairs]
        y_min = min(all_vals.min(), min(emm_mins))
        y_max = max(all_vals.max(), max(emm_maxs))
        y_pad = (y_max - y_min) * 0.1
        ax.set_ylim(y_min - y_pad, y_max + y_pad)
    elif len(all_vals) > 0:
        y_pad = (all_vals.max() - all_vals.min()) * 0.1
        ax.set_ylim(all_vals.min() - y_pad, all_vals.max() + y_pad)
    
    plt.tight_layout()
    
    # Save
    if out_dir:
        plot_file = os.path.join(str(out_dir),
                                 f"emm_plot_{type_value}_{img_value}_{mask_value}_{denoise_value}_publication.png")
        plt.savefig(plot_file, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"  EMM plot saved: {plot_file}")
    
    plt.show()
    return fig


def process_sp_and_visualize(base_dir, acq_params, type_value, img_value,
                              mask_value, denoise_value, out_dir=None):
    """Full pipeline: extract, analyze, visualize SP subject data."""
    if out_dir is None:
        out_dir = str(output_dir)
    os.makedirs(str(out_dir), exist_ok=True)

    print(f"\nProcessing: type={type_value}, img={img_value}, mask={mask_value}, denoise={denoise_value}")

    # Extract
    data_by_subject = extract_sp_file_data(base_dir, acq_params, type_value, img_value, mask_value, denoise_value)
    if not data_by_subject:
        print("  No matching files found.")
        return pd.DataFrame(columns=['subject'] + acq_params), None, None

    df = create_sp_dataframe(data_by_subject, acq_params)
    csv_out = os.path.join(str(out_dir), f"multiecho_data_{type_value}_{img_value}_{mask_value}_{denoise_value}.csv")
    df.to_csv(csv_out, index=False)
    print(f"  Data saved: {csv_out}")
    print(f"  {len(df)} subjects, missing per col:\n{df.isnull().sum()}")

    # Reshape for stats
    df_long = pd.melt(df, id_vars=['subject'], value_vars=acq_params,
                       var_name='acq', value_name=img_value)
    df_long['acq'] = pd.Categorical(df_long['acq'], categories=acq_params, ordered=True)
    df_long_clean = df_long.dropna(subset=[img_value])

    # Python LMM
    result = None
    if len(df_long_clean) >= 2:
        try:
            model = smf.mixedlm(f"{img_value} ~ acq", df_long_clean, groups=df_long_clean["subject"])
            result = model.fit()
            print(result.summary())
            tukey = pairwise_tukeyhsd(endog=df_long_clean[img_value], groups=df_long_clean['acq'], alpha=0.05)
            print(tukey)
        except Exception as e:
            print(f"  LMM failed: {e}")

    # R-based EMMs
    emm_df = None
    if len(df_long_clean) >= 2:
        try:
            rdf = pandas2ri.py2rpy(df_long_clean)
            ro.globalenv['rdf'] = rdf
            ro.globalenv['img_value'] = img_value

            ro.r('''
            library(lme4); library(emmeans)
            rdf$subject <- as.factor(rdf$subject)
            rdf$acq <- factor(rdf$acq, levels = c("mb3me1fa50","mb3me3","mb3me3ip0","mb2me4","mb3me4","mb3me4fa50"))
            formula_str <- paste(img_value, "~ acq + (1 | subject)")
            model <- lmer(as.formula(formula_str), data = rdf)
            emm <- emmeans(model, ~ acq)
            emm_df <- as.data.frame(emm)
            ''')

            emm_df = pandas2ri.rpy2py(ro.globalenv['emm_df'])
        except Exception as e:
            print(f"  R EMM analysis failed: {e}")

    # Create EMM plot with individual points
    emm_fig = create_sp_emm_plot(emm_df, df_long_clean, acq_params, img_value, 
                                  mask_value, out_dir, type_value, denoise_value)

    return df, emm_fig, result


# =============================================================================
# RUN ANALYSES
# =============================================================================

# Figure 9: Beta estimates for VS in SP subjects
print("\n" + "=" * 80)
print("FIGURE 9: Beta estimates for activation (VS, SP subjects)")
print("=" * 80)
df_beta, emm_fig_beta, result_beta = process_sp_and_visualize(
    base_dir=SP_BASE_DIR, acq_params=SP_ACQ_PARAMS,
    type_value="act", img_value="beta",
    mask_value="VSconstrained", denoise_value="base"
)

# Supp. Figure 9: Zstat for VS in SP subjects
print("\n" + "=" * 80)
print("SUPP. FIGURE 9: Zstat for activation (VS, SP subjects)")
print("=" * 80)
df_zstat, emm_fig_zstat, result_zstat = process_sp_and_visualize(
    base_dir=SP_BASE_DIR, acq_params=SP_ACQ_PARAMS,
    type_value="act", img_value="zstat",
    mask_value="VSconstrained", denoise_value="base"
)

# =============================================================================
# WEIGHTED BETA ANALYSIS FOR SP SUBJECTS (Inverse-Variance Weighting)
# =============================================================================

print("\n" + "=" * 80)
print("SUPP. FIGURE 9b: Weighted Beta for activation (VS, SP subjects)")
print("=" * 80)

# Extract both beta and varcope data
print("\nExtracting beta and varcope data for weighting...")
beta_data = extract_sp_file_data(SP_BASE_DIR, SP_ACQ_PARAMS, "act", "beta", "VSconstrained", "base")
varcope_data = extract_sp_file_data(SP_BASE_DIR, SP_ACQ_PARAMS, "act", "varcope", "VSconstrained", "base")

if beta_data and varcope_data:
    # Find common subjects
    common_subjects = set(beta_data.keys()) & set(varcope_data.keys())
    print(f"  Common subjects with both beta and varcope: {len(common_subjects)}")

    if common_subjects:
        # Build combined dataframe
        records = []
        for subj in common_subjects:
            for acq in SP_ACQ_PARAMS:
                beta_val = beta_data[subj].get(acq, np.nan)
                varcope_val = varcope_data[subj].get(acq, np.nan)
                if pd.notna(beta_val) and pd.notna(varcope_val) and varcope_val > 0:
                    records.append({
                        'subject': subj,
                        'acq': acq,
                        'beta': beta_val,
                        'varcope': varcope_val
                    })

        df_weighted = pd.DataFrame(records)
        print(f"  Combined records: {len(df_weighted)}")

        if len(df_weighted) >= 2:
            # Compute weights
            df_weighted['weight'] = 1.0 / df_weighted['varcope']
            weight_cap = np.percentile(df_weighted['weight'], 99)
            df_weighted['weight'] = df_weighted['weight'].clip(upper=weight_cap)
            df_weighted['weight'] = df_weighted['weight'] / df_weighted['weight'].mean()

            print(f"  Weight range: [{df_weighted['weight'].min():.3f}, {df_weighted['weight'].max():.3f}]")

            # Run weighted LME via R
            emm_df_w = None
            try:
                df_weighted['subject'] = df_weighted['subject'].astype(str)
                df_weighted['acq'] = df_weighted['acq'].astype(str)

                rdf = pandas2ri.py2rpy(df_weighted)
                ro.globalenv['rdf_w'] = rdf

                ro.r('''
                library(lme4); library(lmerTest); library(emmeans)
                rdf_w$subject <- as.factor(rdf_w$subject)
                rdf_w$acq <- factor(rdf_w$acq, levels = c("mb3me1fa50","mb3me3","mb3me3ip0","mb2me4","mb3me4","mb3me4fa50"))

                weighted_model_sp <- lmer(beta ~ acq + (1 | subject), data = rdf_w, weights = weight, REML = TRUE)
                anova_result_sp <- anova(weighted_model_sp, type = 3, ddf = "Satterthwaite")
                emm_sp <- emmeans(weighted_model_sp, ~ acq)
                emm_df_sp <- as.data.frame(emm_sp)
                ''')

                # Print ANOVA results
                anova_r = ro.r('as.data.frame(anova_result_sp)')
                anova_df = pandas2ri.rpy2py(anova_r)
                print("\nWeighted LME ANOVA (Type III):")
                print(anova_df.to_string())

                emm_df_w = pandas2ri.rpy2py(ro.globalenv['emm_df_sp'])

            except Exception as e:
                print(f"  Weighted LME failed: {e}")

            # Create weighted beta plot
            df_weighted['acq'] = pd.Categorical(df_weighted['acq'], categories=SP_ACQ_PARAMS, ordered=True)
            emm_fig_weighted = create_sp_emm_plot(
                emm_df_w, df_weighted, SP_ACQ_PARAMS, "beta",
                "VSconstrained", str(output_dir), "act", "weighted_beta"
            )

            print("  Weighted beta analysis complete.")

        else:
            print("  Insufficient data for weighted analysis.")
else:
    print("  Missing beta or varcope data for weighted analysis.")

print("\n" + "=" * 80)
print("SP SUBJECT ANALYSIS COMPLETE")
print("=" * 80)